# Experiment: FMA-Medium Scale-Up, Hard Negative Mining, And Multi-Segment Temporal Aggregation

This notebook is the final project-stage evaluation and consolidation notebook. It scales the benchmark from `fma_small` to `fma_medium`, re-evaluates all major historical model variants under the harder Notebook 3 retrieval protocol, and adds two targeted improvements aimed at the remaining failure floor: hard negative mining during training and multi-segment temporal aggregation during inference.


## Setup
Install only missing notebook dependencies in the active Colab runtime. Deliberately avoid churning the core PyTorch stack unless a package is missing.

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

MODULE_TO_PACKAGE: dict[str, str] = {
    "faiss": "faiss-cpu",
    "librosa": "librosa",
    "soundfile": "soundfile",
    "sklearn": "scikit-learn",
    "transformers": "transformers",
    "tqdm": "tqdm",
    "pandas": "pandas",
    "numpy": "numpy",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
}


def install_missing_packages(module_to_package: dict[str, str]) -> list[str]:
    """Install only the packages that are missing from the current kernel."""
    missing_packages = [
        package_name
        for module_name, package_name in module_to_package.items()
        if importlib.util.find_spec(module_name) is None
    ]
    if missing_packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *missing_packages])
    return missing_packages


INSTALLED_NOW = install_missing_packages(MODULE_TO_PACKAGE)
if INSTALLED_NOW:
    print("Installed packages:", INSTALLED_NOW)
else:
    print("All required non-core packages were already present.")


## Setup
Imports, preliminary runtime inspection, and early failure on unsupported runtimes.

In [ ]:
from __future__ import annotations

import ast
import gc
import hashlib
import json
import math
import os
import pickle
import platform
import random
import shutil
import tempfile
import time
import urllib.request
import warnings
import zipfile
from collections import defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Callable, Literal, TypedDict

import faiss
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.signal
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from IPython.display import Markdown, display
from sklearn.decomposition import PCA
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, Wav2Vec2FeatureExtractor
from functools import lru_cache


def seed_everything(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch for reproducible notebook execution."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def get_gpu_name() -> str:
    """Return the active CUDA device name, or a CPU marker when unavailable."""
    if torch.cuda.is_available():
        return torch.cuda.get_device_name(torch.cuda.current_device())
    return "CPU"


def get_total_vram_gb() -> float:
    """Return total GPU memory in GiB when CUDA is available."""
    if not torch.cuda.is_available():
        return 0.0
    total_bytes = torch.cuda.get_device_properties(torch.cuda.current_device()).total_memory
    return float(total_bytes / (1024 ** 3))


seed_everything(42)

PRELIMINARY_RUNTIME_REPORT = pd.DataFrame(
    [
        {"component": "Python", "value": platform.python_version()},
        {"component": "PyTorch", "value": torch.__version__},
        {"component": "torchaudio", "value": torchaudio.__version__},
        {"component": "Transformers", "value": __import__("transformers").__version__},
        {"component": "FAISS", "value": faiss.__version__},
        {"component": "CUDA available", "value": str(torch.cuda.is_available())},
        {"component": "GPU", "value": get_gpu_name()},
        {"component": "Total VRAM (GiB)", "value": f"{get_total_vram_gb():.2f}"},
        {
            "component": "BF16 supported",
            "value": str(bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())),
        },
    ]
)
display(PRELIMINARY_RUNTIME_REPORT)

if not torch.cuda.is_available():
    raise RuntimeError(
        "This notebook is designed for a single-GPU Colab runtime. "
        "CUDA is unavailable, so stop here and switch to a GPU runtime."
    )


## Plan

Scientific questions:

- Does the Notebook 3 ranking survive when the retrieval benchmark scales from `fma_small` to `fma_medium`?
- Which historical models collapse most sharply on worst-case conditions once the database grows?
- Does multi-segment aggregation recover the floor more efficiently than retraining?
- Does hard-negative mining improve discriminability without wiping out average retrieval quality?

Notebook 4 execution policy:

1. Re-evaluate all historical runs on `fma_medium`.
2. Keep `realistic_hard` and `multi5_even` active by default.
3. Apply aggregation to every historical checkpoint.
4. Retrain only the representative hard-negative subset unless the runtime is unusually large.
5. Export all tables, markdown summaries, plots, and the final zip bundle.


## Config
Configuration with enough compatibility to reuse the pre-existing Notebook 2 helpers.


In [ ]:
ModelName = Literal["cnn", "hybrid_transformer", "frozen_mert"]
InputMode = Literal["spectrogram", "mert_waveform"]
AugmentationProfileName = str
QueryRegimeName = Literal[
    "clean_current",
    "short_centered",
    "short_offcenter",
    "combined_moderate",
    "multi_segment_same_track",
    "realistic_hard",
    "multi_segment_fragmented_weighted",
    "multi_segment_temporally_jittered",
    "hard_negative_short_queries",
]
WindowingSpecName = Literal["single_center", "multi3_even", "multi5_even"]
AggregationModeName = Literal[
    "single_segment_top1",
    "segment_vote_topk",
    "weighted_segment_vote",
    "temporal_consistency_huber",
]
ExecutionModeName = Literal["quick", "standard", "full", "recovery"]
IndexKind = Literal["exact_ip", "ivfflat", "ivfpq"]
RetrievalConditionName = Literal[
    "clean",
    "noise",
    "pitch",
    "time",
    "lowpass",
    "highpass",
    "combined_moderate",
    "realistic_hard",
]


@dataclass(frozen=True)
class RuntimeInfo:
    """Resolved runtime properties for the active Colab session."""

    python_version: str
    torch_version: str
    torchaudio_version: str
    transformers_version: str
    faiss_version: str
    cuda_available: bool
    device_name: str
    total_vram_gb: float
    bf16_supported: bool


@dataclass(frozen=True)
class PathConfig:
    """Filesystem locations used by Notebook 4 inside Colab."""

    data_root: Path
    output_root: Path
    drive_mount_point: Path
    cache_dir: Path
    checkpoint_dir: Path
    embedding_cache_dir: Path
    index_dir: Path
    metrics_dir: Path
    plot_dir: Path
    export_dir: Path
    log_dir: Path

    @property
    def metadata_zip(self) -> Path:
        return self.data_root / "fma_metadata.zip"

    @property
    def metadata_dir(self) -> Path:
        return self.data_root / "fma_metadata"

    @property
    def audio_zip(self) -> Path:
        return self.data_root / "fma_medium.zip"

    @property
    def audio_dir(self) -> Path:
        return self.data_root / "fma_medium"

    @property
    def artifact_zip(self) -> Path:
        return self.output_root / "notebook4_outputs.zip"

    @property
    def waveform_cache_dir(self) -> Path:
        return self.data_root / "waveform_cache_sr16000"

    @property
    def hard_negative_cache_dir(self) -> Path:
        return self.cache_dir / "hard_negatives"

    @property
    def query_manifest_dir(self) -> Path:
        return self.cache_dir / "query_manifests"

    @property
    def reference_manifest_dir(self) -> Path:
        return self.cache_dir / "reference_manifests"

    def all_directories(self) -> tuple[Path, ...]:
        return (
            self.data_root,
            self.output_root,
            self.cache_dir,
            self.checkpoint_dir,
            self.embedding_cache_dir,
            self.index_dir,
            self.metrics_dir,
            self.plot_dir,
            self.export_dir,
            self.log_dir,
            self.waveform_cache_dir,
            self.hard_negative_cache_dir,
            self.query_manifest_dir,
            self.reference_manifest_dir,
        )


@dataclass(frozen=True)
class ModelRunConfig:
    """Configuration for one trainable run in Notebook 4."""

    model_name: ModelName
    augmentation_profile: AugmentationProfileName
    embed_dim: int
    epochs: int
    learning_rate: float
    weight_decay: float
    temperature: float
    freeze_backbone: bool
    enabled: bool = True

    @property
    def run_id(self) -> str:
        return f"{self.model_name}_{self.augmentation_profile}_embed{self.embed_dim}"


@dataclass(frozen=True)
class HardNegativeConfig:
    """Configuration for offline hard-negative mining and retraining."""

    target_run_ids: tuple[str, ...]
    candidate_track_cap: int
    candidate_segments_per_track: int
    neighbors_to_search: int
    negatives_per_anchor: int
    hard_negative_ratio: float
    triplet_margin: float
    triplet_weight: float
    warm_start_epochs: int
    retrain_epochs: int
    refresh_after_epoch: int | None
    enabled: bool = True


@dataclass(frozen=True)
class AggregationConfig:
    """Configuration for multi-segment aggregation and temporal consistency scoring."""

    modes: tuple[AggregationModeName, ...]
    top_k_per_segment: int
    min_similarity_threshold: float
    residual_tolerance_seconds: float
    minimum_inlier_count: int
    weighted_similarity_weight: float
    vote_count_weight: float
    temporal_inlier_weight: float
    fit_quality_weight: float
    enabled: bool = True


@dataclass(frozen=True)
class ExperimentConfig:
    """Top-level Notebook 4 configuration."""

    metadata_url: str
    audio_url: str
    paths: PathConfig
    seed: int
    batch_size: int
    mert_batch_size: int
    num_workers: int
    dataloader_prefetch_factor: int
    dataloader_persistent_workers: bool
    sample_rate: int
    n_fft: int
    hop_length: int
    n_mels: int
    segment_seconds: float
    amp_enabled: bool
    gradient_checkpointing: bool
    early_stopping_patience: int
    top_k: tuple[int, ...]
    projection_hidden_dim: int
    max_train_tracks: int | None
    max_eval_tracks: int | None
    smoke_test_track_count: int
    gradient_clip_norm: float
    run_presets: tuple[ModelRunConfig, ...]
    strict_artifact_validation: bool
    allow_retrain_missing_baselines: bool
    mount_google_drive: bool
    save_to_drive: bool
    use_torch_compile: bool
    run_error_analysis: bool
    run_pexeso: bool
    run_optional_finetune: bool
    execution_mode: ExecutionModeName
    run_baseline_historical_eval: bool
    run_aggregation_eval: bool
    run_hard_negative_retraining: bool
    run_failure_analysis: bool
    run_plot_exports: bool
    run_zip_export: bool
    resume_from_partial_artifacts: bool
    skip_existing_exports: bool
    force_recompute_embeddings: bool
    force_rebuild_indices: bool
    quick_mode: bool
    short_query_lengths_seconds: tuple[float, ...]
    multi_segment_group_size: int
    search_oversample_factor: int
    faiss_nprobe_values: tuple[int, ...]
    active_training_policy_names: tuple[str, ...]
    active_control_run_ids: tuple[str, ...]
    historical_run_ids: tuple[str, ...]
    active_query_regime_names: tuple[QueryRegimeName, ...]
    active_windowing_names: tuple[WindowingSpecName, ...]
    active_index_names: tuple[str, ...]
    hard_negative_config: HardNegativeConfig
    aggregation_config: AggregationConfig
    optional_partial_unfreeze_last_layers: int = 2
    pexeso_root: Path | None = None

    @property
    def segment_samples(self) -> int:
        return int(self.segment_seconds * self.sample_rate)

    @property
    def spectrogram_frames(self) -> int:
        return 1 + self.segment_samples // self.hop_length

    @property
    def device(self) -> torch.device:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

    @property
    def device_type(self) -> str:
        return "cuda" if torch.cuda.is_available() else "cpu"


def build_runtime_info() -> RuntimeInfo:
    """Capture runtime properties that influence Notebook 4 execution."""

    return RuntimeInfo(
        python_version=platform.python_version(),
        torch_version=torch.__version__,
        torchaudio_version=torchaudio.__version__,
        transformers_version=__import__("transformers").__version__,
        faiss_version=faiss.__version__,
        cuda_available=bool(torch.cuda.is_available()),
        device_name=get_gpu_name(),
        total_vram_gb=get_total_vram_gb(),
        bf16_supported=bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    )


def build_default_config(runtime_info: RuntimeInfo) -> ExperimentConfig:
    """Construct the default Notebook 4 configuration."""

    output_root = Path("/content/song_fingerprinting_outputs/notebook4_fma_medium_scaleup")
    paths = PathConfig(
        data_root=Path("/content/data"),
        output_root=output_root,
        drive_mount_point=Path("/content/drive"),
        cache_dir=output_root / "cache",
        checkpoint_dir=output_root / "checkpoints",
        embedding_cache_dir=output_root / "embeddings",
        index_dir=output_root / "indexes",
        metrics_dir=output_root / "metrics",
        plot_dir=output_root / "plots",
        export_dir=output_root / "exports",
        log_dir=output_root / "logs",
    )
    run_presets = (
        ModelRunConfig("hybrid_transformer", "filters_only", 128, 6, 1e-4, 1e-4, 0.07, False, True),
        ModelRunConfig("hybrid_transformer", "one_of_k", 128, 6, 1e-4, 1e-4, 0.07, False, True),
        ModelRunConfig("hybrid_transformer", "severity_controlled", 128, 6, 1e-4, 1e-4, 0.07, False, True),
        ModelRunConfig("hybrid_transformer", "two_of_k", 128, 6, 1e-4, 1e-4, 0.07, False, False),
        ModelRunConfig("hybrid_transformer", "curriculum_filters_late", 128, 6, 1e-4, 1e-4, 0.07, False, False),
    )
    hard_negative_config = HardNegativeConfig(
        target_run_ids=(
            "cnn_baseline_embed128",
            "hybrid_transformer_baseline_embed128",
            "hybrid_transformer_one_of_k_embed128",
            "frozen_mert_extended_embed128",
        ),
        candidate_track_cap=2048,
        candidate_segments_per_track=3,
        neighbors_to_search=20,
        negatives_per_anchor=2,
        hard_negative_ratio=0.5,
        triplet_margin=0.2,
        triplet_weight=0.25,
        warm_start_epochs=1,
        retrain_epochs=3,
        refresh_after_epoch=2,
        enabled=True,
    )
    aggregation_config = AggregationConfig(
        modes=(
            "single_segment_top1",
            "segment_vote_topk",
            "weighted_segment_vote",
            "temporal_consistency_huber",
        ),
        top_k_per_segment=10,
        min_similarity_threshold=0.05,
        residual_tolerance_seconds=0.75,
        minimum_inlier_count=2,
        weighted_similarity_weight=0.55,
        vote_count_weight=0.15,
        temporal_inlier_weight=0.20,
        fit_quality_weight=0.10,
        enabled=True,
    )
    return ExperimentConfig(
        metadata_url="https://os.unil.cloud.switch.ch/fma/fma_metadata.zip",
        audio_url="https://os.unil.cloud.switch.ch/fma/fma_medium.zip",
        paths=paths,
        seed=42,
        batch_size=24,
        mert_batch_size=8,
        num_workers=4,
        dataloader_prefetch_factor=4,
        dataloader_persistent_workers=True,
        sample_rate=16_000,
        n_fft=1024,
        hop_length=256,
        n_mels=128,
        segment_seconds=3.0,
        amp_enabled=runtime_info.cuda_available,
        gradient_checkpointing=False,
        early_stopping_patience=3,
        top_k=(1, 5, 10),
        projection_hidden_dim=1024,
        max_train_tracks=None,
        max_eval_tracks=None,
        smoke_test_track_count=16,
        gradient_clip_norm=1.0,
        run_presets=run_presets,
        strict_artifact_validation=True,
        allow_retrain_missing_baselines=False,
        mount_google_drive=True,
        save_to_drive=True,
        use_torch_compile=False,
        run_error_analysis=True,
        run_pexeso=False,
        run_optional_finetune=False,
        execution_mode="standard",
        run_baseline_historical_eval=True,
        run_aggregation_eval=True,
        run_hard_negative_retraining=True,
        run_failure_analysis=True,
        run_plot_exports=True,
        run_zip_export=True,
        resume_from_partial_artifacts=True,
        skip_existing_exports=True,
        force_recompute_embeddings=False,
        force_rebuild_indices=False,
        quick_mode=False,
        short_query_lengths_seconds=(1.0, 2.0, 3.0),
        multi_segment_group_size=3,
        search_oversample_factor=16,
        faiss_nprobe_values=(1, 4, 8),
        active_training_policy_names=("filters_only", "one_of_k", "severity_controlled"),
        active_control_run_ids=(
            "cnn_baseline_embed128",
            "hybrid_transformer_baseline_embed128",
            "hybrid_transformer_extended_embed128",
            "hybrid_transformer_extended_embed256",
            "frozen_mert_extended_embed128",
        ),
        historical_run_ids=(
            "cnn_baseline_embed128",
            "hybrid_transformer_baseline_embed128",
            "hybrid_transformer_extended_embed128",
            "hybrid_transformer_extended_embed256",
            "frozen_mert_extended_embed128",
            "hybrid_transformer_filters_only_embed128",
            "hybrid_transformer_one_of_k_embed128",
            "hybrid_transformer_severity_controlled_embed128",
        ),
        active_query_regime_names=(
            "clean_current",
            "short_centered",
            "short_offcenter",
            "combined_moderate",
            "multi_segment_same_track",
            "realistic_hard",
        ),
        active_windowing_names=("single_center", "multi3_even", "multi5_even"),
        active_index_names=("exact_ip", "ivfflat_nprobe1", "ivfflat_nprobe4", "ivfflat_nprobe8", "ivfpq_nprobe4"),
        hard_negative_config=hard_negative_config,
        aggregation_config=aggregation_config,
    )


RUNTIME_INFO = build_runtime_info()
NOTEBOOK4_CONFIG = build_default_config(RUNTIME_INFO)
CONFIG = NOTEBOOK4_CONFIG

for output_directory in CONFIG.paths.all_directories():
    output_directory.mkdir(parents=True, exist_ok=True)

CONFIG_OVERVIEW = pd.DataFrame(
    [
        {"field": "device", "value": str(CONFIG.device)},
        {"field": "gpu_name", "value": RUNTIME_INFO.device_name},
        {"field": "vram_gb", "value": f"{RUNTIME_INFO.total_vram_gb:.2f}"},
        {"field": "execution_mode", "value": CONFIG.execution_mode},
        {"field": "output_root", "value": str(CONFIG.paths.output_root)},
        {"field": "run_baseline_historical_eval", "value": str(CONFIG.run_baseline_historical_eval)},
        {"field": "run_aggregation_eval", "value": str(CONFIG.run_aggregation_eval)},
        {"field": "run_hard_negative_retraining", "value": str(CONFIG.run_hard_negative_retraining)},
        {"field": "active_query_regimes", "value": ", ".join(CONFIG.active_query_regime_names)},
        {"field": "active_windowing", "value": ", ".join(CONFIG.active_windowing_names)},
    ]
)
display(CONFIG_OVERVIEW)


## Environment, Utilities, Runtime Reports, And Artifact Discovery


In [ ]:
# Shared utilities, runtime reporting, and historical artifact discovery.
from dataclasses import replace


def serialize_jsonable(value: object) -> object:
    """Recursively convert notebook objects into JSON-safe values."""

    if isinstance(value, Path):
        return str(value)
    if isinstance(value, tuple):
        return [serialize_jsonable(item) for item in value]
    if isinstance(value, list):
        return [serialize_jsonable(item) for item in value]
    if isinstance(value, dict):
        return {str(key): serialize_jsonable(item) for key, item in value.items()}
    if hasattr(value, "__dataclass_fields__"):
        return {
            field_name: serialize_jsonable(getattr(value, field_name))
            for field_name in value.__dataclass_fields__.keys()
        }
    return value


def safe_mkdir(path: Path) -> Path:
    """Create a directory and return the resolved path."""

    path.mkdir(parents=True, exist_ok=True)
    return path


def save_json(filepath: Path, payload: object) -> None:
    """Write a JSON artifact with stable indentation."""

    safe_mkdir(filepath.parent)
    with filepath.open("w", encoding="utf-8") as handle:
        json.dump(serialize_jsonable(payload), handle, indent=2)
        handle.write("\n")


def load_json(filepath: Path) -> object:
    """Load a JSON artifact from disk."""

    with filepath.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def save_pickle(filepath: Path, payload: object) -> None:
    """Persist an object using pickle for cache-heavy notebook phases."""

    safe_mkdir(filepath.parent)
    with filepath.open("wb") as handle:
        pickle.dump(payload, handle)


def load_pickle(filepath: Path) -> object:
    """Load a pickle payload from disk."""

    with filepath.open("rb") as handle:
        return pickle.load(handle)


def save_dataframe(filepath: Path, dataframe: pd.DataFrame) -> None:
    """Write a dataframe to CSV after ensuring the parent directory exists."""

    safe_mkdir(filepath.parent)
    dataframe.to_csv(filepath, index=False)


def checkpoint_exists(path: Path | None) -> bool:
    """Return whether a checkpoint path exists and is a regular file."""

    return bool(path is not None and path.is_file())


def normalize_embeddings(embeddings: np.ndarray) -> np.ndarray:
    """L2-normalize a float32 embedding matrix without mutating the caller's array."""

    matrix = np.asarray(embeddings, dtype=np.float32).copy()
    faiss.normalize_L2(matrix)
    return matrix


def safe_to_device(batch: object, device: torch.device, non_blocking: bool = True) -> object:
    """Move nested tensors to a device while leaving non-tensors untouched."""

    if isinstance(batch, torch.Tensor):
        return batch.to(device, non_blocking=non_blocking)
    if isinstance(batch, dict):
        return {
            key: safe_to_device(value, device, non_blocking=non_blocking)
            for key, value in batch.items()
        }
    if isinstance(batch, list):
        return [safe_to_device(value, device, non_blocking=non_blocking) for value in batch]
    return batch


def count_parameters(model: nn.Module, trainable_only: bool = False) -> int:
    """Count model parameters, optionally restricting to trainable tensors."""

    parameters = model.parameters()
    if trainable_only:
        parameters = (parameter for parameter in parameters if parameter.requires_grad)
    return int(sum(parameter.numel() for parameter in parameters))


def format_seconds(total_seconds: float) -> str:
    """Format a duration in seconds into a compact human-readable string."""

    total_seconds = max(0.0, float(total_seconds))
    minutes, seconds = divmod(int(total_seconds), 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f"{hours}h {minutes}m {seconds}s"
    if minutes:
        return f"{minutes}m {seconds}s"
    return f"{seconds}s"


def hash_experiment_config(payload: object) -> str:
    """Hash a JSON-serializable payload into a short stable cache key."""

    encoded = json.dumps(serialize_jsonable(payload), sort_keys=True).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()[:16]


def dataframe_hash(dataframe: pd.DataFrame) -> str:
    """Create a stable hash for a dataframe used in notebook caches."""

    hashed_values = pd.util.hash_pandas_object(dataframe, index=True).values.tobytes()
    return hashlib.sha256(hashed_values).hexdigest()[:16]


def print_phase_banner(title: str) -> None:
    """Print a compact phase banner for long-running notebook stages."""

    print(f"\n{'=' * 20} {title} {'=' * 20}")


def clear_memory() -> None:
    """Release cached Python and CUDA memory between notebook phases."""

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def maybe_mount_google_drive(config: ExperimentConfig) -> Path | None:
    """Mount Google Drive when explicitly enabled and available in the runtime."""

    if not config.mount_google_drive:
        print("Google Drive mounting is disabled by configuration.")
        return None
    try:
        from google.colab import drive as colab_drive
    except ImportError:
        print("google.colab is unavailable; continuing without Drive.")
        return None
    mount_point = config.paths.drive_mount_point
    colab_drive.mount(str(mount_point))
    return mount_point


def get_total_ram_gb() -> float:
    """Estimate total system RAM in GiB."""

    if hasattr(os, "sysconf") and "SC_PAGE_SIZE" in os.sysconf_names and "SC_PHYS_PAGES" in os.sysconf_names:
        page_size = os.sysconf("SC_PAGE_SIZE")
        page_count = os.sysconf("SC_PHYS_PAGES")
        return float(page_size * page_count / (1024 ** 3))
    return float("nan")


def get_free_disk_gb(path: Path) -> float:
    """Return the free disk capacity at a path in GiB."""

    usage = shutil.disk_usage(str(path))
    return float(usage.free / (1024 ** 3))


def can_write_to_directory(path: Path) -> bool:
    """Return whether the current runtime can create and delete a file in a directory."""

    safe_mkdir(path)
    probe_path = path / ".notebook4_write_probe"
    try:
        probe_path.write_text("probe", encoding="utf-8")
        probe_path.unlink()
        return True
    except Exception:
        return False


def estimate_required_local_disk_gb(config: ExperimentConfig) -> float:
    """Estimate the local disk required for a standard FMA-Medium run."""

    base_archive_gb = 22.0
    extracted_audio_gb = 26.0
    embeddings_and_indexes_gb = 10.0 if config.execution_mode == "quick" else 18.0
    slack_gb = 8.0
    return base_archive_gb + extracted_audio_gb + embeddings_and_indexes_gb + slack_gb


def resolve_execution_mode_for_runtime(config: ExperimentConfig, free_disk_gb: float) -> ExperimentConfig:
    """Downgrade the execution mode when the runtime cannot safely support the requested scope."""

    required_standard_gb = estimate_required_local_disk_gb(config)
    if free_disk_gb >= required_standard_gb or config.execution_mode in {"quick", "recovery"}:
        return config
    downgraded_regimes = (
        "clean_current",
        "short_centered",
        "short_offcenter",
        "combined_moderate",
        "multi_segment_same_track",
        "realistic_hard",
    )
    downgraded_windowing = ("single_center", "multi3_even")
    return replace(
        config,
        execution_mode="quick",
        quick_mode=True,
        run_hard_negative_retraining=False,
        run_plot_exports=True,
        active_query_regime_names=downgraded_regimes,
        active_windowing_names=downgraded_windowing,
        max_train_tracks=2048 if config.max_train_tracks is None else min(config.max_train_tracks, 2048),
        max_eval_tracks=1024 if config.max_eval_tracks is None else min(config.max_eval_tracks, 1024),
    )


def resolve_notebook2_output_roots(drive_root: Path | None) -> list[Path]:
    """Resolve plausible Notebook 2 output roots from local and Drive-backed locations."""

    candidate_roots = [Path("/content/song_fingerprinting_outputs")]
    if drive_root is not None:
        candidate_roots.extend(
            [
                drive_root / "MyDrive" / "song_fingerprinting_outputs",
                drive_root / "MyDrive" / "transformer-fingerprinting" / "song_fingerprinting_outputs",
                drive_root / "MyDrive" / "Colab Notebooks" / "song_fingerprinting_outputs",
            ]
        )
        notebook2_export_root = drive_root / "MyDrive" / "transformer_fingerprinting" / "artifacts" / "notebook2"
        if notebook2_export_root.exists():
            for run_export_dir in sorted(notebook2_export_root.glob("run_*")):
                exported_output_root = run_export_dir / "song_fingerprinting_outputs"
                if exported_output_root.exists():
                    candidate_roots.append(exported_output_root)
    unique_roots: list[Path] = []
    seen: set[str] = set()
    for root in candidate_roots:
        root_key = str(root)
        if root_key in seen:
            continue
        seen.add(root_key)
        unique_roots.append(root)
    return unique_roots


def resolve_notebook_output_roots(drive_root: Path | None, notebook_name: str) -> list[Path]:
    """Resolve plausible notebook output roots for Notebook 3 or Notebook 4."""

    local_root = Path("/content/song_fingerprinting_outputs") / notebook_name
    candidate_roots = [local_root]
    if drive_root is not None:
        notebook_export_root = drive_root / "MyDrive" / "transformer_fingerprinting" / "artifacts" / notebook_name.split("_")[0]
        if notebook_export_root.exists():
            for run_export_dir in sorted(notebook_export_root.glob("run_*")):
                exported_output_root = run_export_dir / notebook_name
                if exported_output_root.exists():
                    candidate_roots.append(exported_output_root)
    unique_roots: list[Path] = []
    seen: set[str] = set()
    for root in candidate_roots:
        root_key = str(root)
        if root_key in seen:
            continue
        seen.add(root_key)
        unique_roots.append(root)
    return unique_roots


HISTORICAL_RUN_REGISTRY: dict[str, dict[str, object]] = {
    "cnn_baseline_embed128": {"source": "notebook2", "model_name": "cnn", "augmentation_profile": "baseline", "embed_dim": 128, "freeze_backbone": False},
    "hybrid_transformer_baseline_embed128": {"source": "notebook2", "model_name": "hybrid_transformer", "augmentation_profile": "baseline", "embed_dim": 128, "freeze_backbone": False},
    "hybrid_transformer_extended_embed128": {"source": "notebook2", "model_name": "hybrid_transformer", "augmentation_profile": "extended_existing", "embed_dim": 128, "freeze_backbone": False},
    "hybrid_transformer_extended_embed256": {"source": "notebook2", "model_name": "hybrid_transformer", "augmentation_profile": "extended_existing", "embed_dim": 256, "freeze_backbone": False},
    "frozen_mert_extended_embed128": {"source": "notebook2", "model_name": "frozen_mert", "augmentation_profile": "extended_existing", "embed_dim": 128, "freeze_backbone": True},
    "hybrid_transformer_filters_only_embed128": {"source": "notebook3", "model_name": "hybrid_transformer", "augmentation_profile": "filters_only", "embed_dim": 128, "freeze_backbone": False},
    "hybrid_transformer_one_of_k_embed128": {"source": "notebook3", "model_name": "hybrid_transformer", "augmentation_profile": "one_of_k", "embed_dim": 128, "freeze_backbone": False},
    "hybrid_transformer_severity_controlled_embed128": {"source": "notebook3", "model_name": "hybrid_transformer", "augmentation_profile": "severity_controlled", "embed_dim": 128, "freeze_backbone": False},
}


def try_load_json(path: Path) -> tuple[bool, str]:
    """Return whether a JSON file can be parsed successfully."""

    if not path.exists():
        return False, "missing"
    try:
        load_json(path)
    except Exception as exc:
        return False, f"json_error:{type(exc).__name__}"
    return True, "ok"


def try_load_checkpoint(path: Path) -> tuple[bool, str]:
    """Return whether a checkpoint file is readable by PyTorch."""

    if not path.exists():
        return False, "missing"
    try:
        torch.load(path, map_location="cpu")
    except Exception as exc:
        return False, f"checkpoint_error:{type(exc).__name__}"
    return True, "ok"


def build_runtime_report(config: ExperimentConfig, drive_root: Path | None) -> pd.DataFrame:
    """Construct the runtime report used to gate large Colab runs."""

    output_root_writable = can_write_to_directory(config.paths.output_root)
    free_disk_gb = get_free_disk_gb(config.paths.data_root)
    required_disk_gb = estimate_required_local_disk_gb(config)
    report_rows = [
        {"component": "python_version", "value": platform.python_version()},
        {"component": "torch_version", "value": torch.__version__},
        {"component": "torchaudio_version", "value": torchaudio.__version__},
        {"component": "transformers_version", "value": __import__("transformers").__version__},
        {"component": "faiss_version", "value": faiss.__version__},
        {"component": "cuda_available", "value": str(torch.cuda.is_available())},
        {"component": "gpu_name", "value": get_gpu_name()},
        {"component": "total_vram_gb", "value": f"{get_total_vram_gb():.2f}"},
        {"component": "bf16_supported", "value": str(bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported()))},
        {"component": "total_ram_gb", "value": f"{get_total_ram_gb():.2f}"},
        {"component": "free_disk_gb", "value": f"{free_disk_gb:.2f}"},
        {"component": "required_local_disk_gb", "value": f"{required_disk_gb:.2f}"},
        {"component": "drive_mounted", "value": str(drive_root is not None)},
        {"component": "output_root_writable", "value": str(output_root_writable)},
        {"component": "execution_mode", "value": config.execution_mode},
    ]
    return pd.DataFrame(report_rows)


def discover_historical_artifacts(config: ExperimentConfig, notebook2_roots: list[Path], notebook3_roots: list[Path], notebook4_roots: list[Path]) -> pd.DataFrame:
    """Discover historical model artifacts across Notebook 2, Notebook 3, and Notebook 4 roots."""

    rows: list[dict[str, object]] = []
    for run_id in config.historical_run_ids:
        metadata = HISTORICAL_RUN_REGISTRY[run_id]
        source = str(metadata["source"])
        if source == "notebook2":
            candidate_paths = [root / run_id for root in notebook2_roots]
            expected_files = {"checkpoint": "checkpoint_best.pt", "config": "config.json", "history": "history.json", "degradation": "degradation_breakdown.csv"}
        else:
            candidate_paths = [root / "checkpoints" / run_id for root in notebook3_roots]
            candidate_paths.extend(root / "checkpoints" / run_id for root in notebook4_roots)
            expected_files = {"checkpoint": "checkpoint_best.pt", "config": "training_summary.json", "history": "history.json", "degradation": "training_summary.json"}
        resolved_dir = next((path for path in candidate_paths if path.exists()), None)
        checkpoint_path = resolved_dir / expected_files["checkpoint"] if resolved_dir is not None else None
        config_path = resolved_dir / expected_files["config"] if resolved_dir is not None else None
        history_path = resolved_dir / expected_files["history"] if resolved_dir is not None else None
        degradation_path = resolved_dir / expected_files["degradation"] if resolved_dir is not None else None
        checkpoint_ok, checkpoint_state = try_load_checkpoint(checkpoint_path) if checkpoint_path is not None else (False, "missing")
        config_ok, config_state = try_load_json(config_path) if config_path is not None and config_path.suffix == ".json" else (bool(config_path and config_path.exists()), "ok" if config_path and config_path.exists() else "missing")
        history_ok, history_state = try_load_json(history_path) if history_path is not None and history_path.suffix == ".json" else (bool(history_path and history_path.exists()), "ok" if history_path and history_path.exists() else "missing")
        degradation_exists = bool(degradation_path is not None and degradation_path.exists())
        if resolved_dir is None:
            status = "missing"
        elif checkpoint_ok and config_ok and history_ok:
            status = "valid"
        elif checkpoint_path is not None and checkpoint_path.exists():
            status = "incomplete"
        else:
            status = "corrupt"
        rows.append({"run_id": run_id, "expected_source": source, "resolved_dir": str(resolved_dir) if resolved_dir is not None else "", "status": status, "checkpoint_path": str(checkpoint_path) if checkpoint_path is not None else "", "checkpoint_state": checkpoint_state, "config_path": str(config_path) if config_path is not None else "", "config_state": config_state, "history_path": str(history_path) if history_path is not None else "", "history_state": history_state, "degradation_path": str(degradation_path) if degradation_path is not None else "", "degradation_exists": degradation_exists, "model_name": str(metadata["model_name"]), "augmentation_profile": str(metadata["augmentation_profile"]), "embed_dim": int(metadata["embed_dim"]), "freeze_backbone": bool(metadata["freeze_backbone"])})
    artifact_df = pd.DataFrame(rows).sort_values(["status", "run_id"]).reset_index(drop=True)
    if config.strict_artifact_validation:
        invalid_rows = artifact_df[artifact_df["status"].isin(["missing", "corrupt"])]
        if not invalid_rows.empty and not config.allow_retrain_missing_baselines:
            missing_run_ids = ", ".join(invalid_rows["run_id"].tolist())
            raise FileNotFoundError("Historical artifacts required by Notebook 4 are missing or corrupt. " f"Problematic runs: {missing_run_ids}. " "Mount the correct Drive mirror or set allow_retrain_missing_baselines=True before continuing.")
    return artifact_df


print_phase_banner("Runtime reporting and historical artifact discovery")
DRIVE_ROOT = maybe_mount_google_drive(CONFIG)
RUNTIME_REPORT_DF = build_runtime_report(CONFIG, DRIVE_ROOT)
display(RUNTIME_REPORT_DF)
save_dataframe(CONFIG.paths.export_dir / "runtime_report.csv", RUNTIME_REPORT_DF)

CONFIG = resolve_execution_mode_for_runtime(CONFIG, get_free_disk_gb(CONFIG.paths.data_root))

NOTEBOOK2_OUTPUT_ROOTS = resolve_notebook2_output_roots(DRIVE_ROOT)
NOTEBOOK3_OUTPUT_ROOTS = resolve_notebook_output_roots(DRIVE_ROOT, "notebook3_robustness")
NOTEBOOK4_OUTPUT_ROOTS = resolve_notebook_output_roots(DRIVE_ROOT, "notebook4_fma_medium_scaleup")
CONFIG_HASH = hash_experiment_config({"config": CONFIG, "runtime": RUNTIME_INFO})
HISTORICAL_ARTIFACTS_DF = discover_historical_artifacts(CONFIG, NOTEBOOK2_OUTPUT_ROOTS, NOTEBOOK3_OUTPUT_ROOTS, NOTEBOOK4_OUTPUT_ROOTS)
display(HISTORICAL_ARTIFACTS_DF)
save_dataframe(CONFIG.paths.export_dir / "artifact_check_report.csv", HISTORICAL_ARTIFACTS_DF)
save_json(CONFIG.paths.export_dir / "notebook4_runtime_info.json", {"runtime": RUNTIME_INFO, "config_hash": CONFIG_HASH, "drive_root": DRIVE_ROOT, "notebook2_output_roots": NOTEBOOK2_OUTPUT_ROOTS, "notebook3_output_roots": NOTEBOOK3_OUTPUT_ROOTS, "notebook4_output_roots": NOTEBOOK4_OUTPUT_ROOTS})
save_json(CONFIG.paths.export_dir / "notebook4_config.json", {"config": CONFIG, "runtime": RUNTIME_INFO, "config_hash": CONFIG_HASH})


## Dataset Bootstrap For `fma_medium`


In [ ]:
# Dataset bootstrap helpers for FMA-Medium.
def run_command(command: list[str], cwd: Path | None = None) -> None:
    """Run a subprocess command and surface failures immediately."""

    subprocess.run(command, cwd=str(cwd) if cwd is not None else None, check=True)


def download_file(url: str, destination: Path) -> None:
    """Download a file with curl and overwrite any partial artifact."""

    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {url} -> {destination}")
    run_command(["curl", "-L", "--fail", "--output", str(destination), url])


def extract_zip_file(zip_path: Path, destination_dir: Path) -> None:
    """Extract a zip archive into the requested directory."""

    if not zip_path.exists():
        raise FileNotFoundError(f"Zip archive not found: {zip_path}")
    destination_dir.mkdir(parents=True, exist_ok=True)
    run_command(["unzip", "-q", "-o", str(zip_path), "-d", str(destination_dir)])


def resolve_drive_cached_archive(drive_root: Path | None, archive_name: str) -> Path | None:
    """Resolve a Drive-backed archive cache candidate when one exists."""

    if drive_root is None:
        return None
    candidates = [
        drive_root / "MyDrive" / "transformer_fingerprinting" / "datasets" / archive_name,
        drive_root / "MyDrive" / "transformer_fingerprinting" / "data" / archive_name,
        drive_root / "MyDrive" / archive_name,
    ]
    return next((candidate for candidate in candidates if candidate.exists()), None)


def ensure_local_archive(url: str, destination: Path, drive_root: Path | None) -> Path:
    """Ensure an archive exists locally, preferring a Drive cache before downloading."""

    if destination.exists():
        return destination
    drive_archive = resolve_drive_cached_archive(drive_root, destination.name)
    if drive_archive is not None:
        safe_mkdir(destination.parent)
        shutil.copy2(drive_archive, destination)
        return destination
    download_file(url, destination)
    return destination


def download_fma_assets(config: ExperimentConfig, drive_root: Path | None) -> None:
    """Ensure FMA metadata and FMA-Medium audio are available locally."""

    if not config.paths.metadata_dir.exists():
        ensure_local_archive(config.metadata_url, config.paths.metadata_zip, drive_root)
        extract_zip_file(config.paths.metadata_zip, config.paths.data_root)
    else:
        print(f"Metadata already present at {config.paths.metadata_dir}")

    if not config.paths.audio_dir.exists():
        ensure_local_archive(config.audio_url, config.paths.audio_zip, drive_root)
        extract_zip_file(config.paths.audio_zip, config.paths.data_root)
    else:
        print(f"Audio already present at {config.paths.audio_dir}")


def load_tracks(filepath: Path) -> pd.DataFrame:
    """Load `tracks.csv` using the official MultiIndex column structure."""

    tracks = pd.read_csv(filepath, index_col=0, header=[0, 1])
    list_columns = [("track", "tags"), ("track", "genres"), ("track", "genres_all")]
    for column in list_columns:
        tracks[column] = tracks[column].map(lambda value: ast.literal_eval(value) if isinstance(value, str) else value)
    datetime_columns = [
        ("track", "date_created"),
        ("track", "date_recorded"),
        ("album", "date_created"),
        ("album", "date_released"),
        ("artist", "date_created"),
        ("artist", "active_year_begin"),
        ("artist", "active_year_end"),
    ]
    for column in datetime_columns:
        tracks[column] = pd.to_datetime(tracks[column], errors="coerce")
    subset_dtype = pd.CategoricalDtype(categories=("small", "medium", "large"), ordered=True)
    split_dtype = pd.CategoricalDtype(categories=("training", "validation", "test"), ordered=True)
    tracks["set", "subset"] = tracks["set", "subset"].astype(subset_dtype)
    tracks["set", "split"] = tracks["set", "split"].astype(split_dtype)
    return tracks


def load_metadata_tables(config: ExperimentConfig) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load the FMA metadata tables required by the experiment notebook."""

    metadata_dir = config.paths.metadata_dir
    tracks = load_tracks(metadata_dir / "tracks.csv")
    genres = pd.read_csv(metadata_dir / "genres.csv", index_col=0)
    features = pd.read_csv(metadata_dir / "features.csv", index_col=0, header=[0, 1, 2])
    return tracks, genres, features


def get_audio_path(audio_dir: Path, track_id: int) -> Path:
    """Build the FMA-Medium file path for a track ID."""

    track_key = f"{track_id:06d}"
    return audio_dir / track_key[:3] / f"{track_key}.mp3"


def select_medium_subset(tracks: pd.DataFrame) -> pd.DataFrame:
    """Return the nested FMA-Medium subset, including the embedded FMA-Small tracks."""

    return tracks[tracks["set", "subset"] <= "medium"].copy()


def verify_dataset_layout(audio_dir: Path, subset_tracks: pd.DataFrame) -> dict[str, int]:
    """Check the expected FMA-Medium layout and count missing audio files."""

    if not audio_dir.exists():
        raise FileNotFoundError(f"Audio directory does not exist: {audio_dir}")
    subset_ids = [int(track_id) for track_id in subset_tracks.index.tolist()]
    missing_count = sum(0 if get_audio_path(audio_dir, track_id).exists() else 1 for track_id in subset_ids)
    return {"total_tracks": len(subset_ids), "training_tracks": int((subset_tracks["set", "split"] == "training").sum()), "validation_tracks": int((subset_tracks["set", "split"] == "validation").sum()), "test_tracks": int((subset_tracks["set", "split"] == "test").sum()), "missing_audio_files": missing_count}


def find_undecodable_audio_tracks(audio_dir: Path, track_ids: list[int], probe_sample_rate: int) -> pd.DataFrame:
    """Return tracks that exist on disk but fail to decode in the current runtime."""

    import warnings

    failure_rows: list[dict[str, object]] = []
    for track_id in tqdm(track_ids, desc="probe-audio", leave=False):
        filepath = get_audio_path(audio_dir, track_id)
        if not filepath.is_file():
            failure_rows.append({"track_id": int(track_id), "path": str(filepath), "error_type": "FileNotFoundError", "error_message": "File does not exist or is not a regular file."})
            continue
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                librosa.load(str(filepath), sr=probe_sample_rate, mono=True, duration=0.1)
        except Exception as exc:
            failure_rows.append({"track_id": int(track_id), "path": str(filepath), "error_type": type(exc).__name__, "error_message": str(exc)})
    return pd.DataFrame(failure_rows)


In [ ]:
download_fma_assets(CONFIG, DRIVE_ROOT)
tracks, genres, features = load_metadata_tables(CONFIG)
tracks_medium = select_medium_subset(tracks)
tracks_small = tracks_medium.copy()

probe_track_ids = [int(track_id) for track_id in tracks_medium.index.tolist()]
if CONFIG.execution_mode != "full":
    probe_track_ids = probe_track_ids[: min(len(probe_track_ids), 1024)]

audio_probe_failures = find_undecodable_audio_tracks(audio_dir=CONFIG.paths.audio_dir, track_ids=probe_track_ids, probe_sample_rate=CONFIG.sample_rate)

known_bad_ids = {99134, 108925, 133297}
detected_bad_ids = set(audio_probe_failures["track_id"].tolist()) if not audio_probe_failures.empty else set()
excluded_track_ids = sorted(known_bad_ids | detected_bad_ids)

tracks_medium = tracks_medium.loc[~tracks_medium.index.isin(excluded_track_ids)].copy()
tracks_small = tracks_medium.copy()

dataset_summary = verify_dataset_layout(CONFIG.paths.audio_dir, tracks_medium)
dataset_summary["excluded_bad_audio_files"] = len(excluded_track_ids)
dataset_summary["probe_track_count"] = len(probe_track_ids)
dataset_summary["subset_name"] = "fma_medium"

split_distribution = tracks_medium["set", "split"].value_counts().rename_axis("split").reset_index(name="tracks")
genre_coverage = (
    tracks_medium.groupby([("set", "split"), ("track", "genre_top")])
    .size()
    .rename("tracks")
    .reset_index()
    .rename(columns={("set", "split"): "split", ("track", "genre_top"): "genre"})
)

DATASET_LAYOUT_REPORT = pd.DataFrame([dataset_summary])
save_dataframe(CONFIG.paths.export_dir / "dataset_layout_report.csv", DATASET_LAYOUT_REPORT)
if not audio_probe_failures.empty:
    save_dataframe(CONFIG.paths.export_dir / "undecodable_audio_tracks.csv", audio_probe_failures.sort_values("track_id").reset_index(drop=True))

print("Dataset summary:")
print(json.dumps(dataset_summary, indent=2))
print(f"Excluded bad track IDs: {excluded_track_ids[:20]}{'...' if len(excluded_track_ids) > 20 else ''}")

display(DATASET_LAYOUT_REPORT)
if not audio_probe_failures.empty:
    display(audio_probe_failures.sort_values("track_id").reset_index(drop=True).head(20))
display(split_distribution)
display(genre_coverage.pivot(index="genre", columns="split", values="tracks").fillna(0).astype(int))


## Preprocessing

Reuse the working 16 kHz log-mel path from the exploration notebook for the CNN and hybrid transformer. MERT uses a separate raw-waveform path and its own processor sample rate. Every helper here validates inputs and returns deterministic shapes so later training cells do not depend on hidden notebook state.

### Audio loading, segmentation, normalization, and spectrogram helpers

In [ ]:
def validate_positive_int(name: str, value: int) -> None:
    """Validate that an integer configuration value is strictly positive."""
    if value <= 0:
        raise ValueError(f"{name} must be positive, received {value}.")


def validate_non_empty_waveform(waveform: np.ndarray) -> None:
    """Validate that an audio array contains at least one sample."""
    if waveform.ndim != 1:
        raise ValueError(f"Waveform must be 1-D, received shape {waveform.shape}.")
    if waveform.size == 0:
        raise ValueError("Waveform is empty.")

def get_waveform_cache_path(cache_dir: Path, filepath: Path, target_sr: int, mono: bool) -> Path:
    """Return the on-disk cache path for a decoded and resampled waveform."""
    sr_token = f"sr{target_sr}"
    channel_token = "mono" if mono else "multi"
    return cache_dir / filepath.parent.name / f"{filepath.stem}.{sr_token}.{channel_token}.npy"

def load_audio_file(
    filepath: Path,
    target_sr: int | None,
    mono: bool = True,
    cache_dir: Path | None = None,
) -> tuple[np.ndarray, int]:
    """Load an audio file, preferring torchaudio/TorchCodec and falling back to librosa when needed."""
    if not filepath.exists():
        raise FileNotFoundError(f"Audio file not found: {filepath}")
    if target_sr is None:
        raise ValueError("This notebook expects target_sr to be set explicitly.")

    validate_positive_int("target_sr", target_sr)

    cache_path: Path | None = None
    if cache_dir is not None:
        cache_path = get_waveform_cache_path(cache_dir, filepath, target_sr, mono)
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        if cache_path.exists():
            waveform = np.load(cache_path, allow_pickle=False)
            validate_non_empty_waveform(waveform)
            return waveform.astype(np.float32, copy=False), target_sr

    waveform: np.ndarray | None = None
    sample_rate: int | None = None
    torchaudio_error: Exception | None = None

    try:
        waveform_tensor, sr = torchaudio.load(str(filepath))
        if mono:
            waveform_tensor = waveform_tensor.mean(dim=0, keepdim=True)
        if int(sr) != int(target_sr):
            waveform_tensor = torchaudio.functional.resample(
                waveform_tensor,
                orig_freq=int(sr),
                new_freq=int(target_sr),
            )
            sr = int(target_sr)

        waveform = waveform_tensor.squeeze(0).contiguous().numpy().astype(np.float32)
        sample_rate = int(sr)

    except Exception as exc:
        torchaudio_error = exc

    if waveform is None or sample_rate is None:
        try:
            waveform, sample_rate = librosa.load(
                str(filepath),
                sr=target_sr,
                mono=mono,
            )
            waveform = waveform.astype(np.float32, copy=False)
            sample_rate = int(sample_rate)
        except Exception as librosa_error:
            raise RuntimeError(
                f"Failed to decode audio file with both torchaudio and librosa: {filepath}\n"
                f"torchaudio error: {repr(torchaudio_error)}\n"
                f"librosa error: {repr(librosa_error)}"
            ) from librosa_error

    validate_non_empty_waveform(waveform)

    if cache_path is not None:
        np.save(cache_path, waveform, allow_pickle=False)

    return waveform, sample_rate


def trim_or_pad_waveform(waveform: np.ndarray, target_length: int) -> np.ndarray:
    """Return a waveform with exactly `target_length` samples."""
    validate_non_empty_waveform(waveform)
    validate_positive_int("target_length", target_length)
    if waveform.shape[0] >= target_length:
        return waveform[:target_length].astype(np.float32)
    padded = np.zeros(target_length, dtype=np.float32)
    padded[: waveform.shape[0]] = waveform
    return padded


def normalize_waveform(waveform: np.ndarray) -> np.ndarray:
    """Peak-normalize a waveform while guarding against silent inputs."""
    validate_non_empty_waveform(waveform)
    peak = float(np.max(np.abs(waveform)))
    if peak < 1e-8:
        return waveform.astype(np.float32)
    return (waveform / peak).astype(np.float32)


def extract_random_segment(waveform: np.ndarray, target_length: int, rng: np.random.Generator) -> tuple[np.ndarray, int]:
    """Sample a random contiguous segment and return the segment plus its start index."""
    validate_non_empty_waveform(waveform)
    validate_positive_int("target_length", target_length)
    if waveform.shape[0] <= target_length:
        return trim_or_pad_waveform(waveform, target_length), 0
    max_start = waveform.shape[0] - target_length
    start_index = int(rng.integers(0, max_start + 1))
    segment = waveform[start_index : start_index + target_length]
    return segment.astype(np.float32), start_index


def extract_center_segment(waveform: np.ndarray, target_length: int) -> tuple[np.ndarray, int]:
    """Extract the centered segment used for deterministic retrieval evaluation."""
    validate_non_empty_waveform(waveform)
    validate_positive_int("target_length", target_length)
    if waveform.shape[0] <= target_length:
        return trim_or_pad_waveform(waveform, target_length), 0
    start_index = (waveform.shape[0] - target_length) // 2
    segment = waveform[start_index : start_index + target_length]
    return segment.astype(np.float32), start_index

def extract_segment_by_start(waveform: np.ndarray, start_sample: int, target_length: int) -> np.ndarray:
    """Extract a deterministic contiguous segment starting at `start_sample`.

    If the requested window extends past the end of the waveform, the result is
    zero-padded to exactly `target_length` samples.
    """
    validate_non_empty_waveform(waveform)
    validate_positive_int("target_length", target_length)

    if start_sample < 0:
        raise ValueError(f"start_sample must be non-negative, received {start_sample}.")

    if start_sample >= waveform.shape[0]:
        return np.zeros(target_length, dtype=np.float32)

    segment = waveform[start_sample : start_sample + target_length]
    return trim_or_pad_waveform(segment, target_length).astype(np.float32, copy=False)

@lru_cache(maxsize=None)
def build_mel_transform(sample_rate: int, n_fft: int, hop_length: int, n_mels: int) -> torchaudio.transforms.MelSpectrogram:
    """Create and cache the mel spectrogram transform used across the notebook."""
    return torchaudio.transforms.MelSpectrogram(
        sample_rate=sample_rate,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        power=2.0,
    )


@lru_cache(maxsize=1)
def build_db_transform() -> torchaudio.transforms.AmplitudeToDB:
    """Create and cache the amplitude-to-decibel transform."""
    return torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80.0)

def compute_log_mel_spectrogram(
    waveform: np.ndarray,
    sample_rate: int,
    n_fft: int,
    hop_length: int,
    n_mels: int,
) -> np.ndarray:
    """Convert a raw waveform into a log-mel spectrogram using torchaudio."""
    validate_non_empty_waveform(waveform)
    validate_positive_int("sample_rate", sample_rate)
    validate_positive_int("n_fft", n_fft)
    validate_positive_int("hop_length", hop_length)
    validate_positive_int("n_mels", n_mels)

    waveform_tensor = torch.from_numpy(waveform).float().unsqueeze(0)
    mel_transform = build_mel_transform(sample_rate, n_fft, hop_length, n_mels)
    db_transform = build_db_transform()

    mel = mel_transform(waveform_tensor)
    log_mel = db_transform(mel)

    return log_mel.squeeze(0).cpu().numpy().astype(np.float32)


def normalize_spectrogram(spectrogram: np.ndarray) -> np.ndarray:
    """Standardize a spectrogram on a per-sample basis."""
    if spectrogram.ndim != 2:
        raise ValueError(f"Spectrogram must be 2-D, received shape {spectrogram.shape}.")
    mean = float(spectrogram.mean())
    std = float(spectrogram.std())
    if std < 1e-6:
        return (spectrogram - mean).astype(np.float32)
    return ((spectrogram - mean) / std).astype(np.float32)


def spectrogram_tensor_from_waveform(waveform: np.ndarray, config: ExperimentConfig) -> torch.Tensor:
    """Create a normalized `(1, n_mels, time_frames)` tensor from a waveform segment."""
    spectrogram = compute_log_mel_spectrogram(
        waveform=waveform,
        sample_rate=config.sample_rate,
        n_fft=config.n_fft,
        hop_length=config.hop_length,
        n_mels=config.n_mels,
    )
    spectrogram = normalize_spectrogram(spectrogram)
    return torch.from_numpy(spectrogram).unsqueeze(0).float()


## Augmentations

Two augmentation presets are kept on purpose:
- `baseline`: random gain, additive noise, mild time stretch, and mild pitch shift
- `extended`: the baseline stack plus low-pass and high-pass filtering

Every augmentation validates its parameters and trims or pads back to the configured segment length immediately after mutation.

### Augmentation helpers and visualization

In [ ]:
def add_gaussian_noise(waveform: np.ndarray, rng: np.random.Generator, snr_db: float) -> np.ndarray:
    """Add Gaussian noise at a specified signal-to-noise ratio."""
    if snr_db <= 0:
        raise ValueError(f"snr_db must be positive, received {snr_db}.")
    signal_power = float(np.mean(np.square(waveform)))
    noise_power = signal_power / (10.0 ** (snr_db / 10.0))
    noise = rng.normal(0.0, np.sqrt(noise_power), size=waveform.shape[0]).astype(np.float32)
    return (waveform + noise).astype(np.float32)


def apply_random_gain(waveform: np.ndarray, rng: np.random.Generator, min_db: float, max_db: float) -> np.ndarray:
    """Apply a random volume change in decibels."""
    if min_db > max_db:
        raise ValueError("min_db must be <= max_db.")
    gain_db = float(rng.uniform(min_db, max_db))
    gain = 10.0 ** (gain_db / 20.0)
    return (waveform * gain).astype(np.float32)


def apply_time_stretch(waveform: np.ndarray, rate: float) -> np.ndarray:
    """Time-stretch a waveform without changing pitch."""
    if rate <= 0:
        raise ValueError(f"rate must be positive, received {rate}.")
    return librosa.effects.time_stretch(waveform, rate=rate).astype(np.float32)


def apply_pitch_shift(waveform: np.ndarray, sample_rate: int, n_steps: float) -> np.ndarray:
    """Pitch-shift a waveform by the requested number of semitones."""
    return librosa.effects.pitch_shift(waveform, sr=sample_rate, n_steps=n_steps).astype(np.float32)


def apply_lowpass_filter(waveform: np.ndarray, sample_rate: int, cutoff_hz: float, order: int = 5) -> np.ndarray:
    """Apply a Butterworth low-pass filter."""
    nyquist = sample_rate / 2.0
    if cutoff_hz <= 0 or cutoff_hz >= nyquist:
        raise ValueError(f"cutoff_hz must be in (0, {nyquist}), received {cutoff_hz}.")
    b, a = scipy.signal.butter(order, cutoff_hz / nyquist, btype="low")
    return scipy.signal.filtfilt(b, a, waveform).astype(np.float32)


def apply_highpass_filter(waveform: np.ndarray, sample_rate: int, cutoff_hz: float, order: int = 5) -> np.ndarray:
    """Apply a Butterworth high-pass filter."""
    nyquist = sample_rate / 2.0
    if cutoff_hz <= 0 or cutoff_hz >= nyquist:
        raise ValueError(f"cutoff_hz must be in (0, {nyquist}), received {cutoff_hz}.")
    b, a = scipy.signal.butter(order, cutoff_hz / nyquist, btype="high")
    return scipy.signal.filtfilt(b, a, waveform).astype(np.float32)


def apply_augmentation_profile(
    waveform: np.ndarray,
    sample_rate: int,
    target_length: int,
    profile: AugmentationProfileName,
    rng: np.random.Generator,
) -> np.ndarray:
    """Apply the selected augmentation profile to a waveform segment."""
    segment = trim_or_pad_waveform(normalize_waveform(waveform), target_length)
    segment = trim_or_pad_waveform(apply_random_gain(segment, rng=rng, min_db=-6.0, max_db=6.0), target_length)
    segment = trim_or_pad_waveform(add_gaussian_noise(segment, rng=rng, snr_db=float(rng.uniform(10.0, 25.0))), target_length)
    segment = trim_or_pad_waveform(apply_time_stretch(segment, rate=float(rng.uniform(0.92, 1.08))), target_length)
    segment = trim_or_pad_waveform(apply_pitch_shift(segment, sample_rate=sample_rate, n_steps=float(rng.uniform(-2.0, 2.0))), target_length)

    if profile == "extended":
        segment = trim_or_pad_waveform(
            apply_lowpass_filter(segment, sample_rate=sample_rate, cutoff_hz=float(rng.uniform(2500.0, 5000.0))),
            target_length,
        )
        segment = trim_or_pad_waveform(
            apply_highpass_filter(segment, sample_rate=sample_rate, cutoff_hz=float(rng.uniform(120.0, 900.0))),
            target_length,
        )

    return normalize_waveform(segment)


def apply_retrieval_degradation(
    waveform: np.ndarray,
    sample_rate: int,
    target_length: int,
    condition: RetrievalConditionName,
    rng: np.random.Generator,
) -> np.ndarray:
    """Apply a deterministic query degradation for retrieval evaluation."""
    segment = trim_or_pad_waveform(normalize_waveform(waveform), target_length)
    if condition == "clean":
        return segment
    if condition == "noise":
        return trim_or_pad_waveform(add_gaussian_noise(segment, rng=rng, snr_db=8.0), target_length)
    if condition == "pitch":
        return trim_or_pad_waveform(apply_pitch_shift(segment, sample_rate=sample_rate, n_steps=2.0), target_length)
    if condition == "time":
        return trim_or_pad_waveform(apply_time_stretch(segment, rate=1.10), target_length)
    if condition == "lowpass":
        return trim_or_pad_waveform(apply_lowpass_filter(segment, sample_rate=sample_rate, cutoff_hz=3000.0), target_length)
    if condition == "highpass":
        return trim_or_pad_waveform(apply_highpass_filter(segment, sample_rate=sample_rate, cutoff_hz=500.0), target_length)
    raise ValueError(f"Unsupported retrieval condition: {condition}")


def visualize_preprocessing_example(config: ExperimentConfig, tracks_subset: pd.DataFrame) -> None:
    """Plot one clean segment and one extended-augmentation segment for sanity checking."""
    example_track_id = int(tracks_subset.index[0])
    waveform, _ = load_audio_file(
        get_audio_path(config.paths.audio_dir, example_track_id),
        target_sr=config.sample_rate,
        mono=True,
        cache_dir=config.paths.waveform_cache_dir,
    )
    clean_segment, _ = extract_center_segment(waveform, config.segment_samples)
    augmented_segment = apply_augmentation_profile(
        waveform=clean_segment,
        sample_rate=config.sample_rate,
        target_length=config.segment_samples,
        profile="extended",
        rng=np.random.default_rng(config.seed),
    )
    clean_spec = compute_log_mel_spectrogram(clean_segment, config.sample_rate, config.n_fft, config.hop_length, config.n_mels)
    augmented_spec = compute_log_mel_spectrogram(augmented_segment, config.sample_rate, config.n_fft, config.hop_length, config.n_mels)

    fig, axes = plt.subplots(2, 2, figsize=(16, 8))
    axes[0, 0].plot(clean_segment)
    axes[0, 0].set_title(f"Clean waveform | track {example_track_id}")
    axes[0, 1].plot(augmented_segment)
    axes[0, 1].set_title("Extended-augmentation waveform")
    axes[1, 0].imshow(clean_spec, origin="lower", aspect="auto", cmap="magma")
    axes[1, 0].set_title("Clean log-mel spectrogram")
    axes[1, 1].imshow(augmented_spec, origin="lower", aspect="auto", cmap="magma")
    axes[1, 1].set_title("Augmented log-mel spectrogram")
    plt.tight_layout()
    plt.show()


visualize_preprocessing_example(CONFIG, tracks_small)


## Datasets And Loaders

The training set yields anchor and positive pairs for contrastive learning. Retrieval datasets produce one deterministic reference or query segment per track. Spectrogram models receive `(1, n_mels, time_frames)` tensors, while MERT receives raw waveforms and uses a custom collate function built around `Wav2Vec2FeatureExtractor`.

### Dataset classes, collate functions, and split helpers

In [ ]:
class PairBatch(TypedDict):
    anchor: torch.Tensor
    positive: torch.Tensor
    track_id: torch.Tensor
    anchor_start_time: torch.Tensor


class RetrievalBatch(TypedDict, total=False):
    inputs: torch.Tensor
    attention_mask: torch.Tensor
    track_id: torch.Tensor
    metadata_rows: list[dict[str, object]]


def seed_worker(worker_id: int) -> None:
    """Seed NumPy and Python RNGs inside a DataLoader worker."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def build_torch_generator(seed: int) -> torch.Generator:
    """Build a deterministic PyTorch generator for DataLoader shuffling."""
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


class ContrastivePairDataset(Dataset[dict[str, object]]):
    """Yield anchor and positive pairs for contrastive training."""

    def __init__(
        self,
        audio_dir: Path,
        track_ids: list[int],
        segment_length: int,
        sample_rate: int,
        input_mode: InputMode,
        config: ExperimentConfig,
        augmentation_profile: AugmentationProfileName,
    ) -> None:
        self.audio_dir = audio_dir
        self.track_ids = track_ids
        self.segment_length = segment_length
        self.sample_rate = sample_rate
        self.input_mode = input_mode
        self.config = config
        self.augmentation_profile = augmentation_profile

    def __len__(self) -> int:
        return len(self.track_ids)

    def __getitem__(self, index: int) -> dict[str, object]:
        track_id = self.track_ids[index]
        seed = int(torch.randint(low=0, high=2**31 - 1, size=(1,)).item())
        rng = np.random.default_rng(seed)
        waveform, _ = load_audio_file(
            get_audio_path(self.audio_dir, track_id),
            target_sr=self.sample_rate,
            mono=True,
            cache_dir=self.config.paths.waveform_cache_dir,
        )
        segment, start_index = extract_random_segment(waveform, self.segment_length, rng=rng)
        segment = normalize_waveform(trim_or_pad_waveform(segment, self.segment_length))
        positive_waveform = apply_augmentation_profile(
            waveform=segment.copy(),
            sample_rate=self.sample_rate,
            target_length=self.segment_length,
            profile=self.augmentation_profile,
            rng=rng,
        )

        if self.input_mode == "spectrogram":
            anchor_value: torch.Tensor = spectrogram_tensor_from_waveform(segment, self.config)
            positive_value: torch.Tensor = spectrogram_tensor_from_waveform(positive_waveform, self.config)
        else:
            anchor_value = torch.from_numpy(segment).float()
            positive_value = torch.from_numpy(positive_waveform).float()

        return {
            "anchor": anchor_value,
            "positive": positive_value,
            "track_id": int(track_id),
            "anchor_start_time": float(start_index / self.sample_rate),
        }


class RetrievalReferenceDataset(Dataset[dict[str, object]]):
    """Yield deterministic reference segments for FAISS indexing."""

    def __init__(
        self,
        audio_dir: Path,
        track_ids: list[int],
        segment_length: int,
        sample_rate: int,
        input_mode: InputMode,
        config: ExperimentConfig,
    ) -> None:
        self.audio_dir = audio_dir
        self.track_ids = track_ids
        self.segment_length = segment_length
        self.sample_rate = sample_rate
        self.input_mode = input_mode
        self.config = config

    def __len__(self) -> int:
        return len(self.track_ids)

    def __getitem__(self, index: int) -> dict[str, object]:
        track_id = self.track_ids[index]
        waveform, _ = load_audio_file(
            get_audio_path(self.audio_dir, track_id),
            target_sr=self.sample_rate,
            mono=True,
            cache_dir=self.config.paths.waveform_cache_dir,
        )
        segment, _ = extract_center_segment(waveform, self.segment_length)
        if self.input_mode == "spectrogram":
            model_input: torch.Tensor = spectrogram_tensor_from_waveform(segment, self.config)
        else:
            model_input = torch.from_numpy(segment).float()
        return {"inputs": model_input, "track_id": int(track_id)}


class RetrievalQueryDataset(Dataset[dict[str, object]]):
    """Yield deterministic degraded query segments for retrieval evaluation."""

    def __init__(
        self,
        audio_dir: Path,
        track_ids: list[int],
        segment_length: int,
        sample_rate: int,
        input_mode: InputMode,
        config: ExperimentConfig,
        condition: RetrievalConditionName,
    ) -> None:
        self.audio_dir = audio_dir
        self.track_ids = track_ids
        self.segment_length = segment_length
        self.sample_rate = sample_rate
        self.input_mode = input_mode
        self.config = config
        self.condition = condition

    def __len__(self) -> int:
        return len(self.track_ids)

    def __getitem__(self, index: int) -> dict[str, object]:
        track_id = self.track_ids[index]
        rng = np.random.default_rng(self.config.seed + index)
        waveform, _ = load_audio_file(
            get_audio_path(self.audio_dir, track_id),
            target_sr=self.sample_rate,
            mono=True,
            cache_dir=self.config.paths.waveform_cache_dir,
        )
        segment, _ = extract_center_segment(waveform, self.segment_length)
        degraded = apply_retrieval_degradation(
            waveform=segment,
            sample_rate=self.sample_rate,
            target_length=self.segment_length,
            condition=self.condition,
            rng=rng,
        )
        if self.input_mode == "spectrogram":
            model_input: torch.Tensor = spectrogram_tensor_from_waveform(degraded, self.config)
        else:
            model_input = torch.from_numpy(degraded).float()
        return {"inputs": model_input, "track_id": int(track_id)}


def collate_pair_batch(batch: list[dict[str, object]]) -> PairBatch:
    """Stack spectrogram or waveform pair samples into one batch."""
    anchors = torch.stack([sample["anchor"] for sample in batch])
    positives = torch.stack([sample["positive"] for sample in batch])
    track_ids = torch.tensor([int(sample["track_id"]) for sample in batch], dtype=torch.long)
    start_times = torch.tensor([float(sample["anchor_start_time"]) for sample in batch], dtype=torch.float32)
    return {
        "anchor": anchors,
        "positive": positives,
        "track_id": track_ids,
        "anchor_start_time": start_times,
    }


def collate_retrieval_batch(batch: list[dict[str, object]]) -> RetrievalBatch:
    """Stack spectrogram or waveform retrieval samples into one batch."""
    inputs = torch.stack([sample["inputs"] for sample in batch])
    track_ids = torch.tensor([int(sample["track_id"]) for sample in batch], dtype=torch.long)
    return {"inputs": inputs, "track_id": track_ids}


def build_mert_pair_collate_fn(feature_extractor: Wav2Vec2FeatureExtractor) -> Callable[[list[dict[str, object]]], dict[str, torch.Tensor]]:
    """Build the MERT pair collate function."""

    def collate(batch: list[dict[str, object]]) -> dict[str, torch.Tensor]:
        anchor_arrays = [np.asarray(sample["anchor"], dtype=np.float32) for sample in batch]
        positive_arrays = [np.asarray(sample["positive"], dtype=np.float32) for sample in batch]
        anchor_inputs = feature_extractor(anchor_arrays, sampling_rate=feature_extractor.sampling_rate, return_tensors="pt", padding=True)
        positive_inputs = feature_extractor(positive_arrays, sampling_rate=feature_extractor.sampling_rate, return_tensors="pt", padding=True)
        return {
            "anchor_input_values": anchor_inputs["input_values"],
            "anchor_attention_mask": anchor_inputs["attention_mask"],
            "positive_input_values": positive_inputs["input_values"],
            "positive_attention_mask": positive_inputs["attention_mask"],
            "track_id": torch.tensor([int(sample["track_id"]) for sample in batch], dtype=torch.long),
            "anchor_start_time": torch.tensor([float(sample["anchor_start_time"]) for sample in batch], dtype=torch.float32),
        }

    return collate


def build_mert_retrieval_collate_fn(
    feature_extractor: Wav2Vec2FeatureExtractor,
) -> Callable[[list[dict[str, object]]], dict[str, object]]:
    """Build the MERT retrieval collate function."""

    def collate(batch: list[dict[str, object]]) -> dict[str, object]:
        waveform_arrays = [np.asarray(sample["inputs"], dtype=np.float32) for sample in batch]
        feature_batch = feature_extractor(
            waveform_arrays,
            sampling_rate=feature_extractor.sampling_rate,
            return_tensors="pt",
            padding=True,
        )
        return {
            "inputs": feature_batch["input_values"],
            "attention_mask": feature_batch["attention_mask"],
            "track_id": torch.tensor([int(sample["track_id"]) for sample in batch], dtype=torch.long),
            "metadata_rows": [dict(sample["metadata_row"]) for sample in batch],
        }

    return collate


def get_split_track_ids(
    subset_tracks: pd.DataFrame,
    max_train_tracks: int | None = None,
    max_eval_tracks: int | None = None,
) -> dict[str, list[int]]:
    """Collect track IDs for the training, validation, and test splits."""
    training_ids = [int(track_id) for track_id in subset_tracks[subset_tracks["set", "split"] == "training"].index.tolist()]
    validation_ids = [int(track_id) for track_id in subset_tracks[subset_tracks["set", "split"] == "validation"].index.tolist()]
    test_ids = [int(track_id) for track_id in subset_tracks[subset_tracks["set", "split"] == "test"].index.tolist()]

    if max_train_tracks is not None:
        training_ids = training_ids[:max_train_tracks]
    if max_eval_tracks is not None:
        validation_ids = validation_ids[:max_eval_tracks]
        test_ids = test_ids[:max_eval_tracks]

    return {
        "training": training_ids,
        "validation": validation_ids,
        "test": test_ids,
    }


SPLIT_TRACK_IDS = get_split_track_ids(
    tracks_small,
    max_train_tracks=CONFIG.max_train_tracks,
    max_eval_tracks=CONFIG.max_eval_tracks,
)
{split_name: len(track_ids) for split_name, track_ids in SPLIT_TRACK_IDS.items()}


## Minimal Baseline: Loss And Models

This section defines the shared NT-Xent loss, the compact CNN baseline, the hybrid spectrogram transformer, and the frozen MERT fingerprinter. All three models emit L2-normalized embeddings so FAISS can use exact inner-product search as cosine-equivalent retrieval.

### Loss and model definitions

In [ ]:
class NTXentLoss(nn.Module):
    """Symmetric NT-Xent loss for contrastive representation learning."""

    def __init__(self, temperature: float) -> None:
        super().__init__()
        if temperature <= 0:
            raise ValueError("temperature must be positive.")
        self.temperature = temperature

    def forward(self, anchor_embeddings: torch.Tensor, positive_embeddings: torch.Tensor) -> torch.Tensor:
        batch_size = anchor_embeddings.shape[0]
        if batch_size < 2:
            raise ValueError("NT-Xent requires a batch size of at least 2.")
        anchor_embeddings = F.normalize(anchor_embeddings, p=2, dim=1)
        positive_embeddings = F.normalize(positive_embeddings, p=2, dim=1)
        logits = anchor_embeddings @ positive_embeddings.T / self.temperature
        labels = torch.arange(batch_size, device=logits.device)
        loss_anchor = F.cross_entropy(logits, labels)
        loss_positive = F.cross_entropy(logits.T, labels)
        return 0.5 * (loss_anchor + loss_positive)


class ProjectionHead(nn.Module):
    """Two-layer projection head used by every fingerprinting model."""

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        embeddings = self.layers(inputs)
        return F.normalize(embeddings, p=2, dim=1)


class ConvBlock(nn.Module):
    """A simple convolution, normalization, and activation block."""

    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.block(inputs)


class CompactSpectrogramCNN(nn.Module):
    """Compact spectrogram CNN baseline for contrastive fingerprinting."""

    def __init__(self, embed_dim: int, projection_hidden_dim: int) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(1, 32, stride=2),
            ConvBlock(32, 64, stride=2),
            ConvBlock(64, 128, stride=2),
            ConvBlock(128, 256, stride=2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.projection = ProjectionHead(256, projection_hidden_dim, embed_dim)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        features = self.encoder(inputs)
        pooled = self.pool(features).flatten(start_dim=1)
        return self.projection(pooled)


class TransformerEncoderStack(nn.Module):
    """A small Transformer encoder with optional activation checkpointing."""

    def __init__(self, d_model: int, num_heads: int, num_layers: int, dropout: float, use_gradient_checkpointing: bool) -> None:
        super().__init__()
        self.layers = nn.ModuleList(
            [
                nn.TransformerEncoderLayer(
                    d_model=d_model,
                    nhead=num_heads,
                    dim_feedforward=d_model * 4,
                    dropout=dropout,
                    activation="gelu",
                    batch_first=True,
                    norm_first=True,
                )
                for _ in range(num_layers)
            ]
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.use_gradient_checkpointing = use_gradient_checkpointing

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden_states = inputs
        for layer in self.layers:
            if self.use_gradient_checkpointing and self.training:
                hidden_states = checkpoint(lambda tensor: layer(tensor), hidden_states, use_reentrant=False)
            else:
                hidden_states = layer(hidden_states)
        return self.final_norm(hidden_states)


class HybridSpectrogramTransformer(nn.Module):
    """Hybrid spectrogram transformer used as the main scratch model."""

    def __init__(self, config: ExperimentConfig, embed_dim: int) -> None:
        super().__init__()
        self.config = config
        self.conv_stem = nn.Sequential(
            ConvBlock(1, 64, stride=2),
            ConvBlock(64, 128, stride=2),
        )
        self.patch_projection = nn.Conv2d(128, 256, kernel_size=4, stride=4)
        self.encoder = TransformerEncoderStack(256, 4, 4, 0.1, config.gradient_checkpointing)
        token_count = self._compute_token_count(config.n_mels, config.spectrogram_frames)
        self.position_embeddings = nn.Parameter(torch.zeros(1, token_count, 256))
        nn.init.trunc_normal_(self.position_embeddings, std=0.02)
        self.projection = ProjectionHead(256, config.projection_hidden_dim, embed_dim)

    @staticmethod
    def _compute_token_count(input_height: int, input_width: int) -> int:
        """Compute token count after the conv stem and patch projection."""
        height_after_stem = (input_height + 1) // 2
        height_after_stem = (height_after_stem + 1) // 2
        width_after_stem = (input_width + 1) // 2
        width_after_stem = (width_after_stem + 1) // 2
        height_after_patch = ((height_after_stem - 4) // 4) + 1
        width_after_patch = ((width_after_stem - 4) // 4) + 1
        return height_after_patch * width_after_patch

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden = self.conv_stem(inputs)
        hidden = self.patch_projection(hidden)
        tokens = hidden.flatten(start_dim=2).transpose(1, 2)
        tokens = tokens + self.position_embeddings[:, : tokens.shape[1], :]
        encoded = self.encoder(tokens)
        pooled = encoded.mean(dim=1)
        return self.projection(pooled)


class FrozenMertFingerprinter(nn.Module):
    """Frozen MERT backbone with a trainable projection head."""

    def __init__(self, model_name: str, embed_dim: int, projection_hidden_dim: int, freeze_backbone: bool = True, weighted_layer_pooling: bool = False) -> None:
        super().__init__()
        self.model_name = model_name
        self.freeze_backbone = freeze_backbone
        self.weighted_layer_pooling = weighted_layer_pooling
        self.feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
        self.backbone = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        self.hidden_size = int(self.backbone.config.hidden_size)
        self.projection = ProjectionHead(self.hidden_size, projection_hidden_dim, embed_dim)

        if self.weighted_layer_pooling:
            layer_count = int(self.backbone.config.num_hidden_layers) + 1
            self.layer_weights = nn.Parameter(torch.ones(layer_count) / layer_count)
        else:
            self.layer_weights = None

        if self.freeze_backbone:
            for parameter in self.backbone.parameters():
                parameter.requires_grad = False
            self.backbone.eval()

    @property
    def sample_rate(self) -> int:
        return int(self.feature_extractor.sampling_rate)

    def train(self, mode: bool = True) -> "FrozenMertFingerprinter":
        super().train(mode)
        if self.freeze_backbone:
            self.backbone.eval()
        return self

    def _pool_hidden_states(self, outputs: object, attention_mask: torch.Tensor | None) -> torch.Tensor:
        """Pool hidden states with attention-mask awareness.

        For MERT/Hubert-style backbones, the raw waveform attention mask is longer
        than the encoder sequence length because the convolutional feature extractor
        downsamples the time axis. In that case, reduce the mask to feature-vector
        length before masked pooling.
        """
        if self.weighted_layer_pooling:
            hidden_states = getattr(outputs, "hidden_states")
            stacked_hidden_states = torch.stack(list(hidden_states), dim=0)
            normalized_weights = torch.softmax(self.layer_weights, dim=0).view(-1, 1, 1, 1)
            sequence_output = torch.sum(normalized_weights * stacked_hidden_states, dim=0)
        else:
            sequence_output = getattr(outputs, "last_hidden_state")

        if attention_mask is None:
            return sequence_output.mean(dim=1)

        if attention_mask.shape[1] != sequence_output.shape[1]:
            if not hasattr(self.backbone, "_get_feature_vector_attention_mask"):
                raise RuntimeError(
                    "Attention-mask length does not match hidden-state sequence length "
                    f"({attention_mask.shape[1]} vs {sequence_output.shape[1]}), and the "
                  "backbone does not expose _get_feature_vector_attention_mask()."
                )

            attention_mask = self.backbone._get_feature_vector_attention_mask(
                sequence_output.shape[1],
                attention_mask,
            )

        mask = attention_mask.unsqueeze(-1).to(sequence_output.dtype)
        masked_sum = torch.sum(sequence_output * mask, dim=1)
        mask_sum = torch.clamp(mask.sum(dim=1), min=1.0)
        return masked_sum / mask_sum

    def forward(self, input_values: torch.Tensor, attention_mask: torch.Tensor | None = None) -> torch.Tensor:
        backbone_kwargs = {
            "input_values": input_values,
            "attention_mask": attention_mask,
            "output_hidden_states": self.weighted_layer_pooling,
        }
        if self.freeze_backbone:
            with torch.no_grad():
                outputs = self.backbone(**backbone_kwargs)
        else:
            outputs = self.backbone(**backbone_kwargs)
        pooled = self._pool_hidden_states(outputs, attention_mask=attention_mask)
        return self.projection(pooled)


## Training, Retrieval, And Artifacts

The fixed execution order is:
1. dataset and model smoke tests
2. CNN baseline
3. hybrid transformer with baseline augmentation
4. hybrid transformer with extended augmentation
5. hybrid transformer with extended augmentation and a larger embedding
6. frozen MERT baseline

Exact retrieval uses `faiss.IndexFlatIP` over L2-normalized embeddings stored as `float32`.

### Training, retrieval, and plotting helpers

In [ ]:
def serialize_jsonable(value: object) -> object:
    """Recursively convert dataclasses and paths into JSON-safe values."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, tuple):
        return [serialize_jsonable(item) for item in value]
    if isinstance(value, list):
        return [serialize_jsonable(item) for item in value]
    if isinstance(value, dict):
        return {str(key): serialize_jsonable(item) for key, item in value.items()}
    if hasattr(value, "__dataclass_fields__"):
        return {field_name: serialize_jsonable(getattr(value, field_name)) for field_name in value.__dataclass_fields__.keys()}
    return value


def make_run_directory(config: ExperimentConfig, run_config: ModelRunConfig) -> Path:
    """Create the output directory for a single run."""
    run_dir = config.paths.output_root / run_config.run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir


def clear_memory() -> None:
    """Release cached Python and CUDA memory between runs."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def build_model(config: ExperimentConfig, run_config: ModelRunConfig) -> tuple[nn.Module, InputMode, int, Callable[[list[dict[str, object]]], dict[str, torch.Tensor]] | None]:
    """Instantiate a model and return its input mode and optional collate function."""
    if run_config.model_name == "cnn":
        model = CompactSpectrogramCNN(run_config.embed_dim, config.projection_hidden_dim)
        return model, "spectrogram", config.sample_rate, None

    if run_config.model_name == "hybrid_transformer":
        model = HybridSpectrogramTransformer(config, run_config.embed_dim)
        return model, "spectrogram", config.sample_rate, None

    model = FrozenMertFingerprinter("m-a-p/MERT-v1-95M", run_config.embed_dim, config.projection_hidden_dim, run_config.freeze_backbone, False)
    return model, "mert_waveform", model.sample_rate, build_mert_pair_collate_fn(model.feature_extractor)


def build_retrieval_collate_fn(model: nn.Module, input_mode: InputMode) -> Callable[[list[dict[str, object]]], dict[str, torch.Tensor]]:
    """Return the retrieval collate function for a model family."""
    if input_mode == "spectrogram":
        return collate_retrieval_batch
    if not isinstance(model, FrozenMertFingerprinter):
        raise TypeError("MERT retrieval collate requires a FrozenMertFingerprinter instance.")
    return build_mert_retrieval_collate_fn(model.feature_extractor)


def move_batch_to_device(batch: dict[str, torch.Tensor], device: torch.device) -> dict[str, torch.Tensor]:
    """Move all tensor values in a batch dictionary to the requested device."""
    return {key: value.to(device) if isinstance(value, torch.Tensor) else value for key, value in batch.items()}


def forward_pair_batch(model: nn.Module, batch: dict[str, torch.Tensor], input_mode: InputMode, device_type: str, amp_enabled: bool) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute anchor and positive embeddings for one training batch."""
    with torch.amp.autocast(device_type=device_type, enabled=amp_enabled):
        if input_mode == "spectrogram":
            anchor_embeddings = model(batch["anchor"])
            positive_embeddings = model(batch["positive"])
        else:
            if not isinstance(model, FrozenMertFingerprinter):
                raise TypeError("Waveform batches require FrozenMertFingerprinter.")
            anchor_embeddings = model(batch["anchor_input_values"], batch["anchor_attention_mask"])
            positive_embeddings = model(batch["positive_input_values"], batch["positive_attention_mask"])
    return anchor_embeddings, positive_embeddings


def encode_retrieval_batch(model: nn.Module, batch: dict[str, torch.Tensor], input_mode: InputMode, device: torch.device, device_type: str, amp_enabled: bool) -> torch.Tensor:
    """Encode one retrieval batch into embeddings."""
    with torch.no_grad():
        with torch.amp.autocast(device_type=device_type, enabled=amp_enabled):
            if input_mode == "spectrogram":
                return model(batch["inputs"].to(device))
            if not isinstance(model, FrozenMertFingerprinter):
                raise TypeError("Waveform retrieval batches require FrozenMertFingerprinter.")
            return model(batch["inputs"].to(device), batch["attention_mask"].to(device))


def train_one_epoch(model: nn.Module, loader: DataLoader[dict[str, torch.Tensor]], optimizer: torch.optim.Optimizer, loss_fn: NTXentLoss, device: torch.device, input_mode: InputMode, config: ExperimentConfig) -> float:
    """Train a model for one epoch and return the mean loss."""
    model.train()
    if isinstance(model, FrozenMertFingerprinter) and model.freeze_backbone:
        model.backbone.eval()
    scaler = torch.amp.GradScaler(config.device_type, enabled=config.amp_enabled)
    epoch_losses: list[float] = []

    for batch in tqdm(loader, desc="train", leave=False):
        batch = move_batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)
        anchor_embeddings, positive_embeddings = forward_pair_batch(model, batch, input_mode, config.device_type, config.amp_enabled)
        loss = loss_fn(anchor_embeddings, positive_embeddings)
        if not torch.isfinite(loss):
            raise FloatingPointError("Encountered a non-finite training loss.")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.gradient_clip_norm)
        scaler.step(optimizer)
        scaler.update()
        epoch_losses.append(float(loss.detach().cpu().item()))

    return float(np.mean(epoch_losses))


def validate_epoch(model: nn.Module, loader: DataLoader[dict[str, torch.Tensor]], loss_fn: NTXentLoss, device: torch.device, input_mode: InputMode, config: ExperimentConfig) -> float:
    """Evaluate mean contrastive loss on the validation loader."""
    model.eval()
    losses: list[float] = []
    for batch in tqdm(loader, desc="validate", leave=False):
        batch = move_batch_to_device(batch, device)
        anchor_embeddings, positive_embeddings = forward_pair_batch(model, batch, input_mode, config.device_type, config.amp_enabled)
        loss = loss_fn(anchor_embeddings, positive_embeddings)
        losses.append(float(loss.detach().cpu().item()))
    return float(np.mean(losses))


### Embedding extraction, retrieval metrics, and artifact writers

In [ ]:
def extract_embeddings(model: nn.Module, loader: DataLoader[dict[str, torch.Tensor]], input_mode: InputMode, device: torch.device, config: ExperimentConfig) -> tuple[np.ndarray, np.ndarray]:
    """Encode a loader into a NumPy embedding matrix and aligned track IDs."""
    model.eval()
    embedding_chunks: list[np.ndarray] = []
    track_id_chunks: list[np.ndarray] = []
    for batch in tqdm(loader, desc="embed", leave=False):
        track_ids = batch["track_id"].detach().cpu().numpy().astype(np.int64)
        encoded = encode_retrieval_batch(model, batch, input_mode, device, config.device_type, config.amp_enabled)
        embedding_chunks.append(encoded.detach().cpu().numpy().astype(np.float32))
        track_id_chunks.append(track_ids)
    return np.concatenate(embedding_chunks, axis=0), np.concatenate(track_id_chunks, axis=0)


def build_faiss_index(reference_embeddings: np.ndarray) -> faiss.IndexFlatIP:
    """Build an exact FAISS index over L2-normalized float32 embeddings."""
    if reference_embeddings.dtype != np.float32:
        reference_embeddings = reference_embeddings.astype(np.float32)
    faiss.normalize_L2(reference_embeddings)
    index = faiss.IndexFlatIP(reference_embeddings.shape[1])
    index.add(reference_embeddings)
    return index


def compute_rank_metrics(ranked_reference_ids: np.ndarray, query_track_ids: np.ndarray, top_k: tuple[int, ...]) -> dict[str, float]:
    """Compute Top-k accuracy and MRR from ranked reference IDs."""
    metrics: dict[str, float] = {}
    reciprocal_ranks: list[float] = []
    for query_index, query_track_id in enumerate(query_track_ids):
        ranked_ids = ranked_reference_ids[query_index]
        match_indices = np.where(ranked_ids == query_track_id)[0]
        rank = int(match_indices[0]) + 1 if match_indices.size else ranked_ids.shape[0] + 1
        reciprocal_ranks.append(1.0 / rank)
        for k in top_k:
            metrics.setdefault(f"top{k}", 0.0)
            metrics[f"top{k}"] += 1.0 if rank <= k else 0.0

    query_count = float(len(query_track_ids))
    for k in top_k:
        metrics[f"top{k}"] /= query_count
    metrics["mrr"] = float(np.mean(reciprocal_ranks))
    return metrics


def evaluate_retrieval(model: nn.Module, reference_loader: DataLoader[dict[str, torch.Tensor]], query_loaders: dict[RetrievalConditionName, DataLoader[dict[str, torch.Tensor]]], input_mode: InputMode, device: torch.device, config: ExperimentConfig) -> tuple[dict[str, float], pd.DataFrame]:
    """Build a FAISS index and evaluate clean plus degraded retrieval metrics."""
    reference_embeddings, reference_track_ids = extract_embeddings(model, reference_loader, input_mode, device, config)
    index = build_faiss_index(reference_embeddings.copy())
    results: dict[str, float] = {}
    degradation_rows: list[dict[str, object]] = []

    for condition, query_loader in query_loaders.items():
        query_embeddings, query_track_ids = extract_embeddings(model, query_loader, input_mode, device, config)
        faiss.normalize_L2(query_embeddings)
        _, neighbor_indices = index.search(query_embeddings.astype(np.float32), max(config.top_k))
        ranked_reference_ids = reference_track_ids[neighbor_indices]
        condition_metrics = compute_rank_metrics(ranked_reference_ids, query_track_ids, config.top_k)
        degradation_rows.append(
            {
                "condition": condition,
                "top1": condition_metrics["top1"],
                "top5": condition_metrics["top5"],
                "top10": condition_metrics["top10"],
                "mrr": condition_metrics["mrr"],
            }
        )
        if condition == "clean":
            results["clean_top1"] = condition_metrics["top1"]
            results["clean_top5"] = condition_metrics["top5"]
            results["clean_top10"] = condition_metrics["top10"]
            results["clean_mrr"] = condition_metrics["mrr"]
        else:
            results[f"{condition}_top1"] = condition_metrics["top1"]

    return results, pd.DataFrame(degradation_rows)


def save_json(filepath: Path, payload: object) -> None:
    """Write a JSON artifact with stable indentation."""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with filepath.open("w", encoding="utf-8") as handle:
        json.dump(serialize_jsonable(payload), handle, indent=2)
        handle.write("\n")


def plot_training_history(history: list[dict[str, float]], output_path: Path, run_title: str) -> None:
    """Save a compact loss-curve plot for one experiment run."""
    epochs = [int(item["epoch"]) for item in history]
    train_losses = [float(item["train_loss"]) for item in history]
    val_losses = [float(item["val_loss"]) for item in history]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(epochs, train_losses, marker="o", label="Train loss")
    ax.plot(epochs, val_losses, marker="s", label="Validation loss")
    ax.set_title(f"Loss curves | {run_title}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("NT-Xent loss")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


def plot_retrieval_bars(degradation_df: pd.DataFrame, output_path: Path, run_title: str) -> None:
    """Save a degradation-specific Top-1 bar chart for one experiment run."""
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(degradation_df["condition"], degradation_df["top1"], color="steelblue")
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("Top-1 accuracy")
    ax.set_title(f"Retrieval robustness | {run_title}")
    ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


## Additions: Policies, Query Regimes, Multi-Window Indexing, And Caches

The next code cell is where this notebook stops being Notebook 2 with a few extra plots.

It adds:
- policy-driven augmentation beyond `baseline` and `extended`
- richer retrieval conditions and shorter/off-center query regimes
- multi-window reference manifests for realistic indexing
- cache-aware datasets and metadata-rich loaders


### Augmentation policies, query regimes, manifests, and retrieval datasets

In [ ]:
@dataclass(frozen=True)
class AugmentationTransformSpec:
    """A single augmentation transform and its parameter sampling policy."""

    name: str
    probability: float
    parameters: dict[str, float]
    order: int = 0


@dataclass(frozen=True)
class AugmentationPolicySpec:
    """A complete augmentation policy for contrastive positive generation."""

    name: str
    description: str
    selection_strategy: str
    transforms: tuple[AugmentationTransformSpec, ...]
    max_transforms_per_sample: int
    enabled: bool = True
    curriculum_transition_epoch: int | None = None


@dataclass(frozen=True)
class QueryRegimeSpec:
    """How to sample retrieval queries for one evaluation regime."""

    name: QueryRegimeName
    description: str
    lengths_seconds: tuple[float, ...]
    centered: bool
    random_offset: bool
    conditions: tuple[RetrievalConditionName, ...]
    group_size: int
    enabled: bool = True
    apply_fade_envelope: bool = False


@dataclass(frozen=True)
class WindowingSpec:
    """Reference-window construction strategy for indexing."""

    name: WindowingSpecName
    description: str
    window_count: int
    segment_seconds: float
    centered_only: bool
    enabled: bool = True


@dataclass(frozen=True)
class IndexSpec:
    """FAISS index configuration."""

    name: str
    kind: IndexKind
    description: str
    nprobe: int | None = None
    enabled: bool = True
    max_nlist: int = 64


def validate_probability(name: str, value: float) -> None:
    """Validate that a probability lies inside the closed interval [0, 1]."""
    if value < 0.0 or value > 1.0:
        raise ValueError(f"{name} must lie in [0, 1], received {value}.")


def apply_fade_envelope(waveform: np.ndarray, sample_rate: int, fade_seconds: float) -> np.ndarray:
    """Apply a symmetric fade-in/fade-out envelope."""
    validate_non_empty_waveform(waveform)
    fade_samples = min(int(max(0.0, fade_seconds) * sample_rate), waveform.shape[0] // 2)
    if fade_samples <= 1:
        return waveform.astype(np.float32)
    envelope = np.ones_like(waveform, dtype=np.float32)
    fade_curve = np.linspace(0.0, 1.0, fade_samples, dtype=np.float32)
    envelope[:fade_samples] = fade_curve
    envelope[-fade_samples:] = fade_curve[::-1]
    return (waveform * envelope).astype(np.float32)


def build_augmentation_policy_registry() -> dict[str, AugmentationPolicySpec]:
    """Construct the augmentation-policy registry."""
    registry = {
        "baseline": AugmentationPolicySpec(
            "baseline",
            "Exact Notebook 2 baseline policy.",
            "notebook2_exact",
            (
                AugmentationTransformSpec("gain", 1.0, {"min_db": -6.0, "max_db": 6.0}, 0),
                AugmentationTransformSpec("gaussian_noise", 1.0, {"min_snr_db": 10.0, "max_snr_db": 25.0}, 1),
                AugmentationTransformSpec("time_stretch", 1.0, {"min_rate": 0.92, "max_rate": 1.08}, 2),
                AugmentationTransformSpec("pitch_shift", 1.0, {"min_steps": -2.0, "max_steps": 2.0}, 3),
            ),
            4,
        ),
        "extended_existing": AugmentationPolicySpec(
            "extended_existing",
            "Exact Notebook 2 extended policy.",
            "notebook2_exact",
            (
                AugmentationTransformSpec("gain", 1.0, {"min_db": -6.0, "max_db": 6.0}, 0),
                AugmentationTransformSpec("gaussian_noise", 1.0, {"min_snr_db": 10.0, "max_snr_db": 25.0}, 1),
                AugmentationTransformSpec("time_stretch", 1.0, {"min_rate": 0.92, "max_rate": 1.08}, 2),
                AugmentationTransformSpec("pitch_shift", 1.0, {"min_steps": -2.0, "max_steps": 2.0}, 3),
                AugmentationTransformSpec("lowpass", 1.0, {"min_cutoff_hz": 2500.0, "max_cutoff_hz": 5000.0}, 4),
                AugmentationTransformSpec("highpass", 1.0, {"min_cutoff_hz": 120.0, "max_cutoff_hz": 900.0}, 5),
            ),
            6,
        ),
        "filters_only": AugmentationPolicySpec(
            "filters_only",
            "Only low-pass or high-pass filtering.",
            "up_to_k",
            (
                AugmentationTransformSpec("lowpass", 0.55, {"min_cutoff_hz": 2800.0, "max_cutoff_hz": 5200.0}, 0),
                AugmentationTransformSpec("highpass", 0.55, {"min_cutoff_hz": 160.0, "max_cutoff_hz": 720.0}, 1),
            ),
            1,
        ),
        "one_of_k": AugmentationPolicySpec(
            "one_of_k",
            "Apply exactly one augmentation from a broad candidate set.",
            "exactly_one",
            (
                AugmentationTransformSpec("gain", 1.0, {"min_db": -5.0, "max_db": 5.0}, 0),
                AugmentationTransformSpec("gaussian_noise", 1.0, {"min_snr_db": 12.0, "max_snr_db": 24.0}, 1),
                AugmentationTransformSpec("time_stretch", 1.0, {"min_rate": 0.94, "max_rate": 1.06}, 2),
                AugmentationTransformSpec("pitch_shift", 1.0, {"min_steps": -1.5, "max_steps": 1.5}, 3),
                AugmentationTransformSpec("lowpass", 1.0, {"min_cutoff_hz": 2600.0, "max_cutoff_hz": 4800.0}, 4),
                AugmentationTransformSpec("highpass", 1.0, {"min_cutoff_hz": 140.0, "max_cutoff_hz": 700.0}, 5),
            ),
            1,
        ),
        "two_of_k": AugmentationPolicySpec(
            "two_of_k",
            "Apply at most two augmentations from a broad candidate set.",
            "up_to_k",
            (
                AugmentationTransformSpec("gain", 0.45, {"min_db": -5.0, "max_db": 5.0}, 0),
                AugmentationTransformSpec("gaussian_noise", 0.60, {"min_snr_db": 12.0, "max_snr_db": 24.0}, 1),
                AugmentationTransformSpec("time_stretch", 0.40, {"min_rate": 0.95, "max_rate": 1.05}, 2),
                AugmentationTransformSpec("pitch_shift", 0.40, {"min_steps": -1.5, "max_steps": 1.5}, 3),
                AugmentationTransformSpec("lowpass", 0.35, {"min_cutoff_hz": 2800.0, "max_cutoff_hz": 5000.0}, 4),
                AugmentationTransformSpec("highpass", 0.35, {"min_cutoff_hz": 150.0, "max_cutoff_hz": 650.0}, 5),
            ),
            2,
            False,
        ),
        "severity_controlled": AugmentationPolicySpec(
            "severity_controlled",
            "Apply one or two milder transforms with narrower parameter ranges.",
            "up_to_k",
            (
                AugmentationTransformSpec("gain", 0.35, {"min_db": -3.0, "max_db": 3.0}, 0),
                AugmentationTransformSpec("gaussian_noise", 0.55, {"min_snr_db": 16.0, "max_snr_db": 24.0}, 1),
                AugmentationTransformSpec("time_stretch", 0.35, {"min_rate": 0.97, "max_rate": 1.03}, 2),
                AugmentationTransformSpec("pitch_shift", 0.35, {"min_steps": -1.0, "max_steps": 1.0}, 3),
                AugmentationTransformSpec("lowpass", 0.25, {"min_cutoff_hz": 3200.0, "max_cutoff_hz": 4600.0}, 4),
                AugmentationTransformSpec("highpass", 0.25, {"min_cutoff_hz": 180.0, "max_cutoff_hz": 520.0}, 5),
            ),
            2,
        ),
        "curriculum_filters_late": AugmentationPolicySpec(
            "curriculum_filters_late",
            "Start with baseline augmentation, then introduce filters later.",
            "curriculum",
            (
                AugmentationTransformSpec("lowpass", 0.45, {"min_cutoff_hz": 2800.0, "max_cutoff_hz": 5000.0}, 4),
                AugmentationTransformSpec("highpass", 0.45, {"min_cutoff_hz": 150.0, "max_cutoff_hz": 650.0}, 5),
            ),
            2,
            False,
            2,
        ),
    }
    for policy in registry.values():
        if policy.max_transforms_per_sample <= 0:
            raise ValueError(f"Policy {policy.name} must allow at least one transform.")
        for transform in policy.transforms:
            validate_probability(f"{policy.name}:{transform.name}", float(transform.probability))
    return registry


def sample_transform_parameters(transform: AugmentationTransformSpec, rng: np.random.Generator) -> dict[str, float]:
    """Sample concrete transform parameters from a transform specification."""
    sampled: dict[str, float] = {}
    if "min_db" in transform.parameters and "max_db" in transform.parameters:
        sampled["gain_db"] = float(rng.uniform(transform.parameters["min_db"], transform.parameters["max_db"]))
    if "min_snr_db" in transform.parameters and "max_snr_db" in transform.parameters:
        sampled["snr_db"] = float(rng.uniform(transform.parameters["min_snr_db"], transform.parameters["max_snr_db"]))
    if "min_rate" in transform.parameters and "max_rate" in transform.parameters:
        sampled["rate"] = float(rng.uniform(transform.parameters["min_rate"], transform.parameters["max_rate"]))
    if "min_steps" in transform.parameters and "max_steps" in transform.parameters:
        sampled["n_steps"] = float(rng.uniform(transform.parameters["min_steps"], transform.parameters["max_steps"]))
    if "min_cutoff_hz" in transform.parameters and "max_cutoff_hz" in transform.parameters:
        sampled["cutoff_hz"] = float(rng.uniform(transform.parameters["min_cutoff_hz"], transform.parameters["max_cutoff_hz"]))
    return sampled


def apply_transform_with_metadata(
    waveform: np.ndarray,
    sample_rate: int,
    target_length: int,
    transform: AugmentationTransformSpec,
    rng: np.random.Generator,
) -> tuple[np.ndarray, dict[str, object]]:
    """Apply one transform and return the transformed waveform plus metadata."""
    sampled = sample_transform_parameters(transform, rng)
    transformed = waveform.copy()
    if transform.name == "gain":
        transformed = apply_random_gain(transformed, rng=rng, min_db=sampled["gain_db"], max_db=sampled["gain_db"])
    elif transform.name == "gaussian_noise":
        transformed = add_gaussian_noise(transformed, rng=rng, snr_db=sampled["snr_db"])
    elif transform.name == "time_stretch":
        transformed = apply_time_stretch(transformed, rate=sampled["rate"])
    elif transform.name == "pitch_shift":
        transformed = apply_pitch_shift(transformed, sample_rate=sample_rate, n_steps=sampled["n_steps"])
    elif transform.name == "lowpass":
        transformed = apply_lowpass_filter(transformed, sample_rate=sample_rate, cutoff_hz=sampled["cutoff_hz"])
    elif transform.name == "highpass":
        transformed = apply_highpass_filter(transformed, sample_rate=sample_rate, cutoff_hz=sampled["cutoff_hz"])
    else:
        raise ValueError(f"Unsupported transform name: {transform.name}")
    transformed = trim_or_pad_waveform(transformed, target_length)
    return transformed.astype(np.float32), {"transform_name": transform.name, "parameters": sampled}


def select_policy_transforms(
    policy: AugmentationPolicySpec,
    registry: dict[str, AugmentationPolicySpec],
    rng: np.random.Generator,
    current_epoch: int | None,
) -> tuple[AugmentationTransformSpec, ...]:
    """Resolve which transforms to apply for one policy invocation."""
    ordered_transforms = tuple(sorted(policy.transforms, key=lambda item: item.order))
    if policy.selection_strategy == "notebook2_exact":
        return ordered_transforms
    if policy.selection_strategy == "curriculum":
        transition_epoch = int(policy.curriculum_transition_epoch or 0)
        if current_epoch is None or current_epoch < transition_epoch:
            return tuple(sorted(registry["baseline"].transforms, key=lambda item: item.order))
        late_candidates = list(tuple(sorted(registry["baseline"].transforms, key=lambda item: item.order)) + ordered_transforms)
        selected = [candidate for candidate in late_candidates if rng.random() <= float(candidate.probability)]
        if not selected:
            selected = [late_candidates[int(rng.integers(0, len(late_candidates)))]]
        return tuple(selected[: policy.max_transforms_per_sample])
    if policy.selection_strategy == "exactly_one":
        weights = np.asarray([max(1e-6, float(item.probability)) for item in ordered_transforms], dtype=np.float64)
        weights = weights / weights.sum()
        selected_index = int(rng.choice(np.arange(len(ordered_transforms)), p=weights))
        return (ordered_transforms[selected_index],)
    selected = [item for item in ordered_transforms if rng.random() <= float(item.probability)]
    if not selected:
        weights = np.asarray([max(1e-6, float(item.probability)) for item in ordered_transforms], dtype=np.float64)
        weights = weights / weights.sum()
        selected = [ordered_transforms[int(rng.choice(np.arange(len(ordered_transforms)), p=weights))]]
    if len(selected) > policy.max_transforms_per_sample:
        selected_indices = rng.choice(np.arange(len(selected)), size=policy.max_transforms_per_sample, replace=False)
        selected = [selected[int(index)] for index in np.sort(selected_indices)]
    return tuple(sorted(selected, key=lambda item: item.order))


def apply_augmented_policy(
    waveform: np.ndarray,
    sample_rate: int,
    target_length: int,
    profile: str,
    rng: np.random.Generator,
    current_epoch: int | None = None,
) -> tuple[np.ndarray, list[dict[str, object]]]:
    """Apply one named augmentation policy and return waveform plus metadata."""
    if profile in {"baseline", "extended"}:
        legacy_waveform = apply_augmentation_profile(
            waveform=waveform,
            sample_rate=sample_rate,
            target_length=target_length,
            profile="baseline" if profile == "baseline" else "extended",
            rng=rng,
        )
        return legacy_waveform.astype(np.float32), [{"transform_name": profile, "parameters": {"legacy_path": True}}]
    policy = AUGMENTATION_POLICY_REGISTRY[profile]
    segment = trim_or_pad_waveform(normalize_waveform(waveform), target_length)
    metadata_rows: list[dict[str, object]] = []
    for transform in select_policy_transforms(policy, AUGMENTATION_POLICY_REGISTRY, rng, current_epoch):
        segment, metadata = apply_transform_with_metadata(segment, sample_rate, target_length, transform, rng)
        metadata_rows.append(metadata)
    segment = normalize_waveform(segment)
    if not np.isfinite(segment).all():
        raise FloatingPointError(f"Policy {profile} produced non-finite waveform values.")
    return segment.astype(np.float32), metadata_rows


### Query-regime registry, windowing registry, manifest builders, and dataset classes

In [ ]:
class ContrastivePairDataset(Dataset[dict[str, object]]):
    """Yield anchor and positive pairs for contrastive training."""

    def __init__(
        self,
        audio_dir: Path,
        track_ids: list[int],
        segment_length: int,
        sample_rate: int,
        input_mode: InputMode,
        config: ExperimentConfig,
        augmentation_profile: str,
    ) -> None:
        self.audio_dir = audio_dir
        self.track_ids = track_ids
        self.segment_length = segment_length
        self.sample_rate = sample_rate
        self.input_mode = input_mode
        self.config = config
        self.augmentation_profile = augmentation_profile
        self.current_epoch = 0

    def __len__(self) -> int:
        return len(self.track_ids)

    def __getitem__(self, index: int) -> dict[str, object]:
        track_id = int(self.track_ids[index])
        seed = int(torch.randint(low=0, high=2**31 - 1, size=(1,)).item())
        rng = np.random.default_rng(seed)
        waveform, _ = load_audio_file(
            get_audio_path(self.audio_dir, track_id),
            target_sr=self.sample_rate,
            mono=True,
            cache_dir=self.config.paths.waveform_cache_dir,
        )
        segment, start_index = extract_random_segment(waveform, self.segment_length, rng=rng)
        segment = normalize_waveform(trim_or_pad_waveform(segment, self.segment_length))
        positive_waveform, _ = apply_augmented_policy(segment.copy(), self.sample_rate, self.segment_length, self.augmentation_profile, rng, self.current_epoch)
        if self.input_mode == "spectrogram":
            anchor_value: torch.Tensor = spectrogram_tensor_from_waveform(segment, self.config)
            positive_value: torch.Tensor = spectrogram_tensor_from_waveform(positive_waveform, self.config)
        else:
            anchor_value = torch.from_numpy(segment).float()
            positive_value = torch.from_numpy(positive_waveform).float()
        return {"anchor": anchor_value, "positive": positive_value, "track_id": track_id, "anchor_start_time": float(start_index / self.sample_rate)}

def apply_query_condition_v2(
    waveform: np.ndarray,
    sample_rate: int,
    target_length: int,
    condition: RetrievalConditionName,
    rng: np.random.Generator,
    apply_fade: bool = False,
) -> tuple[np.ndarray, list[dict[str, object]]]:
    """Apply one retrieval degradation condition and return transformed audio plus metadata."""
    segment = trim_or_pad_waveform(normalize_waveform(waveform), target_length)
    metadata: list[dict[str, object]] = []

    def record(name: str, parameters: dict[str, object]) -> None:
        metadata.append({"transform_name": name, "parameters": serialize_jsonable(parameters)})

    if condition == "clean":
        output = segment
    elif condition in {"noise", "pitch", "time", "lowpass", "highpass"}:
        output = apply_retrieval_degradation(segment, sample_rate, target_length, condition if condition != "time" else "time", rng)
        record(condition, {"legacy_path": True})
    elif condition == "combined_moderate":
        output = trim_or_pad_waveform(apply_random_gain(segment, rng=rng, min_db=-3.0, max_db=3.0), target_length)
        record("gain", {"policy": "moderate"})
        snr_db = float(rng.uniform(12.0, 18.0))
        output = trim_or_pad_waveform(add_gaussian_noise(output, rng=rng, snr_db=snr_db), target_length)
        record("gaussian_noise", {"snr_db": snr_db})
        secondary_conditions = ["pitch", "time", "lowpass", "highpass"]
        rng.shuffle(secondary_conditions)
        for secondary_condition in secondary_conditions[:2]:
            if secondary_condition == "pitch":
                n_steps = float(rng.uniform(-1.0, 1.0))
                output = trim_or_pad_waveform(apply_pitch_shift(output, sample_rate=sample_rate, n_steps=n_steps), target_length)
                record("pitch_shift", {"n_steps": n_steps})
            elif secondary_condition == "time":
                rate = float(rng.uniform(0.97, 1.03))
                output = trim_or_pad_waveform(apply_time_stretch(output, rate=rate), target_length)
                record("time_stretch", {"rate": rate})
            elif secondary_condition == "lowpass":
                cutoff_hz = float(rng.uniform(3200.0, 4500.0))
                output = trim_or_pad_waveform(apply_lowpass_filter(output, sample_rate=sample_rate, cutoff_hz=cutoff_hz), target_length)
                record("lowpass", {"cutoff_hz": cutoff_hz})
            elif secondary_condition == "highpass":
                cutoff_hz = float(rng.uniform(180.0, 480.0))
                output = trim_or_pad_waveform(apply_highpass_filter(output, sample_rate=sample_rate, cutoff_hz=cutoff_hz), target_length)
                record("highpass", {"cutoff_hz": cutoff_hz})
    elif condition == "realistic_hard":
        output = trim_or_pad_waveform(apply_random_gain(segment, rng=rng, min_db=-4.0, max_db=4.0), target_length)
        record("gain", {"policy": "realistic_hard"})
        snr_db = float(rng.uniform(6.0, 12.0))
        output = trim_or_pad_waveform(add_gaussian_noise(output, rng=rng, snr_db=snr_db), target_length)
        record("gaussian_noise", {"snr_db": snr_db})
        if rng.random() < 0.5:
            cutoff_hz = float(rng.uniform(2600.0, 4200.0))
            output = trim_or_pad_waveform(apply_lowpass_filter(output, sample_rate=sample_rate, cutoff_hz=cutoff_hz), target_length)
            record("lowpass", {"cutoff_hz": cutoff_hz})
        else:
            cutoff_hz = float(rng.uniform(220.0, 620.0))
            output = trim_or_pad_waveform(apply_highpass_filter(output, sample_rate=sample_rate, cutoff_hz=cutoff_hz), target_length)
            record("highpass", {"cutoff_hz": cutoff_hz})
        rate = float(rng.uniform(0.93, 1.07))
        output = trim_or_pad_waveform(apply_time_stretch(output, rate=rate), target_length)
        record("time_stretch", {"rate": rate})
    else:
        raise ValueError(f"Unsupported retrieval condition: {condition}")

    if apply_fade:
        fade_seconds = float(rng.uniform(0.04, 0.12))
        output = apply_fade_envelope(output, sample_rate=sample_rate, fade_seconds=fade_seconds)
        record("fade_envelope", {"fade_seconds": fade_seconds})

    output = normalize_waveform(trim_or_pad_waveform(output, target_length))
    if not np.isfinite(output).all():
        raise FloatingPointError(f"Query condition {condition} produced non-finite waveform values.")
    return output.astype(np.float32), metadata


def choose_random_offcenter_start(total_samples: int, segment_length: int, rng: np.random.Generator) -> int:
    """Sample an off-center segment start while avoiding the exact centered crop when possible."""
    if total_samples <= segment_length:
        return 0
    max_start = total_samples - segment_length
    center_start = max_start // 2
    exclusion_radius = max(1, int(0.1 * max_start))
    for _ in range(32):
        candidate = int(rng.integers(0, max_start + 1))
        if abs(candidate - center_start) >= exclusion_radius:
            return candidate
    return 0 if center_start > max_start / 2 else max_start


def build_even_window_starts(total_samples: int, segment_length: int, window_count: int) -> list[int]:
    """Create evenly spaced segment starts for multi-window indexing."""
    if window_count <= 0:
        raise ValueError(f"window_count must be positive, received {window_count}.")
    if total_samples <= segment_length:
        return [0]
    max_start = total_samples - segment_length
    starts = np.linspace(0, max_start, num=window_count, dtype=np.int64)
    return sorted({int(start_value) for start_value in starts})


def build_query_regime_registry(config: ExperimentConfig) -> dict[QueryRegimeName, QueryRegimeSpec]:
    """Construct the retrieval-query regime registry."""
    return {
        "clean_current": QueryRegimeSpec("clean_current", "Notebook 2 continuity baseline.", (config.segment_seconds,), True, False, ("clean", "noise", "pitch", "time", "lowpass", "highpass"), 1),
        "short_centered": QueryRegimeSpec("short_centered", "Short centered queries.", config.short_query_lengths_seconds, True, False, ("clean", "noise", "pitch", "time", "lowpass", "highpass"), 1),
        "short_offcenter": QueryRegimeSpec("short_offcenter", "Short off-center queries.", config.short_query_lengths_seconds, False, True, ("clean", "noise", "pitch", "time", "lowpass", "highpass"), 1),
        "combined_moderate": QueryRegimeSpec("combined_moderate", "Off-center queries with moderate combined degradations.", config.short_query_lengths_seconds, False, True, ("combined_moderate",), 1),
        "multi_segment_same_track": QueryRegimeSpec("multi_segment_same_track", "Aggregate several short segments from one track.", (1.0, 2.0), False, True, ("clean", "combined_moderate"), config.multi_segment_group_size),
        "realistic_hard": QueryRegimeSpec("realistic_hard", "Harder off-center queries with fade envelopes.", config.short_query_lengths_seconds, False, True, ("realistic_hard",), 1, False, True),
    }


def build_windowing_registry(config: ExperimentConfig) -> dict[WindowingSpecName, WindowingSpec]:
    """Construct the reference-window indexing registry."""
    return {
        "single_center": WindowingSpec("single_center", "Notebook 2 single centered reference segment.", 1, config.segment_seconds, True),
        "multi3_even": WindowingSpec("multi3_even", "Three evenly spaced reference windows.", 3, config.segment_seconds, False),
        "multi5_even": WindowingSpec("multi5_even", "Five evenly spaced reference windows.", 5, config.segment_seconds, False, False),
    }


class ReferenceManifestDataset(Dataset[dict[str, object]]):
    """Yield deterministic reference windows from a precomputed manifest."""

    def __init__(self, audio_dir: Path, manifest_df: pd.DataFrame, sample_rate: int, input_mode: InputMode, config: ExperimentConfig) -> None:
        self.audio_dir = audio_dir
        self.manifest_df = manifest_df.reset_index(drop=True)
        self.sample_rate = sample_rate
        self.input_mode = input_mode
        self.config = config

    def __len__(self) -> int:
        return int(len(self.manifest_df))

    def __getitem__(self, index: int) -> dict[str, object]:
        row = self.manifest_df.iloc[int(index)]
        waveform, _ = load_audio_file(
            get_audio_path(self.audio_dir, int(row["track_id"])),
            target_sr=self.sample_rate,
            mono=True,
            cache_dir=self.config.paths.waveform_cache_dir,
        )
        segment_length = int(float(row["segment_length_seconds"]) * self.sample_rate)
        segment = extract_segment_by_start(waveform, int(row["start_sample"]), segment_length)
        if self.input_mode == "spectrogram":
            model_input: torch.Tensor = spectrogram_tensor_from_waveform(segment, self.config)
        else:
            model_input = torch.from_numpy(segment).float()
        return {"inputs": model_input, "track_id": int(row["track_id"]), "metadata_row": row.to_dict()}


class QueryManifestDataset(Dataset[dict[str, object]]):
    """Yield deterministic retrieval queries from a precomputed manifest."""

    def __init__(self, audio_dir: Path, manifest_df: pd.DataFrame, sample_rate: int, input_mode: InputMode, config: ExperimentConfig) -> None:
        self.audio_dir = audio_dir
        self.manifest_df = manifest_df.reset_index(drop=True)
        self.sample_rate = sample_rate
        self.input_mode = input_mode
        self.config = config

    def __len__(self) -> int:
        return int(len(self.manifest_df))

    def __getitem__(self, index: int) -> dict[str, object]:
        row = self.manifest_df.iloc[int(index)]
        waveform, _ = load_audio_file(
            get_audio_path(self.audio_dir, int(row["track_id"])),
            target_sr=self.sample_rate,
            mono=True,
            cache_dir=self.config.paths.waveform_cache_dir,
        )
        segment_length = int(float(row["query_length_seconds"]) * self.sample_rate)
        segment = extract_segment_by_start(waveform, int(row["start_sample"]), segment_length)
        rng_seed = int(self.config.seed + int(row["track_id"]) * 17 + int(index) * 13)
        rng = np.random.default_rng(rng_seed)
        degraded_segment, degradation_metadata = apply_query_condition_v2(segment, self.sample_rate, segment_length, str(row["condition_name"]), rng, bool(row["apply_fade_envelope"]))
        if self.input_mode == "spectrogram":
            model_input: torch.Tensor = spectrogram_tensor_from_waveform(degraded_segment, self.config)
        else:
            model_input = torch.from_numpy(degraded_segment).float()
        metadata_row = row.to_dict()
        metadata_row["degradation_metadata"] = degradation_metadata
        return {"inputs": model_input, "track_id": int(row["track_id"]), "metadata_row": metadata_row}


def collate_retrieval_manifest_batch(batch: list[dict[str, object]]) -> RetrievalBatch:
    """Stack retrieval samples while preserving per-row metadata."""
    inputs = torch.stack([sample["inputs"] for sample in batch])
    track_ids = torch.tensor([int(sample["track_id"]) for sample in batch], dtype=torch.long)
    metadata_rows = [dict(sample["metadata_row"]) for sample in batch]
    return {"inputs": inputs, "track_id": track_ids, "metadata_rows": metadata_rows}


AUGMENTATION_POLICY_REGISTRY = build_augmentation_policy_registry()
QUERY_REGIME_REGISTRY = build_query_regime_registry(CONFIG)
WINDOWING_REGISTRY = build_windowing_registry(CONFIG)

AUGMENTATION_POLICY_SUMMARY = pd.DataFrame(
    [
        {
            "policy_name": policy.name,
            "selection_strategy": policy.selection_strategy,
            "max_transforms_per_sample": policy.max_transforms_per_sample,
            "enabled": policy.enabled,
            "transforms": ", ".join(transform.name for transform in policy.transforms),
        }
        for policy in AUGMENTATION_POLICY_REGISTRY.values()
    ]
)
QUERY_REGIME_SUMMARY = pd.DataFrame([serialize_jsonable(regime) for regime in QUERY_REGIME_REGISTRY.values()])
WINDOWING_SUMMARY = pd.DataFrame([serialize_jsonable(spec) for spec in WINDOWING_REGISTRY.values()])
display(AUGMENTATION_POLICY_SUMMARY)
display(QUERY_REGIME_SUMMARY)
display(WINDOWING_SUMMARY)


## Experiment Orchestration

This section resolves the active training runs, query regimes, windowing specs, and FAISS indexes, then drives:
- smoke tests
- new augmentation-policy training runs
- full retrieval evaluation
- final exports


### Loaders, manifests, FAISS sweep, training orchestration, and final synthesis helpers

In [ ]:
@dataclass(frozen=True)
class ArtifactDescriptor:
    """A resolved model artifact that can be loaded and evaluated."""

    run_id: str
    source: str
    model_name: ModelName
    augmentation_profile: str
    embed_dim: int
    checkpoint_path: Path
    freeze_backbone: bool
    notes: str = ""


def build_index_spec_registry(config: ExperimentConfig) -> dict[str, IndexSpec]:
    """Construct the bounded FAISS index sweep."""
    registry: dict[str, IndexSpec] = {"exact_ip": IndexSpec("exact_ip", "exact_ip", "Exact inner-product FAISS baseline.")}
    for nprobe_value in config.faiss_nprobe_values:
        registry[f"ivfflat_nprobe{nprobe_value}"] = IndexSpec(f"ivfflat_nprobe{nprobe_value}", "ivfflat", f"IVFFlat with nprobe={nprobe_value}.", int(nprobe_value))
        registry[f"ivfpq_nprobe{nprobe_value}"] = IndexSpec(f"ivfpq_nprobe{nprobe_value}", "ivfpq", f"IVFPQ with nprobe={nprobe_value}.", int(nprobe_value))
    return registry


def build_active_control_artifacts(artifact_df: pd.DataFrame, config: ExperimentConfig) -> list[ArtifactDescriptor]:
    """Convert discovered Notebook 2 checkpoints into active artifact descriptors."""
    available_rows = artifact_df[artifact_df["checkpoint_exists"]].copy()
    descriptors = [
        ArtifactDescriptor(
            run_id=str(row.run_id),
            source="notebook2",
            model_name=str(row.model_name),
            augmentation_profile=str(row.augmentation_profile),
            embed_dim=int(row.embed_dim),
            checkpoint_path=Path(str(row.checkpoint_path)),
            freeze_backbone=bool(row.freeze_backbone),
            notes="Notebook 2 control artifact.",
        )
        for row in available_rows.itertuples(index=False)
        if str(row.run_id) in config.active_control_run_ids
    ]
    return sorted(descriptors, key=lambda descriptor: descriptor.run_id)


def build_reference_manifest(track_ids: list[int], windowing_spec: WindowingSpec, config: ExperimentConfig, force_rebuild: bool = False) -> pd.DataFrame:
    """Build or load the cached reference manifest for one windowing strategy."""
    manifest_dir = safe_mkdir(config.paths.cache_dir / "reference_manifests")
    cache_key = hash_experiment_config({"windowing_spec": windowing_spec, "track_ids": track_ids, "sample_rate": config.sample_rate})
    manifest_path = manifest_dir / f"{windowing_spec.name}_{cache_key}.csv"
    if manifest_path.exists() and not force_rebuild:
        return pd.read_csv(manifest_path)

    rows: list[dict[str, object]] = []
    target_length = int(float(windowing_spec.segment_seconds) * config.sample_rate)
    for track_id in tqdm(track_ids, desc=f"reference-manifest:{windowing_spec.name}", leave=False):
        waveform, _ = load_audio_file(
            get_audio_path(config.paths.audio_dir, int(track_id)),
            target_sr=config.sample_rate,
            mono=True,
            cache_dir=config.paths.waveform_cache_dir,
        )
        total_samples = int(waveform.shape[0])
        if windowing_spec.centered_only:
            _, center_start = extract_center_segment(waveform, target_length)
            starts = [center_start]
        else:
            starts = build_even_window_starts(total_samples, target_length, int(windowing_spec.window_count))
        for window_index, start_sample in enumerate(starts):
            rows.append(
                {
                    "record_id": f"{windowing_spec.name}:{track_id}:{window_index}",
                    "track_id": int(track_id),
                    "windowing_name": windowing_spec.name,
                    "window_index": int(window_index),
                    "segment_length_seconds": float(windowing_spec.segment_seconds),
                    "start_sample": int(start_sample),
                    "end_sample": int(start_sample + target_length),
                    "start_seconds": float(start_sample / config.sample_rate),
                    "end_seconds": float((start_sample + target_length) / config.sample_rate),
                    "cache_key": cache_key,
                }
            )
    manifest_df = pd.DataFrame(rows)
    save_dataframe(manifest_path, manifest_df)
    return manifest_df


def build_query_manifest(track_ids: list[int], regime_spec: QueryRegimeSpec, config: ExperimentConfig, force_rebuild: bool = False) -> pd.DataFrame:
    """Build or load the cached query manifest for one evaluation regime."""
    manifest_dir = safe_mkdir(config.paths.cache_dir / "query_manifests")
    cache_key = hash_experiment_config({"regime_spec": regime_spec, "track_ids": track_ids, "sample_rate": config.sample_rate})
    manifest_path = manifest_dir / f"{regime_spec.name}_{cache_key}.csv"
    if manifest_path.exists() and not force_rebuild:
        return pd.read_csv(manifest_path)

    rows: list[dict[str, object]] = []
    for track_id in tqdm(track_ids, desc=f"query-manifest:{regime_spec.name}", leave=False):
        waveform, _ = load_audio_file(
            get_audio_path(config.paths.audio_dir, int(track_id)),
            target_sr=config.sample_rate,
            mono=True,
            cache_dir=config.paths.waveform_cache_dir,
        )
        total_samples = int(waveform.shape[0])
        for query_length_seconds in regime_spec.lengths_seconds:
            target_length = int(float(query_length_seconds) * config.sample_rate)
            for condition_name in regime_spec.conditions:
                if regime_spec.group_size == 1:
                    rng = np.random.default_rng(int(config.seed + int(track_id) * 31 + int(float(query_length_seconds) * 1000)))
                    if regime_spec.centered:
                        _, start_sample = extract_center_segment(waveform, target_length)
                    elif regime_spec.random_offset:
                        start_sample = choose_random_offcenter_start(total_samples, target_length, rng)
                    else:
                        start_sample = 0
                    query_group_id = f"{regime_spec.name}:{track_id}:{condition_name}:{query_length_seconds:.2f}"
                    rows.append(
                        {
                            "record_id": f"{query_group_id}:0",
                            "query_group_id": query_group_id,
                            "track_id": int(track_id),
                            "regime_name": regime_spec.name,
                            "condition_name": condition_name,
                            "query_length_seconds": float(query_length_seconds),
                            "group_size": 1,
                            "segment_index": 0,
                            "start_sample": int(start_sample),
                            "end_sample": int(start_sample + target_length),
                            "start_seconds": float(start_sample / config.sample_rate),
                            "end_seconds": float((start_sample + target_length) / config.sample_rate),
                            "apply_fade_envelope": bool(regime_spec.apply_fade_envelope),
                            "cache_key": cache_key,
                        }
                    )
                else:
                    rng = np.random.default_rng(int(config.seed + int(track_id) * 101 + int(float(query_length_seconds) * 1000)))
                    base_starts = build_even_window_starts(total_samples, target_length, int(regime_spec.group_size))
                    jitter_limit = max(1, int(0.05 * config.sample_rate))
                    query_group_id = f"{regime_spec.name}:{track_id}:{condition_name}:{query_length_seconds:.2f}"
                    for segment_index, base_start in enumerate(base_starts):
                        jitter = int(rng.integers(-jitter_limit, jitter_limit + 1))
                        max_start = max(0, total_samples - target_length)
                        start_sample = min(max_start, max(0, base_start + jitter))
                        rows.append(
                            {
                                "record_id": f"{query_group_id}:{segment_index}",
                                "query_group_id": query_group_id,
                                "track_id": int(track_id),
                                "regime_name": regime_spec.name,
                                "condition_name": condition_name,
                                "query_length_seconds": float(query_length_seconds),
                                "group_size": int(regime_spec.group_size),
                                "segment_index": int(segment_index),
                                "start_sample": int(start_sample),
                                "end_sample": int(start_sample + target_length),
                                "start_seconds": float(start_sample / config.sample_rate),
                                "end_seconds": float((start_sample + target_length) / config.sample_rate),
                                "apply_fade_envelope": bool(regime_spec.apply_fade_envelope),
                                "cache_key": cache_key,
                            }
                        )
    manifest_df = pd.DataFrame(rows)
    save_dataframe(manifest_path, manifest_df)
    return manifest_df


### Embedding caches, FAISS search, training execution, evaluation, and final synthesis


In [ ]:
def make_pair_loader_v2(
    dataset: Dataset[dict[str, object]],
    batch_size: int,
    shuffle: bool,
    collate_fn: Callable[[list[dict[str, object]]], dict[str, object]],
    seed: int,
    num_workers: int,
) -> DataLoader[dict[str, object]]:
    """Build a deterministic DataLoader for contrastive pair datasets."""
    loader_kwargs: dict[str, object] = {
        "dataset": dataset,
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": num_workers,
        "pin_memory": torch.cuda.is_available(),
        "worker_init_fn": seed_worker,
        "generator": build_torch_generator(seed),
        "collate_fn": collate_fn,
    }

    if num_workers > 0:
        loader_kwargs["persistent_workers"] = CONFIG.dataloader_persistent_workers
        loader_kwargs["prefetch_factor"] = CONFIG.dataloader_prefetch_factor

    return DataLoader(**loader_kwargs)

def make_retrieval_loader_v2(
    dataset: Dataset[dict[str, object]],
    batch_size: int,
    collate_fn: Callable[[list[dict[str, object]]], dict[str, object]],
    num_workers: int,
) -> DataLoader[dict[str, object]]:
    """Build a deterministic retrieval loader for manifest datasets."""
    loader_kwargs: dict[str, object] = {
        "dataset": dataset,
        "batch_size": batch_size,
        "shuffle": False,
        "num_workers": num_workers,
        "pin_memory": torch.cuda.is_available(),
        "worker_init_fn": seed_worker,
        "generator": build_torch_generator(CONFIG.seed),
        "collate_fn": collate_fn,
    }

    if num_workers > 0:
        loader_kwargs["persistent_workers"] = CONFIG.dataloader_persistent_workers
        loader_kwargs["prefetch_factor"] = CONFIG.dataloader_prefetch_factor

    return DataLoader(**loader_kwargs)


def extract_embeddings_with_cache_v2(model: nn.Module, loader: DataLoader[dict[str, object]], input_mode: InputMode, manifest_df: pd.DataFrame, cache_namespace: str, model_cache_key: str) -> tuple[np.ndarray, pd.DataFrame, dict[str, object]]:
    """Encode a manifest into embeddings, using the on-disk cache when available."""
    manifest_hash = dataframe_hash(manifest_df)
    cache_key = hash_experiment_config({"namespace": cache_namespace, "manifest_hash": manifest_hash, "model_cache_key": model_cache_key})
    embedding_path = CONFIG.paths.embedding_cache_dir / f"{cache_key}.npy"
    metadata_path = CONFIG.paths.embedding_cache_dir / f"{cache_key}.csv"
    if embedding_path.exists() and metadata_path.exists():
        return np.load(embedding_path).astype(np.float32), pd.read_csv(metadata_path), {"cache_key": cache_key, "cache_hit": True, "embedding_path": embedding_path, "metadata_path": metadata_path}

    model.eval()
    embedding_chunks: list[np.ndarray] = []
    metadata_rows: list[dict[str, object]] = []
    for batch in tqdm(loader, desc=f"embed:{cache_namespace}", leave=False):
        encoded = encode_retrieval_batch(model, batch, input_mode, CONFIG.device, CONFIG.device_type, CONFIG.amp_enabled)
        embedding_chunks.append(encoded.detach().cpu().numpy().astype(np.float32))
        metadata_rows.extend([dict(metadata_row) for metadata_row in batch["metadata_rows"]])
    embeddings = np.concatenate(embedding_chunks, axis=0).astype(np.float32)
    metadata_df = pd.DataFrame(metadata_rows)
    np.save(embedding_path, embeddings)
    save_dataframe(metadata_path, metadata_df)
    return embeddings, metadata_df, {"cache_key": cache_key, "cache_hit": False, "embedding_path": embedding_path, "metadata_path": metadata_path}


def extract_query_embeddings_grouped_by_length_v2(
    model: nn.Module,
    input_mode: InputMode,
    query_manifest: pd.DataFrame,
    regime_name: str,
    model_cache_key: str,
    batch_size: int,
    collate_fn: Callable[[list[dict[str, object]]], RetrievalBatch],
    sample_rate: int,
) -> tuple[np.ndarray, pd.DataFrame, dict[str, object]]:
    """Encode query embeddings one query length at a time so batches remain shape-consistent."""
    if query_manifest.empty:
        raise ValueError(f"Query manifest for regime '{regime_name}' is empty.")

    embedding_chunks: list[np.ndarray] = []
    metadata_frames: list[pd.DataFrame] = []
    per_length_caches: list[dict[str, object]] = []

    unique_query_lengths = sorted(query_manifest["query_length_seconds"].astype(float).unique().tolist())

    for query_length_seconds in unique_query_lengths:
        query_manifest_subset = (
            query_manifest[
                np.isclose(
                    query_manifest["query_length_seconds"].astype(float).to_numpy(),
                    float(query_length_seconds),
                )
            ]
            .copy()
            .reset_index(drop=True)
        )

        if query_manifest_subset.empty:
            continue

        query_loader = make_retrieval_loader_v2(
            QueryManifestDataset(
                CONFIG.paths.audio_dir,
                query_manifest_subset,
                CONFIG.sample_rate if input_mode == "spectrogram" else sample_rate,
                input_mode,
                CONFIG,
            ),
            batch_size,
            collate_fn,
            CONFIG.num_workers,
        )

        query_embeddings_part, query_metadata_df_part, query_cache_part = extract_embeddings_with_cache_v2(
            model,
            query_loader,
            input_mode,
            query_manifest_subset,
            f"query_{regime_name}_{query_length_seconds:.2f}s",
            model_cache_key,
        )

        embedding_chunks.append(query_embeddings_part.astype(np.float32, copy=False))
        metadata_frames.append(query_metadata_df_part.reset_index(drop=True))
        per_length_caches.append(
            {
                "query_length_seconds": float(query_length_seconds),
                **query_cache_part,
            }
        )

    if not embedding_chunks:
        raise ValueError(f"No query embeddings were produced for regime '{regime_name}'.")

    query_embeddings = np.concatenate(embedding_chunks, axis=0).astype(np.float32, copy=False)
    query_metadata_df = pd.concat(metadata_frames, ignore_index=True)

    query_cache = {
        "cache_hit": bool(all(bool(cache_record.get("cache_hit", False)) for cache_record in per_length_caches)),
        "embedding_path": ";".join(str(cache_record["embedding_path"]) for cache_record in per_length_caches),
        "metadata_path": ";".join(str(cache_record["metadata_path"]) for cache_record in per_length_caches),
        "per_length_caches": per_length_caches,
    }

    return query_embeddings, query_metadata_df, query_cache


def resolve_index_runtime_parameters(index_spec: IndexSpec, reference_count: int, embed_dim: int) -> dict[str, int]:
    """Resolve data-dependent FAISS parameters for an index specification."""
    if index_spec.kind == "exact_ip":
        return {"nlist": 0, "nprobe": 0, "pq_m": 0}
    nlist = max(1, min(int(index_spec.max_nlist), max(1, reference_count // 32)))
    nprobe = min(int(index_spec.nprobe or 1), nlist)
    if index_spec.kind == "ivfpq":
        if embed_dim % 16 == 0:
            pq_m = 16
        elif embed_dim % 8 == 0:
            pq_m = 8
        else:
            raise ValueError(f"Embedding dimension {embed_dim} is incompatible with the planned PQ partitions.")
    else:
        pq_m = 0
    return {"nlist": int(nlist), "nprobe": int(nprobe), "pq_m": int(pq_m)}


def build_or_load_faiss_index_v2(reference_embeddings: np.ndarray, index_spec: IndexSpec, reference_cache_key: str) -> tuple[faiss.Index, dict[str, object]]:
    """Build or load a cached FAISS index for one reference embedding set."""
    index_cache_key = hash_experiment_config({"reference_cache_key": reference_cache_key, "index_name": index_spec.name, "index_kind": index_spec.kind})
    index_path = CONFIG.paths.index_dir / f"{index_cache_key}.faiss"
    metadata_path = CONFIG.paths.index_dir / f"{index_cache_key}.json"
    if index_path.exists() and metadata_path.exists():
        return faiss.read_index(str(index_path)), dict(load_json(metadata_path))

    normalized_embeddings = normalize_embeddings(reference_embeddings)
    reference_count, embed_dim = normalized_embeddings.shape
    runtime_parameters = resolve_index_runtime_parameters(index_spec, reference_count, embed_dim)
    build_started_at = time.perf_counter()
    if index_spec.kind == "exact_ip":
        index = faiss.IndexFlatIP(embed_dim)
        index.add(normalized_embeddings)
    elif index_spec.kind == "ivfflat":
        if reference_count < max(runtime_parameters["nlist"] * 8, 256):
            raise ValueError("Not enough reference windows to train IVFFlat stably.")
        quantizer = faiss.IndexFlatIP(embed_dim)
        index = faiss.IndexIVFFlat(quantizer, embed_dim, runtime_parameters["nlist"], faiss.METRIC_INNER_PRODUCT)
        index.train(normalized_embeddings)
        index.add(normalized_embeddings)
        index.nprobe = runtime_parameters["nprobe"]
    elif index_spec.kind == "ivfpq":
        if reference_count < max(runtime_parameters["nlist"] * 8, 256):
            raise ValueError("Not enough reference windows to train IVFPQ stably.")
        quantizer = faiss.IndexFlatIP(embed_dim)
        index = faiss.IndexIVFPQ(quantizer, embed_dim, runtime_parameters["nlist"], runtime_parameters["pq_m"], 8, faiss.METRIC_INNER_PRODUCT)
        index.train(normalized_embeddings)
        index.add(normalized_embeddings)
        index.nprobe = runtime_parameters["nprobe"]
    else:
        raise ValueError(f"Unsupported index kind: {index_spec.kind}")

    build_seconds = float(time.perf_counter() - build_started_at)
    with tempfile.TemporaryDirectory() as temporary_dir:
        temp_index_path = Path(temporary_dir) / "index.faiss"
        faiss.write_index(index, str(temp_index_path))
        index_size_mb = float(temp_index_path.stat().st_size / (1024 ** 2))
    metadata = {"index_name": index_spec.name, "index_kind": index_spec.kind, "build_seconds": build_seconds, "index_size_mb": index_size_mb, "reference_count": int(reference_count), "embed_dim": int(embed_dim), **runtime_parameters}
    faiss.write_index(index, str(index_path))
    save_json(metadata_path, metadata)
    return index, metadata


def search_unique_track_results(index: faiss.Index, query_embeddings: np.ndarray, reference_track_ids: np.ndarray, top_k: int, oversample_factor: int) -> tuple[np.ndarray, np.ndarray]:
    """Search an index and deduplicate results to unique track identifiers."""
    normalized_queries = normalize_embeddings(query_embeddings)
    total_reference = int(reference_track_ids.shape[0])
    search_k = min(total_reference, max(top_k, top_k * oversample_factor))
    while True:
        scores, neighbor_indices = index.search(normalized_queries.astype(np.float32), search_k)
        unique_track_ids = np.full((normalized_queries.shape[0], top_k), -1, dtype=np.int64)
        unique_scores = np.full((normalized_queries.shape[0], top_k), -np.inf, dtype=np.float32)
        needs_more_neighbors = False
        for query_index in range(normalized_queries.shape[0]):
            seen_track_ids: set[int] = set()
            write_index = 0
            for score_value, reference_row_index in zip(scores[query_index], neighbor_indices[query_index]):
                if reference_row_index < 0:
                    continue
                track_id = int(reference_track_ids[reference_row_index])
                if track_id in seen_track_ids:
                    continue
                seen_track_ids.add(track_id)
                unique_track_ids[query_index, write_index] = track_id
                unique_scores[query_index, write_index] = float(score_value)
                write_index += 1
                if write_index >= top_k:
                    break
            if write_index < top_k and search_k < total_reference:
                needs_more_neighbors = True
        if not needs_more_neighbors:
            return unique_track_ids, unique_scores
        search_k = min(total_reference, search_k * 2)


def aggregate_group_rankings(query_metadata_df: pd.DataFrame, ranked_track_ids: np.ndarray, ranked_scores: np.ndarray) -> pd.DataFrame:
    """Aggregate segment-level rankings into one row per logical query group."""
    grouped_rows: list[dict[str, object]] = []
    grouped = query_metadata_df.reset_index(drop=True).groupby("query_group_id", sort=False)
    for query_group_id, group_df in grouped:
        score_buckets: dict[int, list[float]] = defaultdict(list)
        for query_row_index in group_df.index.tolist():
            for track_id, score_value in zip(ranked_track_ids[query_row_index], ranked_scores[query_row_index]):
                if track_id < 0:
                    continue
                score_buckets[int(track_id)].append(float(score_value))
        aggregated_candidates = sorted(
            [(track_id, float(np.mean(scores))) for track_id, scores in score_buckets.items()],
            key=lambda item: item[1],
            reverse=True,
        )
        true_track_id = int(group_df["track_id"].iloc[0])
        rank = next((candidate_index + 1 for candidate_index, candidate in enumerate(aggregated_candidates) if candidate[0] == true_track_id), len(aggregated_candidates) + 1)
        top_candidates = aggregated_candidates[: max(CONFIG.top_k)]
        grouped_rows.append(
            {
                "query_group_id": query_group_id,
                "true_track_id": true_track_id,
                "regime_name": str(group_df["regime_name"].iloc[0]),
                "condition_name": str(group_df["condition_name"].iloc[0]),
                "query_length_seconds": float(group_df["query_length_seconds"].iloc[0]),
                "group_size": int(group_df["group_size"].iloc[0]),
                "rank": int(rank),
                "correct_top1": bool(rank == 1),
                "top1_track_id": int(top_candidates[0][0]) if top_candidates else -1,
                "top1_score": float(top_candidates[0][1]) if top_candidates else float("nan"),
                "top_candidate_track_ids_json": json.dumps([int(candidate[0]) for candidate in top_candidates]),
                "top_candidate_scores_json": json.dumps([float(candidate[1]) for candidate in top_candidates]),
            }
        )
    return pd.DataFrame(grouped_rows)


### Notebook 4 Query Regimes, Aggregation, Hard Negatives, And Export Helpers


In [ ]:
# Notebook 4 query-regime, windowing, and manifest overrides.
def build_query_regime_registry(config: ExperimentConfig) -> dict[QueryRegimeName, QueryRegimeSpec]:
    """Construct the Notebook 4 retrieval-query regime registry."""

    return {
        "clean_current": QueryRegimeSpec("clean_current", "Continuity baseline with clean and single degradations.", (config.segment_seconds,), True, False, ("clean", "noise", "pitch", "time", "lowpass", "highpass"), 1),
        "short_centered": QueryRegimeSpec("short_centered", "Short centered queries.", config.short_query_lengths_seconds, True, False, ("clean", "noise", "pitch", "time", "lowpass", "highpass"), 1),
        "short_offcenter": QueryRegimeSpec("short_offcenter", "Short off-center queries.", config.short_query_lengths_seconds, False, True, ("clean", "noise", "pitch", "time", "lowpass", "highpass"), 1),
        "combined_moderate": QueryRegimeSpec("combined_moderate", "Off-center queries with combined degradations.", config.short_query_lengths_seconds, False, True, ("combined_moderate",), 1),
        "multi_segment_same_track": QueryRegimeSpec("multi_segment_same_track", "Grouped same-track fragments.", (1.0, 2.0), False, True, ("clean", "combined_moderate"), config.multi_segment_group_size),
        "realistic_hard": QueryRegimeSpec("realistic_hard", "Harder off-center queries with fade envelopes.", config.short_query_lengths_seconds, False, True, ("realistic_hard",), 1, True, True),
        "multi_segment_fragmented_weighted": QueryRegimeSpec("multi_segment_fragmented_weighted", "Grouped fragments with uneven local quality.", (1.0, 2.0), False, True, ("clean", "combined_moderate", "realistic_hard"), config.multi_segment_group_size, True, True),
        "multi_segment_temporally_jittered": QueryRegimeSpec("multi_segment_temporally_jittered", "Grouped fragments with mild temporal jitter.", (1.0, 2.0), False, True, ("clean", "combined_moderate"), config.multi_segment_group_size, True, False),
        "hard_negative_short_queries": QueryRegimeSpec("hard_negative_short_queries", "Short queries designed to expose acoustically similar false matches.", (1.0, 2.0), False, True, ("combined_moderate", "realistic_hard"), 1, True, True),
    }


def build_windowing_registry(config: ExperimentConfig) -> dict[WindowingSpecName, WindowingSpec]:
    """Construct the Notebook 4 reference-window indexing registry."""

    return {
        "single_center": WindowingSpec("single_center", "Notebook 2 single centered reference segment.", 1, config.segment_seconds, True, True),
        "multi3_even": WindowingSpec("multi3_even", "Three evenly spaced reference windows.", 3, config.segment_seconds, False, True),
        "multi5_even": WindowingSpec("multi5_even", "Five evenly spaced reference windows.", 5, config.segment_seconds, False, True),
    }


def _sample_single_query_start(total_samples: int, target_length: int, centered: bool, random_offset: bool, rng: np.random.Generator) -> int:
    """Sample one query start position for a single-segment regime."""

    if total_samples <= target_length:
        return 0
    if centered:
        _, center_start = extract_center_segment(np.zeros(total_samples, dtype=np.float32), target_length)
        return int(center_start)
    if random_offset:
        return int(rng.integers(0, max(1, total_samples - target_length)))
    return 0


def _build_group_starts(total_samples: int, target_length: int, group_size: int, rng: np.random.Generator, jitter_seconds: float = 0.0) -> list[int]:
    """Build grouped query start positions with optional temporal jitter."""

    if group_size <= 1:
        return [_sample_single_query_start(total_samples, target_length, False, True, rng)]
    even_starts = build_even_window_starts(total_samples, target_length, group_size)
    if jitter_seconds <= 0.0:
        return even_starts
    jitter_samples = int(jitter_seconds * CONFIG.sample_rate)
    jittered_starts: list[int] = []
    max_start = max(0, total_samples - target_length)
    for start_sample in even_starts:
        offset = int(rng.integers(-jitter_samples, jitter_samples + 1))
        jittered_starts.append(int(min(max(0, start_sample + offset), max_start)))
    return jittered_starts


def build_query_manifest(track_ids: list[int], regime_spec: QueryRegimeSpec, config: ExperimentConfig, force_rebuild: bool = False) -> pd.DataFrame:
    """Build or load the cached query manifest for one evaluation regime."""

    cache_key = hash_experiment_config({"regime_spec": regime_spec, "track_ids": track_ids, "sample_rate": config.sample_rate})
    manifest_path = config.paths.query_manifest_dir / f"{regime_spec.name}_{cache_key}.csv"
    if manifest_path.exists() and not force_rebuild and not config.force_recompute_embeddings:
        return pd.read_csv(manifest_path)

    rows: list[dict[str, object]] = []
    for track_id in tqdm(track_ids, desc=f"query-manifest:{regime_spec.name}", leave=False):
        waveform, _ = load_audio_file(get_audio_path(config.paths.audio_dir, int(track_id)), target_sr=config.sample_rate, mono=True, cache_dir=config.paths.waveform_cache_dir)
        total_samples = int(waveform.shape[0])
        for query_length_seconds in regime_spec.lengths_seconds:
            target_length = int(float(query_length_seconds) * config.sample_rate)
            base_seed = int(config.seed + int(track_id) * 7919 + int(query_length_seconds * 1000))
            rng = np.random.default_rng(base_seed)
            if regime_spec.group_size == 1:
                for condition_name in list(regime_spec.conditions):
                    start_sample = _sample_single_query_start(total_samples, target_length, regime_spec.centered, regime_spec.random_offset, rng)
                    query_group_id = f"{regime_spec.name}:{track_id}:{condition_name}:{query_length_seconds:.2f}"
                    rows.append({"record_id": f"{query_group_id}:0", "query_group_id": query_group_id, "track_id": int(track_id), "regime_name": regime_spec.name, "condition_name": condition_name, "query_length_seconds": float(query_length_seconds), "group_size": 1, "query_order": 0, "segment_weight": 1.0, "start_sample": int(start_sample), "end_sample": int(start_sample + target_length), "start_seconds": float(start_sample / config.sample_rate), "query_time_seconds": 0.0, "apply_fade_envelope": bool(regime_spec.apply_fade_envelope)})
            else:
                jitter_seconds = 0.2 if regime_spec.name == "multi_segment_temporally_jittered" else 0.0
                start_samples = _build_group_starts(total_samples, target_length, regime_spec.group_size, rng, jitter_seconds)
                query_group_id = f"{regime_spec.name}:{track_id}:{query_length_seconds:.2f}"
                default_weights = np.linspace(1.0, 0.6, regime_spec.group_size, dtype=np.float32).tolist()
                for query_order, start_sample in enumerate(start_samples):
                    if regime_spec.name == "multi_segment_fragmented_weighted":
                        condition_name = regime_spec.conditions[min(query_order, len(regime_spec.conditions) - 1)]
                        segment_weight = float(default_weights[query_order])
                    else:
                        condition_name = regime_spec.conditions[query_order % len(regime_spec.conditions)]
                        segment_weight = 1.0
                    rows.append({"record_id": f"{query_group_id}:{query_order}", "query_group_id": query_group_id, "track_id": int(track_id), "regime_name": regime_spec.name, "condition_name": condition_name, "query_length_seconds": float(query_length_seconds), "group_size": int(regime_spec.group_size), "query_order": int(query_order), "segment_weight": float(segment_weight), "start_sample": int(start_sample), "end_sample": int(start_sample + target_length), "start_seconds": float(start_sample / config.sample_rate), "query_time_seconds": float(query_order * query_length_seconds), "apply_fade_envelope": bool(regime_spec.apply_fade_envelope or regime_spec.name == "multi_segment_fragmented_weighted")})
    manifest_df = pd.DataFrame(rows)
    save_dataframe(manifest_path, manifest_df)
    return manifest_df


QUERY_REGIME_REGISTRY = build_query_regime_registry(CONFIG)
WINDOWING_REGISTRY = build_windowing_registry(CONFIG)
QUERY_REGIME_SUMMARY = pd.DataFrame([serialize_jsonable(regime) for regime in QUERY_REGIME_REGISTRY.values()])
WINDOWING_SUMMARY = pd.DataFrame([serialize_jsonable(spec) for spec in WINDOWING_REGISTRY.values()])
display(QUERY_REGIME_SUMMARY)
display(WINDOWING_SUMMARY)


In [ ]:
from sklearn.linear_model import HuberRegressor


def extract_embeddings_with_cache_v2(
    model: nn.Module,
    loader: DataLoader[dict[str, object]],
    input_mode: InputMode,
    manifest_df: pd.DataFrame,
    cache_namespace: str,
    model_cache_key: str,
) -> tuple[np.ndarray, pd.DataFrame, dict[str, object]]:
    """Encode a manifest into embeddings while honoring Notebook 4 cache overrides."""

    manifest_hash = dataframe_hash(manifest_df)
    cache_key = hash_experiment_config({"namespace": cache_namespace, "manifest_hash": manifest_hash, "model_cache_key": model_cache_key})
    embedding_path = CONFIG.paths.embedding_cache_dir / f"{cache_key}.npy"
    metadata_path = CONFIG.paths.embedding_cache_dir / f"{cache_key}.csv"
    if embedding_path.exists() and metadata_path.exists() and not CONFIG.force_recompute_embeddings:
        return (
            np.load(embedding_path).astype(np.float32),
            pd.read_csv(metadata_path),
            {"cache_key": cache_key, "cache_hit": True, "embedding_path": embedding_path, "metadata_path": metadata_path},
        )

    model.eval()
    embedding_chunks: list[np.ndarray] = []
    metadata_rows: list[dict[str, object]] = []
    for batch in tqdm(loader, desc=f"embed:{cache_namespace}", leave=False):
        encoded = encode_retrieval_batch(model, batch, input_mode, CONFIG.device, CONFIG.device_type, CONFIG.amp_enabled)
        embedding_chunks.append(encoded.detach().cpu().numpy().astype(np.float32))
        metadata_rows.extend([dict(metadata_row) for metadata_row in batch["metadata_rows"]])
    embeddings = np.concatenate(embedding_chunks, axis=0).astype(np.float32)
    metadata_df = pd.DataFrame(metadata_rows)
    np.save(embedding_path, embeddings)
    save_dataframe(metadata_path, metadata_df)
    return embeddings, metadata_df, {"cache_key": cache_key, "cache_hit": False, "embedding_path": embedding_path, "metadata_path": metadata_path}


def build_or_load_faiss_index_v2(reference_embeddings: np.ndarray, index_spec: IndexSpec, reference_cache_key: str) -> tuple[faiss.Index, dict[str, object]]:
    """Build or load one FAISS index while honoring Notebook 4 rebuild controls."""

    index_cache_key = hash_experiment_config({"reference_cache_key": reference_cache_key, "index_name": index_spec.name, "index_kind": index_spec.kind})
    index_path = CONFIG.paths.index_dir / f"{index_cache_key}.faiss"
    metadata_path = CONFIG.paths.index_dir / f"{index_cache_key}.json"
    if index_path.exists() and metadata_path.exists() and not CONFIG.force_rebuild_indices:
        return faiss.read_index(str(index_path)), dict(load_json(metadata_path))

    normalized_embeddings = normalize_embeddings(reference_embeddings)
    reference_count, embed_dim = normalized_embeddings.shape
    runtime_parameters = resolve_index_runtime_parameters(index_spec, reference_count, embed_dim)
    build_started_at = time.perf_counter()
    if index_spec.kind == "exact_ip":
        index = faiss.IndexFlatIP(embed_dim)
        index.add(normalized_embeddings)
    elif index_spec.kind == "ivfflat":
        if reference_count < max(runtime_parameters["nlist"] * 8, 256):
            raise ValueError("Not enough reference windows to train IVFFlat stably.")
        quantizer = faiss.IndexFlatIP(embed_dim)
        index = faiss.IndexIVFFlat(quantizer, embed_dim, runtime_parameters["nlist"], faiss.METRIC_INNER_PRODUCT)
        index.train(normalized_embeddings)
        index.add(normalized_embeddings)
        index.nprobe = runtime_parameters["nprobe"]
    elif index_spec.kind == "ivfpq":
        if reference_count < max(runtime_parameters["nlist"] * 8, 256):
            raise ValueError("Not enough reference windows to train IVFPQ stably.")
        quantizer = faiss.IndexFlatIP(embed_dim)
        index = faiss.IndexIVFPQ(quantizer, embed_dim, runtime_parameters["nlist"], runtime_parameters["pq_m"], 8, faiss.METRIC_INNER_PRODUCT)
        index.train(normalized_embeddings)
        index.add(normalized_embeddings)
        index.nprobe = runtime_parameters["nprobe"]
    else:
        raise ValueError(f"Unsupported index kind: {index_spec.kind}")

    build_seconds = float(time.perf_counter() - build_started_at)
    with tempfile.TemporaryDirectory() as temporary_dir:
        temp_index_path = Path(temporary_dir) / "index.faiss"
        faiss.write_index(index, str(temp_index_path))
        index_size_mb = float(temp_index_path.stat().st_size / (1024 ** 2))
    metadata = {
        "index_name": index_spec.name,
        "index_kind": index_spec.kind,
        "build_seconds": build_seconds,
        "index_size_mb": index_size_mb,
        "reference_count": int(reference_count),
        "embed_dim": int(embed_dim),
        **runtime_parameters,
    }
    faiss.write_index(index, str(index_path))
    save_json(metadata_path, metadata)
    return index, metadata


def search_segment_neighbors(
    index: faiss.Index,
    query_embeddings: np.ndarray,
    reference_metadata_df: pd.DataFrame,
    top_k_per_segment: int,
    oversample_factor: int,
) -> pd.DataFrame:
    """Search raw segment neighbors without deduplicating tracks."""

    if query_embeddings.size == 0:
        return pd.DataFrame()
    normalized_queries = normalize_embeddings(query_embeddings)
    total_reference = int(reference_metadata_df.shape[0])
    search_k = min(total_reference, max(int(top_k_per_segment), int(top_k_per_segment) * int(oversample_factor)))
    search_started_at = time.perf_counter()
    scores, neighbor_indices = index.search(normalized_queries.astype(np.float32), search_k)
    elapsed_seconds = float(time.perf_counter() - search_started_at)

    rows: list[dict[str, object]] = []
    for query_row_index, (score_row, index_row) in enumerate(zip(scores, neighbor_indices)):
        for neighbor_rank, (score_value, reference_row_index) in enumerate(zip(score_row, index_row), start=1):
            if int(reference_row_index) < 0:
                continue
            reference_row = reference_metadata_df.iloc[int(reference_row_index)]
            rows.append(
                {
                    "query_row_index": int(query_row_index),
                    "neighbor_rank": int(neighbor_rank),
                    "similarity": float(score_value),
                    "candidate_track_id": int(reference_row["track_id"]),
                    "reference_row_index": int(reference_row_index),
                    "reference_record_id": str(reference_row["record_id"]),
                    "reference_window_index": int(reference_row.get("window_index", 0)),
                    "reference_start_seconds": float(reference_row.get("start_seconds", 0.0)),
                    "reference_end_seconds": float(reference_row.get("end_seconds", 0.0)),
                }
            )
    neighbors_df = pd.DataFrame(rows)
    neighbors_df.attrs["search_elapsed_seconds"] = elapsed_seconds
    neighbors_df.attrs["search_k"] = search_k
    return neighbors_df


def _fit_huber_temporal_model(query_times: np.ndarray, reference_times: np.ndarray, sample_weights: np.ndarray | None) -> tuple[np.ndarray, float, float]:
    """Fit a robust temporal line and return predictions, slope, and intercept."""

    if query_times.shape[0] < 2 or float(np.ptp(query_times)) <= 1e-8:
        intercept = float(np.mean(reference_times)) if reference_times.size else 0.0
        return np.full_like(reference_times, intercept, dtype=np.float32), 0.0, intercept
    model = HuberRegressor(epsilon=1.35, alpha=1e-4, max_iter=200, fit_intercept=True)
    model.fit(query_times.reshape(-1, 1), reference_times, sample_weight=sample_weights)
    predicted = model.predict(query_times.reshape(-1, 1)).astype(np.float32)
    return predicted, float(model.coef_[0]), float(model.intercept_)


def compute_temporal_consistency(candidate_df: pd.DataFrame, aggregation_config: AggregationConfig) -> dict[str, float]:
    """Score how well a candidate track preserves segment ordering over time."""

    if candidate_df.empty:
        return {
            "temporal_score": 0.0,
            "temporal_slope": 0.0,
            "temporal_intercept": 0.0,
            "temporal_inlier_count": 0.0,
            "temporal_inlier_fraction": 0.0,
            "temporal_mean_abs_residual": float("inf"),
        }

    best_rows = (
        candidate_df.sort_values(["query_row_index", "similarity"], ascending=[True, False])
        .drop_duplicates(subset=["query_row_index"], keep="first")
        .reset_index(drop=True)
    )
    query_times = best_rows["query_time_seconds"].astype(float).to_numpy(dtype=np.float32)
    reference_times = best_rows["reference_start_seconds"].astype(float).to_numpy(dtype=np.float32)
    weights = (
        best_rows["segment_weight"].astype(float).to_numpy(dtype=np.float32)
        * np.clip(best_rows["similarity"].astype(float).to_numpy(dtype=np.float32), a_min=0.0, a_max=None)
    )
    sample_weights = weights if np.any(weights > 0) else None
    predicted_times, slope, intercept = _fit_huber_temporal_model(query_times, reference_times, sample_weights)
    residuals = np.abs(reference_times - predicted_times).astype(np.float32)
    inlier_mask = residuals <= float(aggregation_config.residual_tolerance_seconds)
    inlier_count = int(np.sum(inlier_mask))
    inlier_fraction = float(inlier_count / max(1, residuals.shape[0]))
    mean_abs_residual = float(np.mean(residuals)) if residuals.size else float("inf")
    fit_quality = float(1.0 / (1.0 + mean_abs_residual)) if np.isfinite(mean_abs_residual) else 0.0
    temporal_score = float(
        aggregation_config.temporal_inlier_weight * inlier_fraction
        + aggregation_config.fit_quality_weight * fit_quality
    )
    return {
        "temporal_score": temporal_score,
        "temporal_slope": float(slope),
        "temporal_intercept": float(intercept),
        "temporal_inlier_count": float(inlier_count),
        "temporal_inlier_fraction": inlier_fraction,
        "temporal_mean_abs_residual": mean_abs_residual,
    }


def aggregate_group_rankings_with_mode(
    query_metadata_df: pd.DataFrame,
    segment_neighbors_df: pd.DataFrame,
    aggregation_config: AggregationConfig,
    aggregation_mode: AggregationModeName,
) -> pd.DataFrame:
    """Aggregate segment-level retrieval results using one Notebook 4 scoring mode."""

    if query_metadata_df.empty or segment_neighbors_df.empty:
        return pd.DataFrame()

    query_metadata = query_metadata_df.reset_index(drop=True).copy()
    query_metadata["query_row_index"] = query_metadata.index.astype(int)
    enriched_neighbors = segment_neighbors_df.merge(
        query_metadata[
            [
                "query_row_index",
                "query_group_id",
                "track_id",
                "regime_name",
                "condition_name",
                "query_length_seconds",
                "group_size",
                "query_order",
                "query_time_seconds",
                "segment_weight",
            ]
        ],
        on="query_row_index",
        how="left",
    )

    grouped_rows: list[dict[str, object]] = []
    for query_group_id, group_df in query_metadata.groupby("query_group_id", sort=False):
        group_neighbors = enriched_neighbors[enriched_neighbors["query_group_id"] == query_group_id].copy()
        candidate_rows: list[dict[str, object]] = []
        for candidate_track_id, candidate_df in group_neighbors.groupby("candidate_track_id", sort=False):
            vote_count = int(candidate_df["query_row_index"].nunique())
            mean_similarity = float(candidate_df["similarity"].mean())
            weighted_vote_score = float(
                np.sum(
                    candidate_df["similarity"].astype(float).to_numpy()
                    * candidate_df["segment_weight"].astype(float).to_numpy()
                    * (1.0 / candidate_df["neighbor_rank"].astype(float).to_numpy())
                )
            )
            temporal_metrics = compute_temporal_consistency(candidate_df, aggregation_config)

            if aggregation_mode == "single_segment_top1":
                candidate_score = float(candidate_df["similarity"].max())
            elif aggregation_mode == "segment_vote_topk":
                candidate_score = float(vote_count + 0.05 * mean_similarity)
            elif aggregation_mode == "weighted_segment_vote":
                candidate_score = float(
                    aggregation_config.weighted_similarity_weight * weighted_vote_score
                    + aggregation_config.vote_count_weight * vote_count
                )
            elif aggregation_mode == "temporal_consistency_huber":
                candidate_score = float(
                    aggregation_config.weighted_similarity_weight * weighted_vote_score
                    + aggregation_config.vote_count_weight * vote_count
                    + temporal_metrics["temporal_score"]
                )
            else:
                raise ValueError(f"Unsupported aggregation mode: {aggregation_mode}")

            candidate_rows.append(
                {
                    "candidate_track_id": int(candidate_track_id),
                    "candidate_score": float(candidate_score),
                    "vote_count": vote_count,
                    "mean_similarity": mean_similarity,
                    "weighted_vote_score": weighted_vote_score,
                    **temporal_metrics,
                }
            )

        candidate_rows = sorted(candidate_rows, key=lambda row: row["candidate_score"], reverse=True)
        true_track_id = int(group_df["track_id"].iloc[0])
        rank = next((candidate_index + 1 for candidate_index, row in enumerate(candidate_rows) if int(row["candidate_track_id"]) == true_track_id), len(candidate_rows) + 1)
        top_candidates = candidate_rows[: max(CONFIG.top_k)]
        top_candidate_track_id = int(top_candidates[0]["candidate_track_id"]) if top_candidates else -1
        top_candidate_score = float(top_candidates[0]["candidate_score"]) if top_candidates else float("nan")
        top_temporal_metrics = top_candidates[0] if top_candidates else {}
        grouped_rows.append(
            {
                "query_group_id": str(query_group_id),
                "true_track_id": true_track_id,
                "regime_name": str(group_df["regime_name"].iloc[0]),
                "condition_name": str(group_df["condition_name"].iloc[0]),
                "query_length_seconds": float(group_df["query_length_seconds"].iloc[0]),
                "group_size": int(group_df["group_size"].iloc[0]),
                "aggregation_mode": aggregation_mode,
                "rank": int(rank),
                "correct_top1": bool(rank == 1),
                "top1_track_id": top_candidate_track_id,
                "top1_score": top_candidate_score,
                "top_candidate_track_ids_json": json.dumps([int(row["candidate_track_id"]) for row in top_candidates]),
                "top_candidate_scores_json": json.dumps([float(row["candidate_score"]) for row in top_candidates]),
                "temporal_inlier_count": float(top_temporal_metrics.get("temporal_inlier_count", 0.0)),
                "temporal_inlier_fraction": float(top_temporal_metrics.get("temporal_inlier_fraction", 0.0)),
                "temporal_mean_abs_residual": float(top_temporal_metrics.get("temporal_mean_abs_residual", float("nan"))),
            }
        )
    return pd.DataFrame(grouped_rows)


def build_hard_negative_candidate_manifest(
    track_ids: list[int],
    config: ExperimentConfig,
    split_name: str,
    force_rebuild: bool = False,
) -> pd.DataFrame:
    """Build or load the candidate reference manifest used for hard-negative mining."""

    if not track_ids:
        raise ValueError("Hard-negative candidate manifest requires at least one track ID.")
    capped_track_ids = [int(track_id) for track_id in track_ids[: config.hard_negative_config.candidate_track_cap]]
    candidate_window_spec = WindowingSpec(
        name="multi5_even" if config.hard_negative_config.candidate_segments_per_track > 1 else "single_center",
        description="Hard-negative mining candidate windows.",
        window_count=max(1, int(config.hard_negative_config.candidate_segments_per_track)),
        segment_seconds=config.segment_seconds,
        centered_only=bool(config.hard_negative_config.candidate_segments_per_track <= 1),
        enabled=True,
    )
    cache_key = hash_experiment_config({"split_name": split_name, "track_ids": capped_track_ids, "candidate_window_spec": candidate_window_spec})
    manifest_path = config.paths.hard_negative_cache_dir / f"candidate_manifest_{split_name}_{cache_key}.csv"
    if manifest_path.exists() and not force_rebuild and not config.force_recompute_embeddings:
        return pd.read_csv(manifest_path)
    manifest_df = build_reference_manifest(capped_track_ids, candidate_window_spec, config, force_rebuild=force_rebuild)
    save_dataframe(manifest_path, manifest_df)
    return manifest_df


class HardNegativePairDataset(Dataset[dict[str, object]]):
    """Yield anchor, positive, and mined hard-negative examples for retraining."""

    def __init__(
        self,
        audio_dir: Path,
        negative_pairs_df: pd.DataFrame,
        segment_length: int,
        sample_rate: int,
        input_mode: InputMode,
        config: ExperimentConfig,
        augmentation_profile: str,
    ) -> None:
        if negative_pairs_df.empty:
            raise ValueError("HardNegativePairDataset requires at least one mined negative pair.")
        self.audio_dir = audio_dir
        self.negative_pairs_df = negative_pairs_df.reset_index(drop=True)
        self.segment_length = int(segment_length)
        self.sample_rate = int(sample_rate)
        self.input_mode = input_mode
        self.config = config
        self.augmentation_profile = augmentation_profile
        self.current_epoch = 0

    def __len__(self) -> int:
        return int(self.negative_pairs_df.shape[0])

    def __getitem__(self, index: int) -> dict[str, object]:
        row = self.negative_pairs_df.iloc[int(index)]
        anchor_track_id = int(row["anchor_track_id"])
        negative_track_id = int(row["hard_negative_track_id"])
        seed = int(torch.randint(low=0, high=2**31 - 1, size=(1,)).item())
        rng = np.random.default_rng(seed)

        anchor_waveform, _ = load_audio_file(get_audio_path(self.audio_dir, anchor_track_id), target_sr=self.sample_rate, mono=True, cache_dir=self.config.paths.waveform_cache_dir)
        negative_waveform, _ = load_audio_file(get_audio_path(self.audio_dir, negative_track_id), target_sr=self.sample_rate, mono=True, cache_dir=self.config.paths.waveform_cache_dir)
        anchor_segment, start_index = extract_random_segment(anchor_waveform, self.segment_length, rng=rng)
        anchor_segment = normalize_waveform(trim_or_pad_waveform(anchor_segment, self.segment_length))
        positive_waveform, _ = apply_augmented_policy(anchor_segment.copy(), self.sample_rate, self.segment_length, self.augmentation_profile, rng, self.current_epoch)
        hard_negative_segment, _ = extract_random_segment(negative_waveform, self.segment_length, rng=rng)
        hard_negative_segment = normalize_waveform(trim_or_pad_waveform(hard_negative_segment, self.segment_length))

        if self.input_mode == "spectrogram":
            anchor_value: torch.Tensor = spectrogram_tensor_from_waveform(anchor_segment, self.config)
            positive_value: torch.Tensor = spectrogram_tensor_from_waveform(positive_waveform, self.config)
            negative_value: torch.Tensor = spectrogram_tensor_from_waveform(hard_negative_segment, self.config)
        else:
            anchor_value = torch.from_numpy(anchor_segment).float()
            positive_value = torch.from_numpy(positive_waveform).float()
            negative_value = torch.from_numpy(hard_negative_segment).float()

        return {
            "anchor": anchor_value,
            "positive": positive_value,
            "hard_negative": negative_value,
            "track_id": anchor_track_id,
            "negative_track_id": negative_track_id,
            "anchor_start_time": float(start_index / self.sample_rate),
        }


def collate_hard_negative_pair_batch(batch: list[dict[str, object]]) -> dict[str, object]:
    """Stack spectrogram hard-negative triplets into one batch."""

    return {
        "anchor": torch.stack([sample["anchor"] for sample in batch]),
        "positive": torch.stack([sample["positive"] for sample in batch]),
        "hard_negative": torch.stack([sample["hard_negative"] for sample in batch]),
        "track_id": torch.tensor([int(sample["track_id"]) for sample in batch], dtype=torch.long),
        "negative_track_id": torch.tensor([int(sample["negative_track_id"]) for sample in batch], dtype=torch.long),
        "anchor_start_time": torch.tensor([float(sample["anchor_start_time"]) for sample in batch], dtype=torch.float32),
    }


def build_mert_hard_negative_collate_fn(
    feature_extractor: Wav2Vec2FeatureExtractor,
) -> Callable[[list[dict[str, object]]], dict[str, object]]:
    """Build the waveform collate function used for MERT hard-negative retraining."""

    def collate(batch: list[dict[str, object]]) -> dict[str, object]:
        anchor_arrays = [np.asarray(sample["anchor"], dtype=np.float32) for sample in batch]
        positive_arrays = [np.asarray(sample["positive"], dtype=np.float32) for sample in batch]
        negative_arrays = [np.asarray(sample["hard_negative"], dtype=np.float32) for sample in batch]
        anchor_inputs = feature_extractor(anchor_arrays, sampling_rate=feature_extractor.sampling_rate, return_tensors="pt", padding=True)
        positive_inputs = feature_extractor(positive_arrays, sampling_rate=feature_extractor.sampling_rate, return_tensors="pt", padding=True)
        negative_inputs = feature_extractor(negative_arrays, sampling_rate=feature_extractor.sampling_rate, return_tensors="pt", padding=True)
        return {
            "anchor_input_values": anchor_inputs["input_values"],
            "anchor_attention_mask": anchor_inputs["attention_mask"],
            "positive_input_values": positive_inputs["input_values"],
            "positive_attention_mask": positive_inputs["attention_mask"],
            "negative_input_values": negative_inputs["input_values"],
            "negative_attention_mask": negative_inputs["attention_mask"],
            "track_id": torch.tensor([int(sample["track_id"]) for sample in batch], dtype=torch.long),
            "negative_track_id": torch.tensor([int(sample["negative_track_id"]) for sample in batch], dtype=torch.long),
            "anchor_start_time": torch.tensor([float(sample["anchor_start_time"]) for sample in batch], dtype=torch.float32),
        }

    return collate


class HardNegativeLoss(nn.Module):
    """Combine the existing NT-Xent objective with a triplet margin term."""

    def __init__(self, temperature: float, triplet_margin: float, triplet_weight: float) -> None:
        super().__init__()
        self.ntxent = NTXentLoss(temperature)
        self.triplet = nn.TripletMarginLoss(margin=float(triplet_margin), p=2)
        self.triplet_weight = float(triplet_weight)

    def forward(
        self,
        anchor_embeddings: torch.Tensor,
        positive_embeddings: torch.Tensor,
        negative_embeddings: torch.Tensor,
    ) -> torch.Tensor:
        contrastive_loss = self.ntxent(anchor_embeddings, positive_embeddings)
        triplet_loss = self.triplet(
            F.normalize(anchor_embeddings, p=2, dim=1),
            F.normalize(positive_embeddings, p=2, dim=1),
            F.normalize(negative_embeddings, p=2, dim=1),
        )
        return contrastive_loss + self.triplet_weight * triplet_loss


def forward_hard_negative_batch(
    model: nn.Module,
    batch: dict[str, torch.Tensor],
    input_mode: InputMode,
    device_type: str,
    amp_enabled: bool,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute anchor, positive, and hard-negative embeddings for one batch."""

    with torch.amp.autocast(device_type=device_type, enabled=amp_enabled):
        if input_mode == "spectrogram":
            anchor_embeddings = model(batch["anchor"])
            positive_embeddings = model(batch["positive"])
            negative_embeddings = model(batch["hard_negative"])
        else:
            if not isinstance(model, FrozenMertFingerprinter):
                raise TypeError("Waveform batches require FrozenMertFingerprinter.")
            anchor_embeddings = model(batch["anchor_input_values"], batch["anchor_attention_mask"])
            positive_embeddings = model(batch["positive_input_values"], batch["positive_attention_mask"])
            negative_embeddings = model(batch["negative_input_values"], batch["negative_attention_mask"])
    return anchor_embeddings, positive_embeddings, negative_embeddings


def train_one_epoch_with_hard_negatives(
    model: nn.Module,
    loader: DataLoader[dict[str, object]],
    optimizer: torch.optim.Optimizer,
    loss_fn: HardNegativeLoss,
    device: torch.device,
    input_mode: InputMode,
    config: ExperimentConfig,
) -> float:
    """Train one epoch using mined hard negatives."""

    model.train()
    if isinstance(model, FrozenMertFingerprinter) and model.freeze_backbone:
        model.backbone.eval()
    scaler = torch.amp.GradScaler(config.device_type, enabled=config.amp_enabled)
    epoch_losses: list[float] = []
    for batch in tqdm(loader, desc="train-hard-negatives", leave=False):
        batch = move_batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)
        anchor_embeddings, positive_embeddings, negative_embeddings = forward_hard_negative_batch(model, batch, input_mode, config.device_type, config.amp_enabled)
        loss = loss_fn(anchor_embeddings, positive_embeddings, negative_embeddings)
        if not torch.isfinite(loss):
            raise FloatingPointError("Encountered a non-finite hard-negative training loss.")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.gradient_clip_norm)
        scaler.step(optimizer)
        scaler.update()
        epoch_losses.append(float(loss.detach().cpu().item()))
    return float(np.mean(epoch_losses)) if epoch_losses else float("nan")


def validate_epoch_with_hard_negatives(
    model: nn.Module,
    loader: DataLoader[dict[str, object]],
    loss_fn: HardNegativeLoss,
    device: torch.device,
    input_mode: InputMode,
    config: ExperimentConfig,
) -> float:
    """Evaluate hard-negative loss on a validation loader."""

    model.eval()
    losses: list[float] = []
    for batch in tqdm(loader, desc="validate-hard-negatives", leave=False):
        batch = move_batch_to_device(batch, device)
        anchor_embeddings, positive_embeddings, negative_embeddings = forward_hard_negative_batch(model, batch, input_mode, config.device_type, config.amp_enabled)
        loss = loss_fn(anchor_embeddings, positive_embeddings, negative_embeddings)
        losses.append(float(loss.detach().cpu().item()))
    return float(np.mean(losses)) if losses else float("nan")


def mine_hard_negative_cache(
    model: nn.Module,
    input_mode: InputMode,
    descriptor: ArtifactDescriptor,
    candidate_track_ids: list[int],
    split_name: str,
    batch_size: int,
    collate_fn: Callable[[list[dict[str, object]]], dict[str, object]],
    sample_rate: int,
    config: ExperimentConfig,
) -> pd.DataFrame:
    """Mine near-miss negatives from a capped candidate pool and cache the result."""

    if len(candidate_track_ids) < 2:
        return pd.DataFrame(columns=["anchor_track_id", "hard_negative_track_id", "negative_score", "split_name", "source_run_id"])

    capped_track_ids = [int(track_id) for track_id in candidate_track_ids[: config.hard_negative_config.candidate_track_cap]]
    cache_key = hash_experiment_config(
        {
            "descriptor_run_id": descriptor.run_id,
            "split_name": split_name,
            "candidate_track_ids": capped_track_ids,
            "candidate_segments_per_track": config.hard_negative_config.candidate_segments_per_track,
            "neighbors_to_search": config.hard_negative_config.neighbors_to_search,
            "negatives_per_anchor": config.hard_negative_config.negatives_per_anchor,
        }
    )
    cache_path = config.paths.hard_negative_cache_dir / f"{descriptor.run_id}_{split_name}_{cache_key}.csv"
    metadata_path = config.paths.hard_negative_cache_dir / f"{descriptor.run_id}_{split_name}_{cache_key}.json"
    if cache_path.exists() and metadata_path.exists() and not config.force_recompute_embeddings:
        return pd.read_csv(cache_path)

    candidate_manifest = build_hard_negative_candidate_manifest(capped_track_ids, config, split_name, force_rebuild=config.force_recompute_embeddings)
    candidate_loader = make_retrieval_loader_v2(
        ReferenceManifestDataset(config.paths.audio_dir, candidate_manifest, config.sample_rate if input_mode == "spectrogram" else sample_rate, input_mode, config),
        batch_size,
        collate_fn,
        config.num_workers,
    )
    candidate_embeddings, candidate_metadata_df, candidate_cache = extract_embeddings_with_cache_v2(
        model,
        candidate_loader,
        input_mode,
        candidate_manifest,
        f"hard_negative_candidates_{split_name}",
        descriptor.run_id,
    )
    normalized_embeddings = normalize_embeddings(candidate_embeddings)
    index = faiss.IndexFlatIP(normalized_embeddings.shape[1])
    index.add(normalized_embeddings)
    search_k = min(normalized_embeddings.shape[0], max(4, int(config.hard_negative_config.neighbors_to_search)))
    scores, neighbors = index.search(normalized_embeddings.astype(np.float32), search_k)

    rows: list[dict[str, object]] = []
    for anchor_row_index, (score_row, neighbor_row) in enumerate(zip(scores, neighbors)):
        anchor_track_id = int(candidate_metadata_df.iloc[int(anchor_row_index)]["track_id"])
        written = 0
        seen_track_ids: set[int] = set()
        for score_value, neighbor_row_index in zip(score_row, neighbor_row):
            if int(neighbor_row_index) < 0:
                continue
            negative_track_id = int(candidate_metadata_df.iloc[int(neighbor_row_index)]["track_id"])
            if negative_track_id == anchor_track_id or negative_track_id in seen_track_ids:
                continue
            seen_track_ids.add(negative_track_id)
            rows.append(
                {
                    "anchor_track_id": anchor_track_id,
                    "hard_negative_track_id": negative_track_id,
                    "negative_score": float(score_value),
                    "split_name": split_name,
                    "source_run_id": descriptor.run_id,
                }
            )
            written += 1
            if written >= int(config.hard_negative_config.negatives_per_anchor):
                break

    hard_negative_df = pd.DataFrame(rows)
    save_dataframe(cache_path, hard_negative_df)
    save_json(
        metadata_path,
        {
            "descriptor": descriptor,
            "split_name": split_name,
            "candidate_track_ids": capped_track_ids,
            "candidate_manifest_rows": int(candidate_manifest.shape[0]),
            "cache": candidate_cache,
            "pairs": int(hard_negative_df.shape[0]),
        },
    )
    return hard_negative_df


def build_metrics_long_table(detailed_results_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate query-group-level outputs into the Notebook 4 long-form metrics table."""

    if detailed_results_df.empty:
        return pd.DataFrame()
    metrics_df = detailed_results_df.copy()
    if "evaluation_phase" not in metrics_df.columns:
        metrics_df["evaluation_phase"] = "baseline_historical"
    if "aggregation_mode" not in metrics_df.columns:
        metrics_df["aggregation_mode"] = "baseline_mean"
    grouping_columns = [
        "evaluation_phase",
        "aggregation_mode",
        "run_id",
        "source",
        "model_name",
        "augmentation_profile",
        "embed_dim",
        "windowing_name",
        "index_name",
        "index_kind",
        "regime_name",
        "condition_name",
        "query_length_seconds",
    ]
    rows: list[dict[str, object]] = []
    for group_key, group_df in metrics_df.groupby(grouping_columns, dropna=False):
        metrics = compute_rank_metrics_from_series(group_df["rank"], CONFIG.top_k)
        row = dict(zip(grouping_columns, group_key))
        row.update(metrics)
        row["query_group_count"] = int(group_df.shape[0])
        row["mean_rank"] = float(group_df["rank"].astype(float).mean())
        row["median_rank"] = float(group_df["rank"].astype(float).median())
        row["group_size_mean"] = float(group_df["group_size"].astype(float).mean())
        rows.append(row)
    return pd.DataFrame(rows)


def build_phase_summary_table(metrics_long_df: pd.DataFrame, faiss_results_df: pd.DataFrame, evaluation_phase: str) -> pd.DataFrame:
    """Build one evaluation-phase summary table used for ranking and synthesis."""

    if metrics_long_df.empty:
        return pd.DataFrame()
    phase_df = metrics_long_df[metrics_long_df["evaluation_phase"] == evaluation_phase].copy()
    if phase_df.empty:
        return pd.DataFrame()
    configuration_columns = [
        "evaluation_phase",
        "aggregation_mode",
        "run_id",
        "source",
        "model_name",
        "augmentation_profile",
        "embed_dim",
        "windowing_name",
        "index_name",
        "index_kind",
    ]
    condition_rollup = phase_df.groupby(configuration_columns + ["condition_name"], dropna=False)["top1"].mean().reset_index()
    condition_pivot = condition_rollup.pivot_table(index=configuration_columns, columns="condition_name", values="top1", aggfunc="mean").reset_index()
    condition_pivot.columns = [column if isinstance(column, str) else str(column) for column in condition_pivot.columns]
    summary_df = condition_pivot.copy()
    for expected_condition in ["clean", "noise", "pitch", "time", "lowpass", "highpass", "combined_moderate", "realistic_hard"]:
        if expected_condition not in summary_df.columns:
            summary_df[expected_condition] = float("nan")
        summary_df.rename(columns={expected_condition: f"{expected_condition}_top1"}, inplace=True)
    degraded_columns = [column_name for column_name in summary_df.columns if column_name.endswith("_top1") and column_name != "clean_top1"]
    summary_df["mean_degraded_top1"] = summary_df[degraded_columns].mean(axis=1, skipna=True)
    summary_df["worst_condition_top1"] = summary_df[degraded_columns].min(axis=1, skipna=True)
    summary_df["robustness_gap"] = summary_df["clean_top1"] - summary_df["mean_degraded_top1"]

    mrr_rollup = phase_df.groupby(configuration_columns, dropna=False)["mrr"].mean().reset_index()
    query_count_rollup = phase_df.groupby(configuration_columns, dropna=False)["query_group_count"].sum().reset_index()
    regime_coverage_rollup = phase_df.groupby(configuration_columns, dropna=False)["regime_name"].nunique().reset_index().rename(columns={"regime_name": "regime_coverage"})
    summary_df = summary_df.merge(mrr_rollup, on=configuration_columns, how="left")
    summary_df = summary_df.merge(query_count_rollup, on=configuration_columns, how="left")
    summary_df = summary_df.merge(regime_coverage_rollup, on=configuration_columns, how="left")

    faiss_phase_df = faiss_results_df[(faiss_results_df["evaluation_phase"] == evaluation_phase) & (faiss_results_df["status"] == "ok")].copy()
    if not faiss_phase_df.empty:
        faiss_rollup = faiss_phase_df.groupby(configuration_columns, dropna=False)[["latency_ms_per_query", "index_size_mb"]].mean().reset_index()
        summary_df = summary_df.merge(faiss_rollup, on=configuration_columns, how="left")
        summary_df.rename(columns={"latency_ms_per_query": "latency_ms"}, inplace=True)
    else:
        summary_df["latency_ms"] = float("nan")
        summary_df["index_size_mb"] = float("nan")

    worst_latency = float(summary_df["latency_ms"].dropna().max()) if summary_df["latency_ms"].notna().any() else 1.0
    worst_index_size = float(summary_df["index_size_mb"].dropna().max()) if summary_df["index_size_mb"].notna().any() else 1.0
    summary_df["ranking_score"] = (
        0.45 * summary_df["mean_degraded_top1"].fillna(0.0)
        + 0.20 * summary_df["worst_condition_top1"].fillna(0.0)
        + 0.15 * summary_df["mrr"].fillna(0.0)
        + 0.10 * (1.0 / (1.0 + summary_df["latency_ms"].fillna(worst_latency)))
        + 0.05 * (1.0 / (1.0 + summary_df["index_size_mb"].fillna(worst_index_size)))
        + 0.05 * summary_df["clean_top1"].fillna(0.0)
    )
    return summary_df.sort_values("ranking_score", ascending=False).reset_index(drop=True)


def build_baseline_summary_table(metrics_long_df: pd.DataFrame, faiss_results_df: pd.DataFrame) -> pd.DataFrame:
    """Build the baseline historical summary table."""

    return build_phase_summary_table(metrics_long_df, faiss_results_df, "baseline_historical")


def build_aggregation_summary_table(metrics_long_df: pd.DataFrame, faiss_results_df: pd.DataFrame) -> pd.DataFrame:
    """Build the multi-segment aggregation summary table."""

    return build_phase_summary_table(metrics_long_df, faiss_results_df, "aggregation")


def build_hard_negative_summary_table(metrics_long_df: pd.DataFrame, faiss_results_df: pd.DataFrame) -> pd.DataFrame:
    """Build the hard-negative retraining summary table."""

    return build_phase_summary_table(metrics_long_df, faiss_results_df, "hard_negative")


def export_plot(
    dataframe: pd.DataFrame,
    filename: str,
    title: str,
    x_column: str,
    y_column: str,
    color: str = "#1f77b4",
    max_rows: int = 12,
) -> Path | None:
    """Export a compact bar chart for one summary dataframe."""

    if dataframe.empty or x_column not in dataframe.columns or y_column not in dataframe.columns:
        return None
    plot_df = dataframe.head(max_rows).copy()
    output_path = CONFIG.paths.plot_dir / filename
    safe_mkdir(output_path.parent)
    plt.figure(figsize=(max(8, min(16, plot_df.shape[0] + 2)), 6))
    plt.bar(plot_df[x_column].astype(str), plot_df[y_column].astype(float), color=color)
    plt.title(title)
    plt.xlabel(x_column)
    plt.ylabel(y_column)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close()
    return output_path


def ensure_hard_negative_artifact(descriptor: ArtifactDescriptor) -> ArtifactDescriptor:
    """Warm-start one historical model, retrain with mined negatives, and cache the result."""

    target_run_id = f"{descriptor.run_id}_hard_negative"
    run_dir = safe_mkdir(CONFIG.paths.checkpoint_dir / target_run_id)
    checkpoint_path = run_dir / "checkpoint_best.pt"
    history_path = run_dir / "history.json"
    summary_path = run_dir / "training_summary.json"
    if (
        checkpoint_path.exists()
        and history_path.exists()
        and summary_path.exists()
        and CONFIG.resume_from_partial_artifacts
        and not CONFIG.force_recompute_embeddings
    ):
        return ArtifactDescriptor(
            run_id=target_run_id,
            source="notebook4_hard_negative",
            model_name=descriptor.model_name,
            augmentation_profile=descriptor.augmentation_profile,
            embed_dim=descriptor.embed_dim,
            checkpoint_path=checkpoint_path,
            freeze_backbone=descriptor.freeze_backbone,
            notes=f"Cached hard-negative retraining artifact derived from {descriptor.run_id}.",
        )

    run_config = build_hard_negative_run_config(descriptor)
    model, input_mode, sample_rate, pair_collate_override = load_checkpoint_into_model(descriptor)
    batch_size = CONFIG.mert_batch_size if input_mode == "mert_waveform" else CONFIG.batch_size
    retrieval_collate_fn = build_retrieval_collate_fn(model, input_mode)
    hard_negative_collate_fn = (
        collate_hard_negative_pair_batch
        if input_mode == "spectrogram"
        else build_mert_hard_negative_collate_fn(model.feature_extractor)  # type: ignore[arg-type]
    )
    pair_collate_fn = pair_collate_override if pair_collate_override is not None else collate_pair_batch

    train_cache_df = mine_hard_negative_cache(
        model=model,
        input_mode=input_mode,
        descriptor=descriptor,
        candidate_track_ids=SPLIT_TRACK_IDS["training"],
        split_name="training",
        batch_size=batch_size,
        collate_fn=retrieval_collate_fn,
        sample_rate=sample_rate,
        config=CONFIG,
    )
    validation_track_ids = SPLIT_TRACK_IDS["validation"] if len(SPLIT_TRACK_IDS["validation"]) >= 2 else SPLIT_TRACK_IDS["training"][: max(2, min(64, len(SPLIT_TRACK_IDS["training"])))]
    validation_cache_df = mine_hard_negative_cache(
        model=model,
        input_mode=input_mode,
        descriptor=descriptor,
        candidate_track_ids=validation_track_ids,
        split_name="validation",
        batch_size=batch_size,
        collate_fn=retrieval_collate_fn,
        sample_rate=sample_rate,
        config=CONFIG,
    )
    if train_cache_df.empty:
        raise RuntimeError(f"Hard-negative mining did not produce any pairs for {descriptor.run_id}.")
    if validation_cache_df.empty:
        validation_cache_df = train_cache_df.head(min(len(train_cache_df), 256)).copy()

    segment_length = int(CONFIG.segment_seconds * sample_rate)
    train_dataset = HardNegativePairDataset(CONFIG.paths.audio_dir, train_cache_df, segment_length, sample_rate, input_mode, CONFIG, descriptor.augmentation_profile)
    validation_dataset = HardNegativePairDataset(CONFIG.paths.audio_dir, validation_cache_df, segment_length, sample_rate, input_mode, CONFIG, descriptor.augmentation_profile)
    train_loader = make_pair_loader_v2(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=hard_negative_collate_fn, seed=CONFIG.seed, num_workers=CONFIG.num_workers)
    validation_loader = make_pair_loader_v2(validation_dataset, batch_size=batch_size, shuffle=False, collate_fn=hard_negative_collate_fn, seed=CONFIG.seed, num_workers=CONFIG.num_workers)

    optimizer = torch.optim.AdamW([parameter for parameter in model.parameters() if parameter.requires_grad], lr=run_config.learning_rate, weight_decay=run_config.weight_decay)
    hard_negative_loss = HardNegativeLoss(run_config.temperature, CONFIG.hard_negative_config.triplet_margin, CONFIG.hard_negative_config.triplet_weight)
    history_rows: list[dict[str, object]] = []

    if CONFIG.hard_negative_config.warm_start_epochs > 0:
        warm_train_dataset = ContrastivePairDataset(CONFIG.paths.audio_dir, SPLIT_TRACK_IDS["training"], segment_length, sample_rate, input_mode, CONFIG, descriptor.augmentation_profile)
        warm_validation_dataset = ContrastivePairDataset(CONFIG.paths.audio_dir, validation_track_ids, segment_length, sample_rate, input_mode, CONFIG, descriptor.augmentation_profile)
        warm_train_loader = make_pair_loader_v2(warm_train_dataset, batch_size=batch_size, shuffle=True, collate_fn=pair_collate_fn, seed=CONFIG.seed, num_workers=CONFIG.num_workers)
        warm_validation_loader = make_pair_loader_v2(warm_validation_dataset, batch_size=batch_size, shuffle=False, collate_fn=pair_collate_fn, seed=CONFIG.seed, num_workers=CONFIG.num_workers)
        warm_loss = NTXentLoss(run_config.temperature)
        for warm_epoch in range(1, int(CONFIG.hard_negative_config.warm_start_epochs) + 1):
            warm_train_dataset.current_epoch = warm_epoch - 1
            warm_validation_dataset.current_epoch = warm_epoch - 1
            train_loss = train_one_epoch(model, warm_train_loader, optimizer, warm_loss, CONFIG.device, input_mode, CONFIG)
            validation_loss = validate_epoch(model, warm_validation_loader, warm_loss, CONFIG.device, input_mode, CONFIG)
            history_rows.append({"phase": "warm_start", "epoch": int(warm_epoch), "train_loss": train_loss, "val_loss": validation_loss})

    best_val_loss = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    for epoch in range(1, run_config.epochs + 1):
        train_dataset.current_epoch = epoch - 1
        validation_dataset.current_epoch = epoch - 1
        train_loss = train_one_epoch_with_hard_negatives(model, train_loader, optimizer, hard_negative_loss, CONFIG.device, input_mode, CONFIG)
        validation_loss = validate_epoch_with_hard_negatives(model, validation_loader, hard_negative_loss, CONFIG.device, input_mode, CONFIG)
        history_rows.append({"phase": "hard_negative", "epoch": int(epoch), "train_loss": train_loss, "val_loss": validation_loss})

        if validation_loss < best_val_loss:
            best_val_loss = validation_loss
            best_epoch = epoch
            epochs_without_improvement = 0
            if isinstance(model, FrozenMertFingerprinter):
                checkpoint_payload = {
                    "base_run_id": descriptor.run_id,
                    "hard_negative_run_id": target_run_id,
                    "run_config": serialize_jsonable(run_config),
                    "projection_state_dict": model.projection.state_dict(),
                    "backbone_name": model.model_name,
                    "best_epoch": best_epoch,
                    "best_val_loss": best_val_loss,
                }
            else:
                checkpoint_payload = {
                    "base_run_id": descriptor.run_id,
                    "hard_negative_run_id": target_run_id,
                    "run_config": serialize_jsonable(run_config),
                    "model_state_dict": model.state_dict(),
                    "best_epoch": best_epoch,
                    "best_val_loss": best_val_loss,
                }
            torch.save(checkpoint_payload, checkpoint_path)
        else:
            epochs_without_improvement += 1

        if CONFIG.hard_negative_config.refresh_after_epoch is not None and epoch < run_config.epochs and epoch % int(CONFIG.hard_negative_config.refresh_after_epoch) == 0:
            train_cache_df = mine_hard_negative_cache(model, input_mode, descriptor, SPLIT_TRACK_IDS["training"], "training", batch_size, retrieval_collate_fn, sample_rate, CONFIG)
            validation_cache_df = mine_hard_negative_cache(model, input_mode, descriptor, validation_track_ids, "validation", batch_size, retrieval_collate_fn, sample_rate, CONFIG)
            if not train_cache_df.empty:
                train_dataset = HardNegativePairDataset(CONFIG.paths.audio_dir, train_cache_df, segment_length, sample_rate, input_mode, CONFIG, descriptor.augmentation_profile)
                train_loader = make_pair_loader_v2(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=hard_negative_collate_fn, seed=CONFIG.seed + epoch, num_workers=CONFIG.num_workers)
            if not validation_cache_df.empty:
                validation_dataset = HardNegativePairDataset(CONFIG.paths.audio_dir, validation_cache_df, segment_length, sample_rate, input_mode, CONFIG, descriptor.augmentation_profile)
                validation_loader = make_pair_loader_v2(validation_dataset, batch_size=batch_size, shuffle=False, collate_fn=hard_negative_collate_fn, seed=CONFIG.seed + epoch, num_workers=CONFIG.num_workers)

        if epochs_without_improvement >= CONFIG.early_stopping_patience:
            break

    save_json(history_path, history_rows)
    save_json(
        summary_path,
        {
            "base_run_id": descriptor.run_id,
            "target_run_id": target_run_id,
            "run_config": run_config,
            "hard_negative_config": CONFIG.hard_negative_config,
            "best_epoch": best_epoch,
            "best_val_loss": best_val_loss,
        },
    )
    clear_memory()
    return ArtifactDescriptor(
        run_id=target_run_id,
        source="notebook4_hard_negative",
        model_name=descriptor.model_name,
        augmentation_profile=descriptor.augmentation_profile,
        embed_dim=descriptor.embed_dim,
        checkpoint_path=checkpoint_path,
        freeze_backbone=descriptor.freeze_backbone,
        notes=f"Hard-negative retraining artifact derived from {descriptor.run_id}.",
    )


def evaluate_descriptor_baseline(
    descriptor: ArtifactDescriptor,
    regime_names: tuple[str, ...],
    evaluation_phase: str,
) -> tuple[pd.DataFrame, list[dict[str, object]]]:
    """Evaluate one artifact with the baseline Notebook 3 retrieval protocol."""

    detailed_frames: list[pd.DataFrame] = []
    faiss_rows: list[dict[str, object]] = []
    model, input_mode, sample_rate, _ = load_checkpoint_into_model(descriptor)
    retrieval_collate_fn = build_retrieval_collate_fn(model, input_mode)
    batch_size = CONFIG.mert_batch_size if input_mode == "mert_waveform" else CONFIG.batch_size
    index_registry = build_index_spec_registry(CONFIG)

    for windowing_name in CONFIG.active_windowing_names:
        windowing_spec = WINDOWING_REGISTRY[windowing_name]
        if not windowing_spec.enabled:
            continue
        reference_manifest = build_reference_manifest(SPLIT_TRACK_IDS["test"], windowing_spec, CONFIG, force_rebuild=CONFIG.force_recompute_embeddings)
        reference_loader = make_retrieval_loader_v2(
            ReferenceManifestDataset(CONFIG.paths.audio_dir, reference_manifest, CONFIG.sample_rate if input_mode == "spectrogram" else sample_rate, input_mode, CONFIG),
            batch_size,
            retrieval_collate_fn,
            CONFIG.num_workers,
        )
        reference_embeddings, reference_metadata_df, reference_cache = extract_embeddings_with_cache_v2(model, reference_loader, input_mode, reference_manifest, f"reference_{windowing_name}", descriptor.run_id)
        reference_track_ids = reference_metadata_df["track_id"].to_numpy(dtype=np.int64)

        for index_name in CONFIG.active_index_names:
            index_spec = index_registry[index_name]
            try:
                index, index_metadata = build_or_load_faiss_index_v2(reference_embeddings, index_spec, str(reference_cache["cache_key"]))
            except Exception as exc:
                faiss_rows.append(
                    {
                        "evaluation_phase": evaluation_phase,
                        "aggregation_mode": "baseline_mean",
                        "run_id": descriptor.run_id,
                        "source": descriptor.source,
                        "model_name": descriptor.model_name,
                        "augmentation_profile": descriptor.augmentation_profile,
                        "embed_dim": descriptor.embed_dim,
                        "windowing_name": windowing_name,
                        "index_name": index_name,
                        "index_kind": index_spec.kind,
                        "status": "skipped",
                        "skip_reason": str(exc),
                    }
                )
                continue

            for regime_name in regime_names:
                regime_spec = QUERY_REGIME_REGISTRY[str(regime_name)]
                if not regime_spec.enabled:
                    continue
                query_manifest = build_query_manifest(SPLIT_TRACK_IDS["test"], regime_spec, CONFIG, force_rebuild=CONFIG.force_recompute_embeddings)
                query_embeddings, query_metadata_df, _ = extract_query_embeddings_grouped_by_length_v2(
                    model=model,
                    input_mode=input_mode,
                    query_manifest=query_manifest,
                    regime_name=str(regime_name),
                    model_cache_key=descriptor.run_id,
                    batch_size=batch_size,
                    collate_fn=retrieval_collate_fn,
                    sample_rate=sample_rate,
                )
                search_started_at = time.perf_counter()
                ranked_track_ids, ranked_scores = search_unique_track_results(index, query_embeddings, reference_track_ids, max(CONFIG.top_k), CONFIG.search_oversample_factor)
                latency_ms_per_query = float((time.perf_counter() - search_started_at) * 1000.0 / max(1, query_embeddings.shape[0]))
                results_df = aggregate_group_rankings(query_metadata_df, ranked_track_ids, ranked_scores)
                results_df["evaluation_phase"] = evaluation_phase
                results_df["aggregation_mode"] = "baseline_mean"
                results_df["run_id"] = descriptor.run_id
                results_df["source"] = descriptor.source
                results_df["model_name"] = descriptor.model_name
                results_df["augmentation_profile"] = descriptor.augmentation_profile
                results_df["embed_dim"] = descriptor.embed_dim
                results_df["windowing_name"] = windowing_name
                results_df["index_name"] = index_name
                results_df["index_kind"] = index_spec.kind
                detailed_frames.append(results_df)
                faiss_rows.append(
                    {
                        "evaluation_phase": evaluation_phase,
                        "aggregation_mode": "baseline_mean",
                        "run_id": descriptor.run_id,
                        "source": descriptor.source,
                        "model_name": descriptor.model_name,
                        "augmentation_profile": descriptor.augmentation_profile,
                        "embed_dim": descriptor.embed_dim,
                        "windowing_name": windowing_name,
                        "index_name": index_name,
                        "index_kind": index_spec.kind,
                        "regime_name": regime_name,
                        "status": "ok",
                        "query_count": int(query_embeddings.shape[0]),
                        "latency_ms_per_query": latency_ms_per_query,
                        "index_size_mb": float(index_metadata["index_size_mb"]),
                        "build_seconds": float(index_metadata["build_seconds"]),
                    }
                )

    clear_memory()
    return (pd.concat(detailed_frames, ignore_index=True) if detailed_frames else pd.DataFrame(), faiss_rows)


def evaluate_descriptor_aggregation(
    descriptor: ArtifactDescriptor,
    regime_names: tuple[str, ...],
) -> tuple[pd.DataFrame, list[dict[str, object]]]:
    """Evaluate one artifact with Notebook 4 multi-segment aggregation modes."""

    detailed_frames: list[pd.DataFrame] = []
    faiss_rows: list[dict[str, object]] = []
    model, input_mode, sample_rate, _ = load_checkpoint_into_model(descriptor)
    retrieval_collate_fn = build_retrieval_collate_fn(model, input_mode)
    batch_size = CONFIG.mert_batch_size if input_mode == "mert_waveform" else CONFIG.batch_size
    index_registry = build_index_spec_registry(CONFIG)

    for windowing_name in CONFIG.active_windowing_names:
        windowing_spec = WINDOWING_REGISTRY[windowing_name]
        if not windowing_spec.enabled:
            continue
        reference_manifest = build_reference_manifest(SPLIT_TRACK_IDS["test"], windowing_spec, CONFIG, force_rebuild=CONFIG.force_recompute_embeddings)
        reference_loader = make_retrieval_loader_v2(
            ReferenceManifestDataset(CONFIG.paths.audio_dir, reference_manifest, CONFIG.sample_rate if input_mode == "spectrogram" else sample_rate, input_mode, CONFIG),
            batch_size,
            retrieval_collate_fn,
            CONFIG.num_workers,
        )
        reference_embeddings, reference_metadata_df, reference_cache = extract_embeddings_with_cache_v2(model, reference_loader, input_mode, reference_manifest, f"reference_{windowing_name}", descriptor.run_id)

        for index_name in CONFIG.active_index_names:
            index_spec = index_registry[index_name]
            try:
                index, index_metadata = build_or_load_faiss_index_v2(reference_embeddings, index_spec, str(reference_cache["cache_key"]))
            except Exception as exc:
                faiss_rows.append(
                    {
                        "evaluation_phase": "aggregation",
                        "aggregation_mode": "temporal_consistency_huber",
                        "run_id": descriptor.run_id,
                        "source": descriptor.source,
                        "model_name": descriptor.model_name,
                        "augmentation_profile": descriptor.augmentation_profile,
                        "embed_dim": descriptor.embed_dim,
                        "windowing_name": windowing_name,
                        "index_name": index_name,
                        "index_kind": index_spec.kind,
                        "status": "skipped",
                        "skip_reason": str(exc),
                    }
                )
                continue

            for regime_name in regime_names:
                regime_spec = QUERY_REGIME_REGISTRY[str(regime_name)]
                if not regime_spec.enabled:
                    continue
                query_manifest = build_query_manifest(SPLIT_TRACK_IDS["test"], regime_spec, CONFIG, force_rebuild=CONFIG.force_recompute_embeddings)
                query_embeddings, query_metadata_df, _ = extract_query_embeddings_grouped_by_length_v2(
                    model=model,
                    input_mode=input_mode,
                    query_manifest=query_manifest,
                    regime_name=str(regime_name),
                    model_cache_key=descriptor.run_id,
                    batch_size=batch_size,
                    collate_fn=retrieval_collate_fn,
                    sample_rate=sample_rate,
                )
                segment_neighbors_df = search_segment_neighbors(index, query_embeddings, reference_metadata_df, CONFIG.aggregation_config.top_k_per_segment, CONFIG.search_oversample_factor)
                latency_ms_per_query = float(segment_neighbors_df.attrs.get("search_elapsed_seconds", 0.0) * 1000.0 / max(1, query_embeddings.shape[0]))
                for aggregation_mode in CONFIG.aggregation_config.modes:
                    results_df = aggregate_group_rankings_with_mode(query_metadata_df, segment_neighbors_df, CONFIG.aggregation_config, aggregation_mode)
                    results_df["evaluation_phase"] = "aggregation"
                    results_df["run_id"] = descriptor.run_id
                    results_df["source"] = descriptor.source
                    results_df["model_name"] = descriptor.model_name
                    results_df["augmentation_profile"] = descriptor.augmentation_profile
                    results_df["embed_dim"] = descriptor.embed_dim
                    results_df["windowing_name"] = windowing_name
                    results_df["index_name"] = index_name
                    results_df["index_kind"] = index_spec.kind
                    detailed_frames.append(results_df)
                    faiss_rows.append(
                        {
                            "evaluation_phase": "aggregation",
                            "aggregation_mode": aggregation_mode,
                            "run_id": descriptor.run_id,
                            "source": descriptor.source,
                            "model_name": descriptor.model_name,
                            "augmentation_profile": descriptor.augmentation_profile,
                            "embed_dim": descriptor.embed_dim,
                            "windowing_name": windowing_name,
                            "index_name": index_name,
                            "index_kind": index_spec.kind,
                            "regime_name": regime_name,
                            "status": "ok",
                            "query_count": int(query_embeddings.shape[0]),
                            "latency_ms_per_query": latency_ms_per_query,
                            "index_size_mb": float(index_metadata["index_size_mb"]),
                            "build_seconds": float(index_metadata["build_seconds"]),
                        }
                    )

    clear_memory()
    return (pd.concat(detailed_frames, ignore_index=True) if detailed_frames else pd.DataFrame(), faiss_rows)


def build_cross_model_comparison(
    baseline_summary_df: pd.DataFrame,
    aggregation_summary_df: pd.DataFrame,
    hard_negative_summary_df: pd.DataFrame,
) -> pd.DataFrame:
    """Build one comparison table spanning baseline, aggregation, and retraining phases."""

    if baseline_summary_df.empty and aggregation_summary_df.empty and hard_negative_summary_df.empty:
        return pd.DataFrame()

    baseline_best = baseline_summary_df.sort_values("ranking_score", ascending=False).groupby("run_id", dropna=False).head(1).reset_index(drop=True) if not baseline_summary_df.empty else pd.DataFrame()
    aggregation_best = aggregation_summary_df.sort_values("ranking_score", ascending=False).groupby("run_id", dropna=False).head(1).reset_index(drop=True) if not aggregation_summary_df.empty else pd.DataFrame()
    hard_negative_best = hard_negative_summary_df.copy()
    if not hard_negative_best.empty:
        hard_negative_best["base_run_id"] = hard_negative_best["run_id"].astype(str).str.replace("_hard_negative", "", regex=False)
        hard_negative_best = hard_negative_best.sort_values("ranking_score", ascending=False).groupby("base_run_id", dropna=False).head(1).reset_index(drop=True)

    rows: list[dict[str, object]] = []
    for base_row in baseline_best.itertuples(index=False):
        base_run_id = str(base_row.run_id)
        aggregation_row = aggregation_best[aggregation_best["run_id"] == base_run_id].head(1)
        hard_negative_row = hard_negative_best[hard_negative_best["base_run_id"] == base_run_id].head(1) if not hard_negative_best.empty else pd.DataFrame()
        baseline_floor = float(base_row.worst_condition_top1) if pd.notna(base_row.worst_condition_top1) else float("nan")
        aggregation_floor = float(aggregation_row["worst_condition_top1"].iloc[0]) if not aggregation_row.empty else float("nan")
        hard_negative_floor = float(hard_negative_row["worst_condition_top1"].iloc[0]) if not hard_negative_row.empty else float("nan")
        intervention_scores = {
            "baseline": baseline_floor if np.isfinite(baseline_floor) else -np.inf,
            "aggregation": aggregation_floor if np.isfinite(aggregation_floor) else -np.inf,
            "hard_negative": hard_negative_floor if np.isfinite(hard_negative_floor) else -np.inf,
        }
        best_intervention = max(intervention_scores, key=intervention_scores.get)
        rows.append(
            {
                "base_run_id": base_run_id,
                "model_name": str(base_row.model_name),
                "baseline_windowing_name": str(base_row.windowing_name),
                "baseline_index_name": str(base_row.index_name),
                "baseline_ranking_score": float(base_row.ranking_score),
                "baseline_floor_top1": baseline_floor,
                "aggregation_mode": str(aggregation_row["aggregation_mode"].iloc[0]) if not aggregation_row.empty else "",
                "aggregation_floor_top1": aggregation_floor,
                "hard_negative_run_id": str(hard_negative_row["run_id"].iloc[0]) if not hard_negative_row.empty else "",
                "hard_negative_floor_top1": hard_negative_floor,
                "aggregation_gain_over_baseline": float(aggregation_floor - baseline_floor) if np.isfinite(aggregation_floor) and np.isfinite(baseline_floor) else float("nan"),
                "hard_negative_gain_over_baseline": float(hard_negative_floor - baseline_floor) if np.isfinite(hard_negative_floor) and np.isfinite(baseline_floor) else float("nan"),
                "best_intervention": best_intervention,
                "best_floor_top1": float(intervention_scores[best_intervention]) if np.isfinite(intervention_scores[best_intervention]) else float("nan"),
            }
        )
    return pd.DataFrame(rows).sort_values(["best_floor_top1", "baseline_ranking_score"], ascending=[False, False]).reset_index(drop=True)


def build_failure_cases(detailed_results_df: pd.DataFrame, max_rows: int = 250) -> pd.DataFrame:
    """Capture the most severe remaining misses for manual follow-up."""

    if detailed_results_df.empty:
        return pd.DataFrame()
    failure_df = detailed_results_df[detailed_results_df["rank"].astype(int) > 1].copy()
    if failure_df.empty:
        return failure_df
    failure_df["top1_mismatch"] = failure_df["true_track_id"].astype(int) != failure_df["top1_track_id"].astype(int)
    selected_columns = [
        "evaluation_phase",
        "aggregation_mode",
        "run_id",
        "model_name",
        "windowing_name",
        "index_name",
        "regime_name",
        "condition_name",
        "query_length_seconds",
        "query_group_id",
        "true_track_id",
        "top1_track_id",
        "rank",
        "top1_score",
        "top_candidate_track_ids_json",
        "top_candidate_scores_json",
        "top1_mismatch",
    ]
    existing_columns = [column for column in selected_columns if column in failure_df.columns]
    return failure_df.sort_values(["rank", "top1_score"], ascending=[False, True]).head(max_rows)[existing_columns].reset_index(drop=True)


def export_dataframe_if_needed(filepath: Path, dataframe: pd.DataFrame) -> None:
    """Write a CSV export unless the caller requested export reuse."""

    if filepath.exists() and CONFIG.skip_existing_exports and not CONFIG.force_recompute_embeddings:
        return
    save_dataframe(filepath, dataframe)


def export_text_if_needed(filepath: Path, text: str) -> None:
    """Write a text export unless the caller requested export reuse."""

    if filepath.exists() and CONFIG.skip_existing_exports and not CONFIG.force_recompute_embeddings:
        return
    safe_mkdir(filepath.parent)
    filepath.write_text(text, encoding="utf-8")


def write_markdown_reports(
    baseline_summary_df: pd.DataFrame,
    aggregation_summary_df: pd.DataFrame,
    hard_negative_summary_df: pd.DataFrame,
    cross_model_df: pd.DataFrame,
) -> tuple[str, str]:
    """Create the two markdown summaries required by Notebook 4."""

    baseline_top = baseline_summary_df.head(1)
    aggregation_top = aggregation_summary_df.head(1)
    hard_negative_top = hard_negative_summary_df.head(1)
    recommendation = cross_model_df.head(1)

    conclusions_lines = [
        "# Notebook 4 Conclusions",
        "",
        f"- Historical artifacts evaluated: {len(HISTORICAL_DESCRIPTORS)}",
        f"- Hard-negative targets retrained: {len(HARD_NEGATIVE_ARTIFACTS)}",
    ]
    if not baseline_top.empty:
        row = baseline_top.iloc[0]
        conclusions_lines.append(f"- Best baseline historical run: `{row['run_id']}` using `{row['windowing_name']}` + `{row['index_name']}` with ranking score `{row['ranking_score']:.4f}`.")
    if not aggregation_top.empty:
        row = aggregation_top.iloc[0]
        conclusions_lines.append(f"- Best aggregation result: `{row['run_id']}` with mode `{row['aggregation_mode']}` and floor `{row['worst_condition_top1']:.4f}`.")
    if not hard_negative_top.empty:
        row = hard_negative_top.iloc[0]
        conclusions_lines.append(f"- Best hard-negative result: `{row['run_id']}` with floor `{row['worst_condition_top1']:.4f}`.")
    if not recommendation.empty:
        row = recommendation.iloc[0]
        conclusions_lines.append(f"- Highest-floor intervention overall: `{row['best_intervention']}` on base run `{row['base_run_id']}` with floor `{row['best_floor_top1']:.4f}`.")

    presentation_lines = [
        "# Notebook 4 Presentation Summary",
        "",
        "## Recommended storyline",
        "",
        "1. Scale-up from `fma_small` to `fma_medium` changed the retrieval floor more than the average case.",
        "2. Multi-segment aggregation is the cheapest inference-time intervention because it reuses historical checkpoints.",
        "3. Hard-negative retraining is the training-time intervention that matters when acoustically similar false matches dominate.",
        "4. The final recommendation should prioritize the best historical checkpoint, then add aggregation by default, and reserve retraining for the most failure-sensitive deployment path.",
    ]
    return "\n".join(conclusions_lines) + "\n", "\n".join(presentation_lines) + "\n"


def zip_output_bundle(config: ExperimentConfig) -> Path:
    """Create the final Notebook 4 zip bundle containing exported artifacts and plots."""

    bundle_path = config.paths.artifact_zip
    if bundle_path.exists() and config.skip_existing_exports and not config.force_recompute_embeddings:
        return bundle_path
    safe_mkdir(bundle_path.parent)
    with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for export_root in [config.paths.export_dir, config.paths.metrics_dir, config.paths.plot_dir]:
            if not export_root.exists():
                continue
            for filepath in sorted(export_root.rglob("*")):
                if filepath.is_file():
                    archive.write(filepath, arcname=str(filepath.relative_to(config.paths.output_root)))
    return bundle_path


print_phase_banner("Notebook 4 full execution")
INDEX_SPEC_REGISTRY = build_index_spec_registry(CONFIG)
HISTORICAL_DESCRIPTORS = build_historical_artifact_descriptors(HISTORICAL_ARTIFACTS_DF, CONFIG)
HARD_NEGATIVE_ARTIFACTS: list[ArtifactDescriptor] = []
BASELINE_DETAILED_FRAMES: list[pd.DataFrame] = []
AGGREGATION_DETAILED_FRAMES: list[pd.DataFrame] = []
HARD_NEGATIVE_DETAILED_FRAMES: list[pd.DataFrame] = []
FAISS_RESULT_ROWS: list[dict[str, object]] = []

if CONFIG.run_baseline_historical_eval:
    for descriptor in tqdm(HISTORICAL_DESCRIPTORS, desc="baseline-historical", leave=False):
        detailed_df, faiss_rows = evaluate_descriptor_baseline(descriptor, BASELINE_REGIME_NAMES, "baseline_historical")
        if not detailed_df.empty:
            BASELINE_DETAILED_FRAMES.append(detailed_df)
        FAISS_RESULT_ROWS.extend(faiss_rows)

if CONFIG.run_aggregation_eval:
    for descriptor in tqdm(HISTORICAL_DESCRIPTORS, desc="aggregation-eval", leave=False):
        detailed_df, faiss_rows = evaluate_descriptor_aggregation(descriptor, AGGREGATION_REGIME_NAMES)
        if not detailed_df.empty:
            AGGREGATION_DETAILED_FRAMES.append(detailed_df)
        FAISS_RESULT_ROWS.extend(faiss_rows)

if CONFIG.run_hard_negative_retraining:
    for descriptor in tqdm(HISTORICAL_DESCRIPTORS, desc="hard-negative-retraining", leave=False):
        if descriptor.run_id not in CONFIG.hard_negative_config.target_run_ids:
            continue
        hard_negative_artifact = ensure_hard_negative_artifact(descriptor)
        HARD_NEGATIVE_ARTIFACTS.append(hard_negative_artifact)
        detailed_df, faiss_rows = evaluate_descriptor_baseline(hard_negative_artifact, HARD_NEGATIVE_EVAL_REGIME_NAMES, "hard_negative")
        if not detailed_df.empty:
            HARD_NEGATIVE_DETAILED_FRAMES.append(detailed_df)
        FAISS_RESULT_ROWS.extend(faiss_rows)

BASELINE_DETAILED_DF = pd.concat(BASELINE_DETAILED_FRAMES, ignore_index=True) if BASELINE_DETAILED_FRAMES else pd.DataFrame()
AGGREGATION_DETAILED_DF = pd.concat(AGGREGATION_DETAILED_FRAMES, ignore_index=True) if AGGREGATION_DETAILED_FRAMES else pd.DataFrame()
HARD_NEGATIVE_DETAILED_DF = pd.concat(HARD_NEGATIVE_DETAILED_FRAMES, ignore_index=True) if HARD_NEGATIVE_DETAILED_FRAMES else pd.DataFrame()

BASELINE_METRICS_LONG_DF = build_metrics_long_table(BASELINE_DETAILED_DF)
AGGREGATION_METRICS_LONG_DF = build_metrics_long_table(AGGREGATION_DETAILED_DF)
HARD_NEGATIVE_METRICS_LONG_DF = build_metrics_long_table(HARD_NEGATIVE_DETAILED_DF)
FINAL_METRICS_LONG_DF = pd.concat([df for df in [BASELINE_METRICS_LONG_DF, AGGREGATION_METRICS_LONG_DF, HARD_NEGATIVE_METRICS_LONG_DF] if not df.empty], ignore_index=True) if any(not df.empty for df in [BASELINE_METRICS_LONG_DF, AGGREGATION_METRICS_LONG_DF, HARD_NEGATIVE_METRICS_LONG_DF]) else pd.DataFrame()
FAISS_SWEEP_RESULTS_DF = pd.DataFrame(FAISS_RESULT_ROWS)

BASELINE_SUMMARY_DF = build_baseline_summary_table(BASELINE_METRICS_LONG_DF, FAISS_SWEEP_RESULTS_DF)
AGGREGATION_SUMMARY_DF = build_aggregation_summary_table(AGGREGATION_METRICS_LONG_DF, FAISS_SWEEP_RESULTS_DF)
HARD_NEGATIVE_SUMMARY_DF = build_hard_negative_summary_table(HARD_NEGATIVE_METRICS_LONG_DF, FAISS_SWEEP_RESULTS_DF)
FINAL_METRICS_SUMMARY_DF = pd.concat([df for df in [BASELINE_SUMMARY_DF, AGGREGATION_SUMMARY_DF, HARD_NEGATIVE_SUMMARY_DF] if not df.empty], ignore_index=True) if any(not df.empty for df in [BASELINE_SUMMARY_DF, AGGREGATION_SUMMARY_DF, HARD_NEGATIVE_SUMMARY_DF]) else pd.DataFrame()
CROSS_MODEL_COMPARISON_DF = build_cross_model_comparison(BASELINE_SUMMARY_DF, AGGREGATION_SUMMARY_DF, HARD_NEGATIVE_SUMMARY_DF)
COMBINED_DETAILED_DF = pd.concat([df for df in [BASELINE_DETAILED_DF, AGGREGATION_DETAILED_DF, HARD_NEGATIVE_DETAILED_DF] if not df.empty], ignore_index=True) if any(not df.empty for df in [BASELINE_DETAILED_DF, AGGREGATION_DETAILED_DF, HARD_NEGATIVE_DETAILED_DF]) else pd.DataFrame()
FAILURE_CASES_DF = build_failure_cases(COMBINED_DETAILED_DF)

export_dataframe_if_needed(CONFIG.paths.export_dir / "notebook4_base_eval_long.csv", BASELINE_METRICS_LONG_DF)
export_dataframe_if_needed(CONFIG.paths.export_dir / "notebook4_base_eval_summary.csv", BASELINE_SUMMARY_DF)
export_dataframe_if_needed(CONFIG.paths.export_dir / "notebook4_aggregation_eval_long.csv", AGGREGATION_METRICS_LONG_DF)
export_dataframe_if_needed(CONFIG.paths.export_dir / "notebook4_aggregation_eval_summary.csv", AGGREGATION_SUMMARY_DF)
export_dataframe_if_needed(CONFIG.paths.export_dir / "notebook4_hard_negative_eval_long.csv", HARD_NEGATIVE_METRICS_LONG_DF)
export_dataframe_if_needed(CONFIG.paths.export_dir / "notebook4_hard_negative_eval_summary.csv", HARD_NEGATIVE_SUMMARY_DF)
export_dataframe_if_needed(CONFIG.paths.export_dir / "notebook4_cross_model_comparison.csv", CROSS_MODEL_COMPARISON_DF)
export_dataframe_if_needed(CONFIG.paths.export_dir / "notebook4_failure_cases.csv", FAILURE_CASES_DF)
export_dataframe_if_needed(CONFIG.paths.export_dir / "faiss_sweep_results.csv", FAISS_SWEEP_RESULTS_DF)
export_dataframe_if_needed(CONFIG.paths.export_dir / "notebook4_base_eval_detailed.csv", BASELINE_DETAILED_DF)
export_dataframe_if_needed(CONFIG.paths.export_dir / "notebook4_aggregation_eval_detailed.csv", AGGREGATION_DETAILED_DF)
export_dataframe_if_needed(CONFIG.paths.export_dir / "notebook4_hard_negative_eval_detailed.csv", HARD_NEGATIVE_DETAILED_DF)

if CONFIG.run_plot_exports:
    export_plot(BASELINE_SUMMARY_DF, "baseline_ranking_score.png", "Notebook 4 Baseline Ranking Score", "run_id", "ranking_score", "#2c7fb8")
    export_plot(AGGREGATION_SUMMARY_DF, "aggregation_floor.png", "Notebook 4 Aggregation Worst-Condition Floor", "run_id", "worst_condition_top1", "#41ab5d")
    export_plot(HARD_NEGATIVE_SUMMARY_DF, "hard_negative_floor.png", "Notebook 4 Hard-Negative Worst-Condition Floor", "run_id", "worst_condition_top1", "#f16913")
    export_plot(CROSS_MODEL_COMPARISON_DF, "cross_model_best_floor.png", "Notebook 4 Best Floor By Historical Run", "base_run_id", "best_floor_top1", "#756bb1")

CONCLUSIONS_MARKDOWN, PRESENTATION_MARKDOWN = write_markdown_reports(BASELINE_SUMMARY_DF, AGGREGATION_SUMMARY_DF, HARD_NEGATIVE_SUMMARY_DF, CROSS_MODEL_COMPARISON_DF)
export_text_if_needed(CONFIG.paths.export_dir / "notebook4_conclusions.md", CONCLUSIONS_MARKDOWN)
export_text_if_needed(CONFIG.paths.export_dir / "notebook4_presentation_summary.md", PRESENTATION_MARKDOWN)
save_json(
    CONFIG.paths.export_dir / "notebook4_execution_manifest.json",
    {
        "historical_descriptors": HISTORICAL_DESCRIPTORS,
        "hard_negative_artifacts": HARD_NEGATIVE_ARTIFACTS,
        "baseline_regime_names": BASELINE_REGIME_NAMES,
        "aggregation_regime_names": AGGREGATION_REGIME_NAMES,
        "hard_negative_eval_regime_names": HARD_NEGATIVE_EVAL_REGIME_NAMES,
        "export_counts": {
            "baseline_long_rows": int(BASELINE_METRICS_LONG_DF.shape[0]),
            "aggregation_long_rows": int(AGGREGATION_METRICS_LONG_DF.shape[0]),
            "hard_negative_long_rows": int(HARD_NEGATIVE_METRICS_LONG_DF.shape[0]),
            "failure_rows": int(FAILURE_CASES_DF.shape[0]),
        },
    },
)

ZIP_OUTPUT_PATH = zip_output_bundle(CONFIG) if CONFIG.run_zip_export else None

display(BASELINE_SUMMARY_DF.head(10))
display(AGGREGATION_SUMMARY_DF.head(10))
display(HARD_NEGATIVE_SUMMARY_DF.head(10))
display(CROSS_MODEL_COMPARISON_DF.head(10))
print(CONCLUSIONS_MARKDOWN)


In [ ]:
def warm_waveform_cache(audio_dir: Path, track_ids: list[int], sample_rate: int, cache_dir: Path) -> dict[str, object]:
    """Decode and resample each track once so later dataset reads hit the local cache."""
    cache_dir.mkdir(parents=True, exist_ok=True)
    unique_track_ids = sorted(set(int(track_id) for track_id in track_ids))

    failed_records: list[dict[str, object]] = []

    for track_id in tqdm(unique_track_ids, desc="warm-waveform-cache", leave=False):
        filepath = get_audio_path(audio_dir, track_id)
        try:
            _ = load_audio_file(
                filepath,
                target_sr=sample_rate,
                mono=True,
                cache_dir=cache_dir,
            )
        except Exception as exc:
            failed_records.append(
                {
                    "track_id": int(track_id),
                    "filepath": str(filepath),
                    "error": repr(exc),
                }
            )

    summary = {
        "total_tracks": len(unique_track_ids),
        "failed_tracks": len(failed_records),
        "failed_records": failed_records,
    }

    if failed_records:
        failure_report_path = cache_dir / "waveform_cache_failures.json"
        with failure_report_path.open("w", encoding="utf-8") as handle:
            json.dump(summary, handle, indent=2)
            handle.write("\n")
        print(f"Waveform cache warm-up completed with {len(failed_records)} failures.")
        print(f"Failure report written to: {failure_report_path}")
    else:
        print("Waveform cache warm-up completed with 0 failures.")

    return summary


all_experiment_track_ids = (
    SPLIT_TRACK_IDS["training"]
    + SPLIT_TRACK_IDS["validation"]
    + SPLIT_TRACK_IDS["test"]
)

cache_warmup_summary = warm_waveform_cache(
    audio_dir=CONFIG.paths.audio_dir,
    track_ids=all_experiment_track_ids,
    sample_rate=CONFIG.sample_rate,
    cache_dir=CONFIG.paths.waveform_cache_dir,
)

print(f"Waveform cache ready at: {CONFIG.paths.waveform_cache_dir}")
print(json.dumps(
    {
        "total_tracks": cache_warmup_summary["total_tracks"],
        "failed_tracks": cache_warmup_summary["failed_tracks"],
    },
    indent=2,
))

## Smoke Tests, Historical Evaluation, Aggregation, Hard Negatives, And Final Synthesis


### Notebook 4 Smoke Tests, Full Execution, And Exported Deliverables


In [ ]:
def run_smoke_tests() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run lightweight end-to-end sanity checks before the full Notebook 4 matrix."""

    if not SPLIT_TRACK_IDS["training"] or not SPLIT_TRACK_IDS["test"]:
        raise ValueError("Smoke tests require non-empty training and test splits.")

    smoke_rows: list[dict[str, object]] = []
    smoke_train_ids = SPLIT_TRACK_IDS["training"][: max(4, min(CONFIG.smoke_test_track_count, len(SPLIT_TRACK_IDS["training"])))]
    smoke_test_ids = SPLIT_TRACK_IDS["test"][: max(4, min(CONFIG.smoke_test_track_count, len(SPLIT_TRACK_IDS["test"])))]

    smoke_rows.append(
        {
            "check_name": "split_inventory",
            "status": "ok",
            "detail": json.dumps({split_name: len(track_ids) for split_name, track_ids in SPLIT_TRACK_IDS.items()}),
        }
    )

    waveform, _ = load_audio_file(
        get_audio_path(CONFIG.paths.audio_dir, int(smoke_test_ids[0])),
        target_sr=CONFIG.sample_rate,
        mono=True,
        cache_dir=CONFIG.paths.waveform_cache_dir,
    )
    smoke_rows.append(
        {
            "check_name": "audio_decode_probe",
            "status": "ok",
            "detail": json.dumps({"track_id": int(smoke_test_ids[0]), "samples": int(waveform.shape[0])}),
        }
    )

    candidate_specs = [
        ModelRunConfig("cnn", "baseline", 128, 1, 1e-4, 1e-4, 0.07, False),
        ModelRunConfig("hybrid_transformer", "baseline", 128, 1, 1e-4, 1e-4, 0.07, False),
        ModelRunConfig("frozen_mert", "extended_existing", 128, 1, 1e-3, 1e-4, 0.07, True),
    ]

    for run_spec in candidate_specs:
        model, input_mode, sample_rate, pair_collate_override = build_model(CONFIG, run_spec)
        model = model.to(CONFIG.device)
        collate_fn = pair_collate_override if pair_collate_override is not None else collate_pair_batch
        batch_size = CONFIG.mert_batch_size if input_mode == "mert_waveform" else min(CONFIG.batch_size, 4)
        dataset = ContrastivePairDataset(CONFIG.paths.audio_dir, smoke_train_ids, int(CONFIG.segment_seconds * sample_rate), sample_rate, input_mode, CONFIG, "baseline")
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0, collate_fn=collate_fn)
        batch = next(iter(loader))
        batch = safe_to_device(batch, CONFIG.device)
        loss_fn = NTXentLoss(run_spec.temperature)
        optimizer = torch.optim.AdamW([parameter for parameter in model.parameters() if parameter.requires_grad], lr=run_spec.learning_rate, weight_decay=run_spec.weight_decay)
        optimizer.zero_grad(set_to_none=True)
        anchor_embeddings, positive_embeddings = forward_pair_batch(model, batch, input_mode, CONFIG.device_type, CONFIG.amp_enabled)
        loss = loss_fn(anchor_embeddings, positive_embeddings)
        loss.backward()
        optimizer.step()
        smoke_rows.append(
            {
                "check_name": f"forward_backward_{run_spec.model_name}",
                "status": "ok",
                "detail": json.dumps(
                    {
                        "input_mode": input_mode,
                        "loss": float(loss.detach().cpu().item()),
                        "anchor_shape": tuple(anchor_embeddings.shape),
                        "positive_shape": tuple(positive_embeddings.shape),
                    }
                ),
            }
        )
        clear_memory()

    retrieval_spec = candidate_specs[0]
    model, input_mode, sample_rate, _ = build_model(CONFIG, retrieval_spec)
    model = model.to(CONFIG.device)
    retrieval_collate_fn = build_retrieval_collate_fn(model, input_mode)
    index_registry = build_index_spec_registry(CONFIG)

    reference_manifest = build_reference_manifest(smoke_test_ids[: min(8, len(smoke_test_ids))], WINDOWING_REGISTRY["single_center"], CONFIG, force_rebuild=CONFIG.force_recompute_embeddings)
    query_manifest = build_query_manifest(smoke_test_ids[: min(4, len(smoke_test_ids))], QUERY_REGIME_REGISTRY["multi_segment_same_track"], CONFIG, force_rebuild=CONFIG.force_recompute_embeddings)
    grouped_query_manifest = build_query_manifest(smoke_test_ids[: min(4, len(smoke_test_ids))], QUERY_REGIME_REGISTRY["multi_segment_fragmented_weighted"], CONFIG, force_rebuild=CONFIG.force_recompute_embeddings)

    reference_loader = make_retrieval_loader_v2(
        ReferenceManifestDataset(CONFIG.paths.audio_dir, reference_manifest, CONFIG.sample_rate if input_mode == "spectrogram" else sample_rate, input_mode, CONFIG),
        min(CONFIG.batch_size, 8),
        retrieval_collate_fn,
        0,
    )
    reference_embeddings, reference_metadata_df, reference_cache = extract_embeddings_with_cache_v2(model, reference_loader, input_mode, reference_manifest, "smoke_reference", "smoke_reference_model")
    index, index_metadata = build_or_load_faiss_index_v2(reference_embeddings, index_registry["exact_ip"], str(reference_cache["cache_key"]))

    query_embeddings, query_metadata_df, _ = extract_query_embeddings_grouped_by_length_v2(
        model=model,
        input_mode=input_mode,
        query_manifest=query_manifest,
        regime_name="multi_segment_same_track",
        model_cache_key="smoke_reference_model",
        batch_size=min(CONFIG.batch_size, 8),
        collate_fn=retrieval_collate_fn,
        sample_rate=sample_rate,
    )
    segment_neighbors_df = search_segment_neighbors(index, query_embeddings, reference_metadata_df, top_k_per_segment=3, oversample_factor=4)
    aggregated_df = aggregate_group_rankings_with_mode(query_metadata_df, segment_neighbors_df, CONFIG.aggregation_config, "temporal_consistency_huber")
    smoke_rows.append(
        {
            "check_name": "tiny_retrieval_and_aggregation",
            "status": "ok",
            "detail": json.dumps(
                {
                    "reference_rows": int(reference_manifest.shape[0]),
                    "query_rows": int(query_manifest.shape[0]),
                    "grouped_query_rows": int(grouped_query_manifest.shape[0]),
                    "segment_neighbor_rows": int(segment_neighbors_df.shape[0]),
                    "aggregation_rows": int(aggregated_df.shape[0]),
                    "index_size_mb": float(index_metadata["index_size_mb"]),
                }
            ),
        }
    )

    smoke_descriptor = ArtifactDescriptor(
        run_id="smoke_hard_negative",
        source="smoke",
        model_name=retrieval_spec.model_name,
        augmentation_profile=retrieval_spec.augmentation_profile,
        embed_dim=retrieval_spec.embed_dim,
        checkpoint_path=Path("smoke_hard_negative"),
        freeze_backbone=retrieval_spec.freeze_backbone,
        notes="Smoke-test descriptor.",
    )
    hard_negative_cache_df = mine_hard_negative_cache(
        model=model,
        input_mode=input_mode,
        descriptor=smoke_descriptor,
        candidate_track_ids=smoke_train_ids,
        split_name="smoke_training",
        batch_size=min(CONFIG.batch_size, 8),
        collate_fn=retrieval_collate_fn,
        sample_rate=sample_rate,
        config=CONFIG,
    )
    smoke_rows.append(
        {
            "check_name": "hard_negative_cache_roundtrip",
            "status": "ok" if not hard_negative_cache_df.empty else "warning",
            "detail": json.dumps({"pairs": int(hard_negative_cache_df.shape[0]), "cache_files": len(list(CONFIG.paths.hard_negative_cache_dir.glob('smoke_hard_negative_smoke_training_*.csv')))}),
        }
    )

    smoke_report_df = pd.DataFrame(smoke_rows)
    return smoke_report_df, HISTORICAL_ARTIFACTS_DF.copy()


MODEL_SMOKE_TEST_REPORT, ARTIFACT_CHECK_REPORT = run_smoke_tests()
display(MODEL_SMOKE_TEST_REPORT)
display(ARTIFACT_CHECK_REPORT)

FINAL_METRICS_LONG_DF = pd.DataFrame()
FINAL_METRICS_SUMMARY_DF = pd.DataFrame()
FAILURE_CASES_DF = pd.DataFrame()

save_dataframe(CONFIG.paths.metrics_dir / "model_smoke_test_report.csv", MODEL_SMOKE_TEST_REPORT)
save_dataframe(CONFIG.paths.export_dir / "smoke_test_report.csv", MODEL_SMOKE_TEST_REPORT)
save_dataframe(CONFIG.paths.metrics_dir / "artifact_check_report.csv", ARTIFACT_CHECK_REPORT)
save_json(
    CONFIG.paths.export_dir / "experiment_registry.json",
    {
        "config": CONFIG,
        "runtime": RUNTIME_INFO,
        "query_regimes": QUERY_REGIME_REGISTRY,
        "windowing_specs": WINDOWING_REGISTRY,
        "hard_negative_config": CONFIG.hard_negative_config,
        "aggregation_config": CONFIG.aggregation_config,
    },
)


# Historical Baseline, Aggregation, Hard-Negative Retraining, And Final Exports


In [ ]:
BASELINE_REGIME_NAMES = tuple(
    regime_name
    for regime_name in (
        "clean_current",
        "short_centered",
        "short_offcenter",
        "combined_moderate",
        "multi_segment_same_track",
        "realistic_hard",
    )
    if regime_name in QUERY_REGIME_REGISTRY and QUERY_REGIME_REGISTRY[regime_name].enabled
)
AGGREGATION_REGIME_NAMES = tuple(
    regime_name
    for regime_name, regime_spec in QUERY_REGIME_REGISTRY.items()
    if regime_spec.enabled and int(regime_spec.group_size) > 1
)
HARD_NEGATIVE_EVAL_REGIME_NAMES = tuple(
    dict.fromkeys(
        list(BASELINE_REGIME_NAMES)
        + (["hard_negative_short_queries"] if "hard_negative_short_queries" in QUERY_REGIME_REGISTRY and QUERY_REGIME_REGISTRY["hard_negative_short_queries"].enabled else [])
    )
)


def build_historical_artifact_descriptors(artifact_df: pd.DataFrame, config: ExperimentConfig) -> list[ArtifactDescriptor]:
    """Convert the historical artifact registry into loadable descriptors."""

    available_rows = artifact_df[
        artifact_df["checkpoint_path"].astype(str).ne("")
        & artifact_df["status"].isin(["valid", "incomplete"])
    ].copy()
    descriptors = [
        ArtifactDescriptor(
            run_id=str(row.run_id),
            source=str(row.expected_source),
            model_name=str(row.model_name),
            augmentation_profile=str(row.augmentation_profile),
            embed_dim=int(row.embed_dim),
            checkpoint_path=Path(str(row.checkpoint_path)),
            freeze_backbone=bool(row.freeze_backbone),
            notes=f"Recovered from {row.expected_source}.",
        )
        for row in available_rows.itertuples(index=False)
    ]
    return sorted(descriptors, key=lambda descriptor: descriptor.run_id)


def load_checkpoint_into_model(
    descriptor: ArtifactDescriptor,
) -> tuple[nn.Module, InputMode, int, Callable[[list[dict[str, object]]], dict[str, object]] | None]:
    """Load a checkpointed artifact into the corresponding Notebook model."""

    run_config = ModelRunConfig(
        descriptor.model_name,
        descriptor.augmentation_profile,
        descriptor.embed_dim,
        1,
        1e-4,
        1e-4,
        0.07,
        descriptor.freeze_backbone,
    )
    model, input_mode, sample_rate, pair_collate_override = build_model(CONFIG, run_config)
    checkpoint_payload = torch.load(descriptor.checkpoint_path, map_location=CONFIG.device)
    if isinstance(model, FrozenMertFingerprinter):
        projection_state_dict = checkpoint_payload.get("projection_state_dict")
        if projection_state_dict is None:
            raise KeyError(f"MERT checkpoint {descriptor.checkpoint_path} is missing projection_state_dict.")
        model.projection.load_state_dict(projection_state_dict)
    else:
        model_state_dict = checkpoint_payload.get("model_state_dict")
        if model_state_dict is None:
            raise KeyError(f"Checkpoint {descriptor.checkpoint_path} is missing model_state_dict.")
        model.load_state_dict(model_state_dict)
    model = model.to(CONFIG.device)
    return model, input_mode, sample_rate, pair_collate_override


def build_hard_negative_run_config(descriptor: ArtifactDescriptor) -> ModelRunConfig:
    """Construct the retraining config used for one hard-negative target artifact."""

    learning_rate = 5e-5 if descriptor.model_name == "frozen_mert" else 1e-4
    return ModelRunConfig(
        model_name=descriptor.model_name,
        augmentation_profile=descriptor.augmentation_profile,
        embed_dim=descriptor.embed_dim,
        epochs=max(1, int(CONFIG.hard_negative_config.retrain_epochs)),
        learning_rate=learning_rate,
        weight_decay=1e-4,
        temperature=0.07,
        freeze_backbone=descriptor.freeze_backbone,
        enabled=True,
    )


## Final Recommendation Criteria

The final synthesis must answer:

1. Which historical model is best at `fma_medium` scale?
2. Did the Notebook 3 winner remain the winner?
3. Which intervention improves the worst-case floor the most?
4. Which upgrade is cheapest to adopt in practice?
5. What should the final presentation recommend as the system-level design?
